<a href="https://colab.research.google.com/github/peterbmob/CH-PFC/blob/main/CH_spectral_fin_20250902.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Li intercalation in an LFP particle with masked & heterogeneous elasticity (spectral)

This notebook extends the original `spectral_solve_fen3.py` to support two mechanical models:

1. **Uniform (masked) mechanics — Level 1:**
   - Eigenstrain and elastic coupling are restricted to the particle via the mask `H`.
   - Elastic constants are single-valued (LFP), as in the original spectral solver.

2. **Heterogeneous (particle vs reservoir) mechanics — Level 2:**
   - Spatially varying isotropic stiffness: `C(x) = C_res + H(x)*(C_part - C_res)`.
   - Solved by an FFT-based iterative equilibrium scheme using a homogeneous reference operator.

Choose the behavior via `Model["mech_mode"]` in the external config file (`config_mech.py`).

> **Note**: This notebook keeps the original nondimensionalization and Butler–Volmer treatment.

In [2]:
%%writefile /content/config_mech.py
Adapt = {'remesh': 25, 'amrStart': 400, 'beta': [1.0, 0.0], 'tol': 0.005}
Domain = {'Lx': 32, 'Ly': 64, 'Lz': None, 'nde': 0.15, 'emax': 4, 'p': 1, 'q': 2, 'diag': 'right/left', 'PB': [True, False, True]}
Interval = {'timestep': 0.0001, 'restart': False, 'f_checkpoint': 1000, 'time_interval_output': 1000, 'energy': 5, 'domain': 100, 'runtime': 0.001, 'cutback': 0.5, 'growth': 1.5, 'min_iter': 3, 'max_iter': 8, 'maxtimestep': 30000}
Model = {'flux': 'BV', 'k0': 20350.0, 'j0': None, 'Δφ': 0.1, 'Wc': 1e-09, 'sigma': 0.072, 'DLi': 1e-15, 'Omega': 12000.0, 'R': 8.3145, 'To': 298, 'vm': 4.38e-05, 'F': 96485.33, 'NA': 6.02214076e+23, 'λᵣeorg': 8.3, 'Const': 3.358, 'μeq': -2.11e-05, 'E': 125700000000.0, 'ν': 0.252, 'e11': 0.05, 'e22': 0.036, 'E_res': 5000000000.0, 'ν_res': 0.3, 'mech_mode': 'uniform', 'mask_mech_to_particle': True, 'eigenstrain_only_in_particle': True, 'mask_mu_el_to_particle_in_hetero': False, 'hetero_tol': 1e-06, 'hetero_maxit': 100, 'hetero_omega': 0.7, 'hetero_verbose': False, 'mask_shape': 'slab_y', 'r_electrode_Wc': 12.48, 'cx': 16.0, 'cy': 32.0, 'slab_x_center': 16.0, 'slab_x_width': 8.0, 'slab_y_center': 32.0, 'slab_y_width': 32.0, 'c_inside': 0.1, 'c_outside': 1.0, 'seed': 0, 'nsteps': 1000, 'save_frames': True, 'frame_every': 50, 'frame_dir': 'frames', 'cmap': 'viridis', 'V_ref': 3.45, 'kappa_dim': 1.68e-7}

Overwriting /content/config_mech.py


In [70]:
# Imports & config
from __future__ import annotations
import numpy as np
from numpy.fft import fftn, ifftn, fftfreq
import importlib, sys, os
import matplotlib
matplotlib.use("Agg")  # for headless runs
import matplotlib.pyplot as plt

# Load external config (edit config_mech.py)
if not os.path.exists('config_mech.py'):
    # Create a dummy config file if it doesn't exist, for demonstration.
    # In a real scenario, the user would provide this file.
    with open('config_mech.py', 'w') as f:
        Adapt = {'remesh': 25, 'amrStart': 400, 'beta': [1.0, 0.0], 'tol': 0.005}
        Domain = {'Lx': 32, 'Ly': 64, 'Lz': None, 'nde': 0.15, 'emax': 4, 'p': 1, 'q': 2, 'diag': 'right/left', 'PB': [True, False, True]}
        Interval = {'timestep': 0.0001, 'restart': False, 'f_checkpoint': 1000, 'time_interval_output': 1000, 'energy': 5, 'domain': 100, 'runtime': 0.001, 'cutback': 0.5, 'growth': 1.5, 'min_iter': 3, 'max_iter': 8, 'maxtimestep': 30000}
        Model = {'flux': 'BV', 'k0': 20350.0, 'j0': None, 'Δφ': 0.1, 'Wc': 1e-09, 'sigma': 0.072, 'DLi': 1e-15, 'Omega': 12000.0, 'R': 8.3145, 'To': 298, 'vm': 4.38e-05, 'F': 96485.33, 'NA': 6.02214076e+23, 'λᵣeorg': 8.3, 'Const': 3.358, 'μeq': -2.11e-05, 'E': 125700000000.0, 'ν': 0.252, 'e11': 0.05, 'e22': 0.036, 'E_res': 5000000000.0, 'ν_res': 0.3, 'mech_mode': 'uniform', 'mask_mech_to_particle': True, 'eigenstrain_only_in_particle': True, 'mask_mu_el_to_particle_in_hetero': False, 'hetero_tol': 1e-06, 'hetero_maxit': 100, 'hetero_omega': 0.7, 'hetero_verbose': False, 'mask_shape': 'slab_y', 'r_electrode_Wc': 12.48, 'cx': 16.0, 'cy': 32.0, 'slab_x_center': 16.0, 'slab_x_width': 8.0, 'slab_y_center': 32.0, 'slab_y_width': 32.0, 'c_inside': 0.1, 'c_outside': 1.0, 'seed': 0, 'nsteps': 1000, 'save_frames': True, 'frame_every': 50, 'frame_dir': 'frames', 'cmap': 'viridis', 'V_ref': 3.45, 'kappa_dim': 1.68e-11}

spec = importlib.util.spec_from_file_location('config_mech', os.path.join(os.getcwd(), 'config_mech.py'))
config_mech = importlib.util.module_from_spec(spec)
sys.modules['config_mech'] = config_mech
assert spec.loader is not None
spec.loader.exec_module(config_mech)

Adapt   = getattr(config_mech, 'Adapt')
Domain  = getattr(config_mech, 'Domain')
Interval= getattr(config_mech, 'Interval')
Model   = getattr(config_mech, 'Model')

In [71]:
# Nondimensionalization (aligned with the original solver)
Wc    = float(Model["Wc"])         # [m]
sigma = float(Model["sigma"])     # [J/m^2]
DLi   = float(Model["DLi"])       # [m^2/s]
R     = float(Model["R"])         # [J/mol/K]
To    = float(Model["To"])        # [K]
vm    = float(Model["vm"])        # [m^3/mol]
Omega = float(Model["Omega"])     # [J/mol]
Fconst= float(Model.get('F', 96485.33))
NA    = float(Model.get('Nα', Model.get('Nα', Model.get('NA', 6.02214076e23))))
DeltaPhi = float(Model.get('Δφ', Model.get('DeltaPhi', 0.0)))
mu_eq    = float(Model.get('μeq', Model.get('mueq', 0.0)))   # [J/m^3]

# Elastic params (particle)
E_p = float(Model["E"])  # Pa
nu_p = float(Model.get('nu', Model.get('ν', 0.25)))

# Elastic params (reservoir) – used in heterogeneous mode
E_r  = float(Model.get("E_res", E_p))
nu_r = float(Model.get('nu_res', Model.get('ν_res', nu_p)))

# Dimensionless scales
Hscale = sigma / Wc                 # [J/m^3]
tc     = Wc**2 / DLi                # [s]
RTv    = (R * To / vm) / Hscale     # dimensionless
Om     = (Omega / vm) / Hscale      # dimensionless
Dm     = 1.0 / RTv                  # dimensionless mobility prefactor

# BV kinetics parameters
alpha = 0.5
j0 = Model.get('j0', None)
if j0 is None:
    k0 = float(Model.get('k0', 2.035e-4))  # [s^-1]
    j0 = (Fconst/(NA * Wc**2)) * k0        # [A/m^2]
else:
    j0 = float(j0)

j0coeff = (vm * j0 / Fconst) * (tc / Wc)    # dimensionless, like original
mu_elec = (mu_eq - (Fconst * DeltaPhi)/vm) / Hscale

# Depth for converting surface flux to current (2D → 3D)
depth = float(Model.get('depth', Wc))
Iconv = (Fconst / vm) * (Wc**2 / tc) * depth

# Access the dimensional kappa value
kappa_dim = float(Model.get('kappa_dim', 1.680e-10))

# Calculate the dimensionless kappa
kappa_dimless = kappa_dim / (Hscale * Wc**2)

print(f"Dimensional kappa (kappa_dim): {kappa_dim:.3e}")
print(f"Dimensionless kappa (kappa_dimless): {kappa_dimless:.3e}")

print(f"RTv={RTv:.3e}, Om={Om:.3e}, Dm={Dm:.3e}, j0coeff={j0coeff:.3e}")

Dimensional kappa (kappa_dim): 1.680e-07
Dimensionless kappa (kappa_dimless): 2.333e+03
RTv=7.857e-01, Om=3.805e+00, Dm=1.273e+00, j0coeff=1.480e+00


In [72]:
# Domain & grid
Lx = float(Domain["Lx"]) ; Ly = float(Domain["Ly"])
nde = float(Domain["nde"])        # min element size in Wc units
Nx = max(8, int(round(Lx/nde)))
Ny = max(8, int(round(Ly/nde)))
if Nx % 2: Nx += 1
if Ny % 2: Ny += 1
x = np.linspace(0.0, Lx, Nx, endpoint=False)
y = np.linspace(0.0, Ly, Ny, endpoint=False)
X, Y = np.meshgrid(x, y, indexing='ij')

kx = 2*np.pi*fftfreq(Nx, d=Lx/Nx)
ky = 2*np.pi*fftfreq(Ny, d=Ly/Ny)
KX, KY = np.meshgrid(kx, ky, indexing='ij')
K2 = KX**2 + KY**2
K2[0,0] = 1e-30

# spacings in Wc units
dx = Lx / Nx
dy = Ly / Ny

In [73]:
# Mask H(X) + smoothed boundary δΓ ≈ |∇H|
mask_shape = Model.get('mask_shape', 'circle')
r_electrode = float(Model.get('r_electrode_Wc', 0.39*min(Lx, Ly)))
center = (float(Model.get('cx', Lx/2)), float(Model.get('cy', Ly/2)))
# was:
# sigma_smooth = min(Lx/Nx, Ly/Ny)
sigma_smooth = float(Model.get('sigma_smooth',
                               2.0*min(Lx/Nx, Ly/Ny)))  # 2 cells default for slabs


slab_x_center = float(Model.get('slab_x_center', Lx/2.0))
slab_x_width  = float(Model.get('slab_x_width',  Lx/4.0))
slab_y_center = float(Model.get('slab_y_center', Ly/2.0))
slab_y_width  = float(Model.get('slab_y_width',  Ly/4.0))


def gaussian_filter_field(f, sigma_len):
    G = np.exp(-0.5*((KX*sigma_len)**2 + (KY*sigma_len)**2))
    return np.real(ifftn(fftn(f)*G))

def grad_field(f):
    fk = fftn(f)
    fx = np.real(ifftn(1j*KX*fk))
    fy = np.real(ifftn(1j*KY*fk))
    return fx, fy

def make_mask(shape='circle', r=r_electrode, center=center,
              slab_x_center=slab_x_center, slab_x_width=slab_x_width,
              slab_y_center=slab_y_center, slab_y_width=slab_y_width):
    cx, cy = center
    XX = (X - cx); YY = (Y - cy)
    if shape == 'diamond':
        raw = (np.abs(XX) + np.abs(YY)) <= r
    elif shape == 'hex':
        s3o2 = np.sqrt(3)/2
        ax, ay = np.abs(XX), np.abs(YY)
        raw = np.maximum(ay, s3o2*ax + 0.5*ay) <= r
    elif shape == 'slab_x':
        raw = np.abs(X - slab_x_center) <= slab_x_width / 2.0
    elif shape == 'slab_y':
        raw = np.abs(Y - slab_y_center) <= slab_y_width / 2.0
    else: # circle
        raw = (XX*XX + YY*YY) <= r*r
    return raw.astype(float)

H_raw = make_mask(mask_shape)
H = gaussian_filter_field(H_raw, sigma_smooth)
Hx, Hy = grad_field(H)
delta_Gamma = np.sqrt(Hx*Hx + Hy*Hy) + 1e-14

L_iface = np.sum(delta_Gamma)*dx*dy   # should be ~ interface length (≈ 2*Ly for a vertical slab)
print(f"∫δΓ dA ≈ {L_iface:.3f} [Wc]")

# helper
wavg = lambda field, weight: float(np.sum(weight*field)/max(np.sum(weight), 1e-30))

∫δΓ dA ≈ 64.000 [Wc]


In [74]:
# Elasticity utilities

def lam_mu_from_E_nu(E, nu):
    lam = E*nu/((1+nu)*(1-2*nu))
    mu  = E/(2*(1+nu))
    return lam, mu

lam_p, mu_p = lam_mu_from_E_nu(E_p, nu_p)
lam_r, mu_r = lam_mu_from_E_nu(E_r, nu_r)
lam0, mu0 = lam_r, mu_r  # default reference = reservoir (good preconditioner if soft)

# Dimensionless versions for particle & reservoir
lam_p_d = lam_p / Hscale; mu_p_d = mu_p / Hscale
lam_r_d = lam_r / Hscale; mu_r_d = mu_r / Hscale
lam0_d  = lam0  / Hscale; mu0_d  = mu0  / Hscale


def C_iso(lam, mu):
    C = np.zeros((2,2,2,2))
    for i in range(2):
        for j in range(2):
            for k in range(2):
                for l in range(2):
                    C[i,j,k,l] = lam*(i==j)*(k==l) + mu*((i==k)*(j==l) + (i==l)*(j==k))
    return C

# >>> CHOOSE reference stiffness for the k-space operator:
if Model.get('mech_mode', 'uniform') == 'uniform':
    C0 = C_iso(lam_p_d, mu_p_d)     # <-- particle stiffness for uniform mode
else:
    C0 = C_iso(lam_r_d, mu_r_d)     # <-- reservoir as reference for heterogeneous mode

# then (re)build A and invA from C0 as before
K = np.stack((KX, KY), axis=-1)
A = np.einsum('...j,ijkl,...k->...il', K, C0, K)
A11=A[...,0,0]; A12=A[...,0,1]; A21=A[...,1,0]; A22=A[...,1,1]
detA = A11*A22 - A12*A21
mask0 = (np.abs(KX)<1e-14) & (np.abs(KY)<1e-14)
detA[mask0] = 1.0
invA = np.empty_like(A)
invA[...,0,0] = A22/detA; invA[...,0,1] = -A12/detA
invA[...,1,0] = -A21/detA; invA[...,1,1] = A11/detA


## Build stiffness tensor for isotropic medium given (lam, mu) (dimensionless)
#def C_iso(lam, mu):
#    C = np.zeros((2,2,2,2))
#    for i in range(2):
#        for j in range(2):
#            for k in range(2):
#                for l in range(2):
#                    C[i,j,k,l] = lam*(1 if i==j else 0)*(1 if k==l else 0)                                + mu*((1 if i==k else 0)*(1 if j==l else 0)                                    + (1 if i==l else 0)*(1 if j==k else 0))
#    return C
#
#C0 = C_iso(lam0_d, mu0_d)   # reference stiffness (dimensionless)

# Precompute A(k) and its inverse for the reference medium
#K = np.stack((KX, KY), axis=-1)
#A = np.einsum('...j,ijkl,...k->...il', K, C0, K)
#A11=A[...,0,0]; A12=A[...,0,1]; A21=A[...,1,0]; A22=A[...,1,1]
#detA = A11*A22 - A12*A21
#mask0 = (np.abs(KX)<1e-14) & (np.abs(KY)<1e-14)
##detA[mask0] = 1.0
#invA = np.empty_like(A)
#invA[...,0,0] = A22/detA; invA[...,0,1] = -A12/detA
#invA[...,1,0] = -A21/detA; invA[...,1,1] = A11/detA


In [75]:
def f_chem(c):
    ce = np.clip(c, 1e-12, 1.0 - 1e-12)
    base = RTv * (ce * np.log(ce) + (1 - ce) * np.log(1 - ce)) + Om * ce * (1 - ce)
    model = Model.get('fchem_model', 'regular')
    if model == 'cluster':
        alpha1 = float(Model.get('alpha1_J_per_mol', 0.0)) / (vm * sigma / Wc)
        y1 = float(Model.get('y1', 0.5))
        w1 = float(Model.get('w1', 0.1))
        base -= alpha1 * ce * (1 - ce) * np.exp(-((ce - y1)**2) / (2 * w1**2))
    elif model == 'cluster_doping':
        alpha1 = float(Model.get('alpha1_J_per_mol', 0.0)) / (vm * sigma / Wc)
        y1 = float(Model.get('y1', 0.5))
        w1 = float(Model.get('w1', 0.1))
        alpha2 = float(Model.get('alpha2_J_per_mol', 0.0)) / (vm * sigma / Wc)
        y2 = float(Model.get('y2', 0.5))
        w2 = float(Model.get('w2', 0.1))
        C_dopant = float(Model.get('C_dopant', 0.0))
        base -= alpha1 * ce * (1 - ce) * np.exp(-((ce - y1)**2) / (2 * w1**2))
        base -= C_dopant * alpha2 * ce * (1 - ce) * np.exp(-((ce - y2)**2) / (2 * w2**2))
    return base

def dfdc_chem(c):
    ce = np.clip(c, 1e-12, 1.0 - 1e-12)
    base = RTv * (np.log(ce) - np.log(1 - ce)) + Om * (1 - 2 * ce)
    model = Model.get('fchem_model', 'regular')
    if model == 'cluster':
        alpha1 = float(Model.get('alpha1_J_per_mol', 0.0)) / (vm * sigma / Wc)
        y1 = float(Model.get('y1', 0.5))
        w1 = float(Model.get('w1', 0.1))
        exp1 = np.exp(-((ce - y1)**2) / (2 * w1**2))
        base -= alpha1 * ((1 - 2 * ce) * exp1 - ce * (1 - ce) * (ce - y1) / (w1**2) * exp1)
    elif model == 'cluster_doping':
        alpha1 = float(Model.get('alpha1_J_per_mol', 0.0)) / (vm * sigma / Wc)
        y1 = float(Model.get('y1', 0.5))
        w1 = float(Model.get('w1', 0.1))
        alpha2 = float(Model.get('alpha2_J_per_mol', 0.0)) / (vm * sigma / Wc)
        y2 = float(Model.get('y2', 0.5))
        w2 = float(Model.get('w2', 0.1))
        C_dopant = float(Model.get('C_dopant', 0.0))
        exp1 = np.exp(-((ce - y1)**2) / (2 * w1**2))
        exp2 = np.exp(-((ce - y2)**2) / (2 * w2**2))
        base -= alpha1 * ((1 - 2 * ce) * exp1 - ce * (1 - ce) * (ce - y1) / (w1**2) * exp1)
        base -= C_dopant * alpha2 * ((1 - 2 * ce) * exp2 - ce * (1 - ce) * (ce - y2) / (w2**2) * exp2)
    return base

def laplace(f):
    return np.real(ifftn(-K2*fftn(f)))

# BV kinetics (dimensionless)

#def J_BV(mu):
#    eta = (mu_elec - mu) / RTv
#    return j0coeff*(np.exp(alpha*eta) - np.exp(-(1.0-alpha)*eta))

#def J_BV(mu):
#    eta = (mu_elec - mu) / RTv
#    eta_clip = float(Model.get('BV_eta_clip', 40.0))    # ~ exp(±40) ≈ 2.35e17, large but finite
#    eta = np.clip(eta, -eta_clip, eta_clip)
#    return j0coeff*(np.exp(alpha*eta) - np.exp(-(1.0-alpha)*eta))

def J_BV(mu):
    eta = (mu_elec - mu) / RTv
    eta_clip = float(Model.get('BV_eta_clip', 40.0))
    eta = np.clip(eta, -eta_clip, eta_clip)
    # J = j0*(e^{αη} - e^{-(1-α)η}) = j0 * e^{(α-(1-α))η} * 2*sinh(η/2)
    return j0coeff * (np.exp((2*alpha-1.0)*eta) * 2.0*np.sinh(0.5*eta))

In [76]:
# Chemical free energy and CH operators (dimensionless)
_eps_clip = 1e-12

def f_chem(c):
    ce = np.clip(c, _eps_clip, 1.0-_eps_clip)
    return RTv*(ce*np.log(ce) + (1-ce)*np.log(1-ce)) + Om*ce*(1.0-ce)

def dfdc_chem(c):
    ce = np.clip(c, _eps_clip, 1.0-_eps_clip)
    return RTv*(np.log(ce) - np.log(1.0-ce)) + Om*(1.0 - 2.0*ce)


def laplace(f):
    return np.real(ifftn(-K2*fftn(f)))

# BV kinetics (dimensionless)

#def J_BV(mu):
#    eta = (mu_elec - mu) / RTv
#    return j0coeff*(np.exp(alpha*eta) - np.exp(-(1.0-alpha)*eta))

#def J_BV(mu):
#    eta = (mu_elec - mu) / RTv
#    eta_clip = float(Model.get('BV_eta_clip', 40.0))    # ~ exp(±40) ≈ 2.35e17, large but finite
#    eta = np.clip(eta, -eta_clip, eta_clip)
#    return j0coeff*(np.exp(alpha*eta) - np.exp(-(1.0-alpha)*eta))

def J_BV(mu):
    eta = (mu_elec - mu) / RTv
    eta_clip = float(Model.get('BV_eta_clip', 40.0))
    eta = np.clip(eta, -eta_clip, eta_clip)
    # J = j0*(e^{αη} - e^{-(1-α)η}) = j0 * e^{(α-(1-α))η} * 2*sinh(η/2)
    return j0coeff * (np.exp((2*alpha-1.0)*eta) * 2.0*np.sinh(0.5*eta))


In [77]:
# --- Level 1: Uniform (masked) elasticity ---
# Uses constant isotropic stiffness = particle values, but masks eigenstrain to the particle.

Eps0 = np.zeros((2,2))
Eps0[0,0] = float(Model.get('e11', 0.0))
Eps0[1,1] = float(Model.get('e22', 0.0))

C_part = C_iso(lam_p_d, mu_p_d)

mask_mech_to_particle = bool(Model.get('mask_mech_to_particle', True))
eigenstrain_only_in_particle = bool(Model.get('eigenstrain_only_in_particle', True))


def strain_from_u(Ux, Uy, KX, KY):
    Uxk = fftn(Ux); Uyk = fftn(Uy)
    Exx = np.real(ifftn(1j*KX*Uxk))
    Eyy = np.real(ifftn(1j*KY*Uyk))
    Exy = np.real(ifftn(0.5j*(KX*Uyk + KY*Uxk)))
    return Exx, Eyy, Exy


def solve_elastic_uniform(c):
    # eigenstrain masked to particle if requested
    c_eff = (H * c) if eigenstrain_only_in_particle else c
    E0 = np.zeros((Nx,Ny,2,2))
    E0[...,0,0] = c_eff * Eps0[0,0]
    E0[...,1,1] = c_eff * Eps0[1,1]

    # FFT of eigenstrain comps
    E0k = np.zeros_like(E0, dtype=complex)
    for a in range(2):
        for b in range(2):
            E0k[...,a,b] = fftn(E0[...,a,b])

    # Solve for u_k with constant C_part (same as original)
    b = 1j*np.einsum('...j,ijkl,...kl->...i', K, C_part, E0k)
    u_k = np.einsum('...ij,...j->...i', invA, b)   # reuse invA built for C0 — acceptable if C0=C_part; otherwise we could rebuild.
    u_k[mask0,...] = 0.0
    Ux = np.real(ifftn(u_k[...,0])); Uy = np.real(ifftn(u_k[...,1]))

    Exx, Eyy, Exy = strain_from_u(Ux, Uy, KX, KY)

    DE = np.zeros_like(E0)
    DE[...,0,0] = Exx - E0[...,0,0]
    DE[...,1,1] = Eyy - E0[...,1,1]
    DE[...,0,1] = Exy; DE[...,1,0] = Exy

    # Stress and energies with particle stiffness
    sigma = np.einsum('ijkl,...kl->...ij', C_part, DE)
    f_el = 0.5*np.einsum('...ij,ijkl,...kl->...', DE, C_part, DE)
    mu_el = -(sigma[...,0,0]*Eps0[0,0] + sigma[...,1,1]*Eps0[1,1] + 2.0*sigma[...,0,1]*Eps0[0,1])

    if mask_mech_to_particle:
        mu_el *= H
        f_el  *= H

    return mu_el, f_el

In [78]:
# --- Level 2: Heterogeneous (particle vs reservoir) elasticity ---
# Variable isotropic stiffness fields via FFT-based equilibrium iteration.

# Spatial fields (dimensionless) for λ and μ
lam_field = lam_r_d + H*(lam_p_d - lam_r_d)
mu_field  = mu_r_d  + H*(mu_p_d  - mu_r_d)


# Helper: compute stress σ for isotropic C(x) and given strain ε and eigenstrain ε0

def stress_iso_fields(Exx, Eyy, Exy, E0xx, E0yy, E0xy, lamF, muF):
    # ε_dev = ε - ε0
    dExx = Exx - E0xx
    dEyy = Eyy - E0yy
    dExy = Exy - E0xy
    tr = dExx + dEyy
    sxx = 2*muF*dExx + lamF*tr
    syy = 2*muF*dEyy + lamF*tr
    sxy = 2*muF*dExy
    return sxx, syy, sxy


def solve_elastic_hetero(c, tol=1e-6, maxit=100, omega=0.7, verbose=False):
    # eigenstrain only inside particle by default
    c_eff = (H * c) if eigenstrain_only_in_particle else c
    E0xx = c_eff * Eps0[0,0]
    E0yy = c_eff * Eps0[1,1]
    E0xy = 0.0

    # initialize displacement u=0
    Ux = np.zeros((Nx,Ny)); Uy = np.zeros((Nx,Ny))

    # initial residual norm
    # build initial stress for u=0
    sxx, syy, sxy = stress_iso_fields(0.0, 0.0, 0.0, E0xx, E0yy, E0xy, lam_field, mu_field)
    # residual g = div σ
    sxxk = fftn(sxx); syyk = fftn(syy); sxyk = fftn(sxy)
    gkx = 1j*(KX*sxxk + KY*sxyk)
    gky = 1j*(KX*sxyk + KY*syyk)
    g0 = np.sqrt(np.mean(np.abs(gkx)**2 + np.abs(gky)**2))
    if g0 < 1e-30: g0 = 1.0

    for it in range(1, maxit+1):
        # compute strain from current u
        Exx, Eyy, Exy = strain_from_u(Ux, Uy)
        # stress with variable C(x)
        sxx, syy, sxy = stress_iso_fields(Exx, Eyy, Exy, E0xx, E0yy, E0xy, lam_field, mu_field)
        # residual g in k-space
        sxxk = fftn(sxx); syyk = fftn(syy); sxyk = fftn(sxy)
        gkx = 1j*(KX*sxxk + KY*sxyk)
        gky = 1j*(KX*sxyk + KY*syyk)
        # solve A * du_k = -g_k with reference operator invA
        rhsx = -gkx; rhsy = -gky
        dux_k = invA[...,0,0]*rhsx + invA[...,0,1]*rhsy
        duy_k = invA[...,1,0]*rhsx + invA[...,1,1]*rhsy
        dux_k[mask0] = 0.0; duy_k[mask0] = 0.0
        # relaxation update
        Ux += omega*np.real(ifftn(dux_k))
        Uy += omega*np.real(ifftn(duy_k))

        # check convergence
        res = np.sqrt(np.mean(np.abs(gkx)**2 + np.abs(gky)**2)) / g0
        if verbose and (it % 10 == 0 or it==1):
            print(f"hetero it={it:3d}, relres={res:.3e}")
        if res < tol:
            break

    # final strains
    Exx, Eyy, Exy = strain_from_u(Ux, Uy)

    # energy density and μ_el using local C(x)
    dExx = Exx - E0xx; dEyy = Eyy - E0yy; dExy = Exy - 0.0
    tr = dExx + dEyy
    # energy: 1/2 * (2 μ ε_dev:ε_dev + λ tr^2)
    f_el = 0.5*(2*mu_field*(dExx**2 + dEyy**2 + 2*dExy**2) + lam_field*(tr**2))

    # stress components for μ_el
    sxx, syy, sxy = stress_iso_fields(Exx, Eyy, Exy, E0xx, E0yy, 0.0, lam_field, mu_field)
    mu_el = -(sxx*Eps0[0,0] + syy*Eps0[1,1] + 2.0*sxy*Eps0[0,1])

    if bool(Model.get('mask_mu_el_to_particle_in_hetero', False)):
        mu_el *= H

    return mu_el, f_el

In [79]:
def divergence_of_M_grad_mu(Mc, mu):
    muk = fftn(mu)
    mux = np.real(ifftn(1j*KX*muk))
    muy = np.real(ifftn(1j*KY*muk))
    jx = Mc*mux
    jy = Mc*muy
    return np.real(ifftn(1j*KX*fftn(jx) + 1j*KY*fftn(jy)))


mech_mode = Model.get('mech_mode', 'uniform')  # 'uniform' or 'hetero'

def mech_mu_f(c):
    if mech_mode == 'hetero':
        return solve_elastic_hetero(c, tol=float(Model.get('hetero_tol',1e-6)),
                                       maxit=int(Model.get('hetero_maxit',100)),
                                       omega=float(Model.get('hetero_omega',0.7)),
                                       verbose=bool(Model.get('hetero_verbose', False)))
    else:
        return solve_elastic_uniform(c)

# Ak = 1.0 + float(Interval['timestep'])*Mlin*(K2**2) # Original calculation
# Modified Ak calculation using Dm and kappa_dimless separately
Ak = 1.0 + float(Interval['timestep']) * (Dm * K2 + kappa_dimless * K2**2)

def step_CH(c, dt):
    mu_el, f_el = mech_mu_f(c)
    # The chemical potential calculation remains the same
    mu = dfdc_chem(c) - laplace(c) + mu_el
    Mc = Dm*(H * c * (1.0 - c))   # diffusion only in particle
    div_term = divergence_of_M_grad_mu(Mc, mu)
    J = J_BV(mu)
    Rsrc = J * delta_Gamma
    rhs = c + dt*(div_term + Rsrc)
    # The spectral update uses the new Ak
    c_new = np.real(ifftn(fftn(rhs)/Ak))
    # impose reservoir composition outside
    cout = float(Model.get('c_outside', 1.0))
    c_new = H*c_new + (1.0 - H)*cout
    return np.clip(c_new, 1e-8, 1.0-1e-8), {'mu':mu, 'mu_el':mu_el, 'f_el':f_el, 'J':J}

In [80]:
# Time integration & outputs
save_frames   = bool(Model.get('save_frames', True))
frame_every   = int(Model.get('frame_every', 50))
frame_dir     = str(Model.get('frame_dir', 'frames'))
cmap_name     = str(Model.get('cmap', 'viridis'))

if save_frames and not os.path.isdir(frame_dir):
    os.makedirs(frame_dir, exist_ok=True)

Model["slab_y_center"] = 0.55*Ly
Model["slab_y_width"]  = 0.5*Ly      # not so wide it touches y=0 or y=Ly

def save_concentration_frame(it, t_dimless, c, H, Lx, Ly):
    extent = [0, Lx, 0, Ly]
    plt.figure(figsize=(5.2,4.2))
    im = plt.imshow(c.T, origin='lower', extent=extent, vmin=0, vmax=1, cmap=cmap_name)
    plt.contour(H.T, levels=[0.5], colors='red', linewidths=0.8, origin='lower', extent=extent)
    plt.colorbar(im, fraction=0.046)
    plt.title(f"c, step {it} (t = {t_dimless*tc:.3e} s)")
    plt.xlabel("x [Wc]"); plt.ylabel("y [Wc]")
    plt.tight_layout()
    fname = os.path.join(frame_dir, f"frame_{it:06d}.png")
    plt.savefig(fname, dpi=150)
    plt.close()

# Initial condition (inside vs outside), smoothed by H
c_inside  = float(Model.get('c_inside', 0.10))
c_outside = float(Model.get('c_outside', 1.00))

c = c_inside*H + c_outside*(1.0 - H)
# small noise
rng = np.random.default_rng(int(Model.get('seed', 0)))
c = c + (0.01*(rng.random((Nx,Ny)) - 0.5))
c = np.clip(c, 1e-3, 1.0-1e-3)

# Run
nsteps = 1000 #int(Model.get('nsteps', 2000))
dt = float(Interval['timestep'])

print(f"Grid: {Nx}x{Ny}, L=({Lx},{Ly}) Wc; dt={dt:.3e}; mode={mech_mode}")

# logs
times_s = []
current_A = []
c_part_hist = []
c_dom_hist = []
V_hist = []
V_ref = float(Model.get('V_ref', 3.45))

Hscale_local = Hscale  # for clarity

for it in range(1, nsteps+1):
    c, info = step_CH(c, dt)
    t_dim = it*dt

    if save_frames and (it % frame_every == 0 or it == 1):
        save_concentration_frame(it, t_dim, c, H, Lx, Ly)

    times_s.append(t_dim*tc)
    # total current via smoothed boundary integral
    J = info['J']
    Ssum = np.sum(J * delta_Gamma) * dx * dy
    I_A = Iconv * Ssum
    current_A.append(I_A)

    c_part_hist.append(wavg(c, H))
    c_dom_hist.append(float(c.mean()))

    mu_avg_dim = wavg(info['mu'], H)
    V_proxy = V_ref + (mu_avg_dim * Hscale_local * vm / Fconst) - DeltaPhi
    V_hist.append(V_proxy)

    # Add logging for div_term and Rsrc
    div_term = divergence_of_M_grad_mu(Dm*(H * c * (1.0 - c)), info['mu']) # Recalculate div_term
    Rsrc = info['J'] * delta_Gamma # Rsrc was already calculated in step_CH, but recalculating for logging is fine

    if it % max(10, frame_every) == 0 or it==1:
        fchem = float(f_chem(c).mean()); fel=float(info['f_el'].mean()); Jm = J*delta_Gamma
        print(f"step {it:5d} <c>={c.mean():.4f} <f_chem>={fchem:.3e} <f_el>={fel:.3e} I~{(Jm.sum()*(Lx/Nx)*(Ly/Ny)):.3e} |div_term|~{np.mean(np.abs(div_term)):.3e} |Rsrc|~{np.mean(np.abs(Rsrc)):.3e}")


# Save checkpoint
np.savez('lfp_spectral_mech_checkpoint.npz',
         c=c, Nx=Nx, Ny=Ny, Lx=Lx, Ly=Ly, RTv=RTv, Om=Om, Dm=Dm,
         lam_p_d=lam_p_d, mu_p_d=mu_p_d, lam_r_d=lam_r_d, mu_r_d=mu_r_d,
         H=H, delta_Gamma=delta_Gamma, mech_mode=mech_mode)
print('Saved: lfp_spectral_mech_checkpoint.npz')

# Save logs
log = np.column_stack([times_s, current_A, c_part_hist, c_dom_hist, V_hist])
np.savetxt('iv_log_ch.csv', log, delimiter=',', header='time_s,current_A,c_particle,c_domain,voltage_V', comments='')
print('Saved: iv_log_ch.csv')

Grid: 214x428, L=(32.0,64.0) Wc; dt=1.000e-04; mode=uniform
step     1 <c>=0.5559 <f_chem>=5.176e-02 <f_el>=5.005e-02 I~-2.223e+03 |div_term|~2.766e+01 |Rsrc|~1.086e+00
step    50 <c>=0.5864 <f_chem>=6.329e-02 <f_el>=3.417e-01 I~-3.983e+04 |div_term|~5.258e-01 |Rsrc|~1.945e+01
step   100 <c>=0.5900 <f_chem>=6.513e-02 <f_el>=3.785e-01 I~-4.148e+04 |div_term|~4.011e-01 |Rsrc|~2.025e+01
step   150 <c>=0.5913 <f_chem>=6.555e-02 <f_el>=3.951e-01 I~-4.207e+04 |div_term|~3.639e-01 |Rsrc|~2.054e+01
step   200 <c>=0.5916 <f_chem>=6.531e-02 <f_el>=4.037e-01 I~-4.237e+04 |div_term|~3.465e-01 |Rsrc|~2.069e+01
step   250 <c>=0.5915 <f_chem>=6.476e-02 <f_el>=4.086e-01 I~-4.256e+04 |div_term|~3.368e-01 |Rsrc|~2.078e+01
step   300 <c>=0.5911 <f_chem>=6.404e-02 <f_el>=4.119e-01 I~-4.270e+04 |div_term|~3.304e-01 |Rsrc|~2.085e+01
step   350 <c>=0.5905 <f_chem>=6.323e-02 <f_el>=4.143e-01 I~-4.281e+04 |div_term|~3.257e-01 |Rsrc|~2.090e+01
step   400 <c>=0.5899 <f_chem>=6.237e-02 <f_el>=4.164e-01 I~-4.291e+

In [81]:
# Quick analysis plots
plt.figure(figsize=(6,3.8)); plt.plot(times_s, current_A, 'k-'); plt.xlabel('time [s]'); plt.ylabel('current I [A]'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('current_vs_time_ch.png', dpi=150)
plt.figure(figsize=(6,3.8)); plt.plot(c_part_hist, V_hist, 'b.-'); plt.xlabel('particle-avg conc'); plt.ylabel('voltage [V]'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('voltage_vs_concentration_ch.png', dpi=150)
plt.figure(figsize=(6,3.8)); plt.plot(c_part_hist, current_A, 'r.-'); plt.xlabel('particle-avg conc'); plt.ylabel('current [A]'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('current_vs_concentration_ch.png', dpi=150)
print('Saved: current_vs_time.png, voltage_vs_concentration.png, current_vs_concentration.png')


Saved: current_vs_time.png, voltage_vs_concentration.png, current_vs_concentration.png


In [82]:
"""
plot_fields.py — Plot fields from spectral LFP intercalation checkpoints.

Works with either of these checkpoints:
  * lfp_spectral_mech_checkpoint.npz (from the new notebook)
  * ch_spectral_fen_checkpoint.npz   (from the original script)

It reconstructs the chemical potential μ, elastic contribution μ_el, elastic
energy density f_el, chemical energy density f_chem, and gradient energy
density f_grad; and overlays the particle boundary (H=0.5) on the plots.

Usage:
  python plot_fields.py [checkpoint.npz] [config.py]

Defaults:
  checkpoint.npz := 'lfp_spectral_mech_checkpoint.npz' (falls back to
                    'ch_spectral_fen_checkpoint.npz' if not found)
  config.py      := 'config_mech.py' (falls back to 'config 4.py' and
                    then 'config.py' if not found)

Outputs:
  analysis_fields.png
  (if heterogeneous mechanics) stiffness_fields.png
"""
from __future__ import annotations
import sys, os, importlib.util
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fftn, ifftn, fftfreq

# -------- utilities --------

def import_config(preferred: str|None = None):
    candidates = []
    if preferred is not None:
        candidates.append(preferred)
    candidates += ['config_mech.py', 'config 4.py', 'config.py']
    for fn in candidates:
        if os.path.exists(fn):
            spec = importlib.util.spec_from_file_location('cfg_mod', os.path.abspath(fn))
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod, fn
    raise FileNotFoundError("No config file found (tried: config_mech.py, config 4.py, config.py)")


def load_checkpoint(preferred: str|None = None):
    candidates = []
    if preferred is not None:
        candidates.append(preferred)
    candidates += ['lfp_spectral_mech_checkpoint.npz', 'ch_spectral_fen_checkpoint.npz']
    for fn in candidates:
        if os.path.exists(fn):
            return np.load(fn, allow_pickle=True), fn
    raise FileNotFoundError("No checkpoint found (tried lfp_spectral_mech_checkpoint.npz, ch_spectral_fen_checkpoint.npz)")


def C_iso(lam, mu):
    C = np.zeros((2,2,2,2))
    for i in range(2):
        for j in range(2):
            for k in range(2):
                for l in range(2):
                    C[i,j,k,l] = lam*(1 if i==j else 0)*(1 if k==l else 0) \
                               + mu*((1 if i==k else 0)*(1 if j==l else 0) \
                                   + (1 if i==l else 0)*(1 if j==k else 0))
    return C


def build_k_operators(Nx, Ny, Lx, Ly):
    kx = 2*np.pi*fftfreq(Nx, d=Lx/Nx)
    ky = 2*np.pi*fftfreq(Ny, d=Ly/Ny)
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    K2 = KX**2 + KY**2
    K2[0,0] = 1e-30
    return KX, KY, K2


def strain_from_u(Ux, Uy, KX, KY):
    Uxk = fftn(Ux); Uyk = fftn(Uy)
    Exx = np.real(ifftn(1j*KX*Uxk))
    Eyy = np.real(ifftn(1j*KY*Uyk))
    Exy = np.real(ifftn(0.5j*(KX*Uyk + KY*Uxk)))
    return Exx, Eyy, Exy


def laplace(f, K2):
    return np.real(ifftn(-K2*fftn(f)))

# -------- main --------

def main():
    ckpt_arg = sys.argv[1] if len(sys.argv) >= 2 and sys.argv[1].endswith('.npz') else None
    cfg_arg  = sys.argv[2] if len(sys.argv) >= 3 and sys.argv[2].endswith('.py')  else None

    data, ckpt_fn = load_checkpoint(ckpt_arg)
    cfg, cfg_fn   = import_config(cfg_arg)

    print(f"Using checkpoint: {ckpt_fn}")
    print(f"Using config   : {cfg_fn}")

    # --- pull grid & scales from checkpoint ---
    c  = data['c']
    Nx = int(data['Nx']); Ny = int(data['Ny'])
    Lx = float(data['Lx']); Ly = float(data['Ly'])

    RTv = float(data['RTv']); Om = float(data['Om'])
    H   = data['H'] if 'H' in data.files else None

    # mechanical mode and stiffness info
    mech_mode = str(data['mech_mode']) if 'mech_mode' in data.files else getattr(cfg, 'Model').get('mech_mode','uniform')

    if 'lam_d' in data.files and 'mu_d' in data.files:
        # Legacy: single uniform stiffness
        lam_p_d = float(data['lam_d']); mu_p_d = float(data['mu_d'])
        lam_r_d, mu_r_d = lam_p_d, mu_p_d
    else:
        lam_p_d = float(data['lam_p_d']); mu_p_d = float(data['mu_p_d'])
        lam_r_d = float(data['lam_r_d']); mu_r_d = float(data['mu_r_d'])

    # eigenstrain (from config)
    Model = getattr(cfg, 'Model')
    e11 = float(Model.get('e11', 0.0)); e22 = float(Model.get('e22', 0.0))
    Eps0 = np.zeros((2,2)); Eps0[0,0]=e11; Eps0[1,1]=e22

    # mask defaults if missing in legacy file
    if H is None:
        print("No H in checkpoint; reconstructing a trivial mask (all ones) for plotting…")
        H = np.ones((Nx,Ny))

    KX, KY, K2 = build_k_operators(Nx, Ny, Lx, Ly)

    # chemical free-energy pieces
    def f_chem(c):
        ce = np.clip(c, 1e-12, 1-1e-12)
        return RTv*(ce*np.log(ce)+(1-ce)*np.log(1-ce)) + Om*ce*(1-ce)

    def dfdc_chem(c):
        ce = np.clip(c, 1e-12, 1-1e-12)
        return RTv*(np.log(ce)-np.log(1-ce)) + Om*(1-2*ce)

    # reference operator in k-space
    if mech_mode == 'uniform':
        lam0_d, mu0_d = lam_p_d, mu_p_d   # use particle as reference
    else:
        lam0_d, mu0_d = lam_r_d, mu_r_d   # reservoir as reference
    C0 = C_iso(lam0_d, mu0_d)

    K = np.stack((KX, KY), axis=-1)
    A = np.einsum('...j,ijkl,...k->...il', K, C0, K)
    A11=A[...,0,0]; A12=A[...,0,1]; A21=A[...,1,0]; A22=A[...,1,1]
    detA = A11*A22 - A12*A21
    mask0 = (np.abs(KX)<1e-14) & (np.abs(KY)<1e-14)
    detA[mask0] = 1.0
    invA = np.empty_like(A)
    invA[...,0,0] = A22/detA; invA[...,0,1] = -A12/detA
    invA[...,1,0] = -A21/detA; invA[...,1,1] = A11/detA

    # helpers for mechanics
    def solve_uniform_mu_el(c):
        C_part = C_iso(lam_p_d, mu_p_d)
        c_eff = H * c  # eigenstrain only inside
        E0 = np.zeros((Nx,Ny,2,2))
        E0[...,0,0] = c_eff*Eps0[0,0]
        E0[...,1,1] = c_eff*Eps0[1,1]
        E0k = np.zeros_like(E0, dtype=complex)
        for a in range(2):
            for b in range(2):
                E0k[...,a,b] = fftn(E0[...,a,b])
        b = 1j*np.einsum('...j,ijkl,...kl->...i', K, C_part, E0k)
        u_k = np.einsum('...ij,...j->...i', invA, b)
        u_k[mask0,...]=0.0
        Ux = np.real(ifftn(u_k[...,0])); Uy = np.real(ifftn(u_k[...,1]))
        Exx,Eyy,Exy = strain_from_u(Ux,Uy,KX,KY)
        DE = np.zeros_like(E0)
        DE[...,0,0]=Exx-E0[...,0,0]; DE[...,1,1]=Eyy-E0[...,1,1]
        DE[...,0,1]=Exy; DE[...,1,0]=Exy
        sigma = np.einsum('ijkl,...kl->...ij', C_part, DE)
        f_el = 0.5*np.einsum('...ij,ijkl,...kl->...', DE, C_part, DE)
        mu_el = -(sigma[...,0,0]*Eps0[0,0] + sigma[...,1,1]*Eps0[1,1] + 2.0*sigma[...,0,1]*Eps0[0,1])
        mu_el *= H; f_el *= H
        return mu_el, f_el

    def solve_hetero_mu_el(c, tol=1e-6, maxit=100, omega=0.7):
        lamF = lam_r_d + H*(lam_p_d - lam_r_d)
        muF  = mu_r_d  + H*(mu_p_d  - mu_r_d)
        c_eff = H*c
        E0xx = c_eff*Eps0[0,0]; E0yy = c_eff*Eps0[1,1]; E0xy = 0.0
        Ux = np.zeros((Nx,Ny)); Uy = np.zeros((Nx,Ny))
        def stress(Exx,Eyy,Exy):
            dExx=Exx-E0xx; dEyy=Eyy-E0yy; dExy=Exy-E0xy
            tr = dExx+dEyy
            sxx = 2*muF*dExx + lamF*tr
            syy = 2*muF*dEyy + lamF*tr
            sxy = 2*muF*dExy
            return sxx, syy, sxy
        # initial residual norm
        sxx,syy,sxy = stress(0.0,0.0,0.0)
        sxxk, syyk, sxyk = fftn(sxx), fftn(syy), fftn(sxy)
        gkx = 1j*(KX*sxxk + KY*sxyk)
        gky = 1j*(KX*sxyk + KY*syyk)
        g0 = np.sqrt(np.mean(np.abs(gkx)**2 + np.abs(gky)**2)) or 1.0
        for it in range(maxit):
            Exx,Eyy,Exy = strain_from_u(Ux,Uy,KX,KY)
            sxx,syy,sxy = stress(Exx,Eyy,Exy)
            sxxk, syyk, sxyk = fftn(sxx), fftn(syy), fftn(sxy)
            gkx = 1j*(KX*sxxk + KY*sxyk)
            gky = 1j*(KX*sxyk + KY*syyk)
            rhsx, rhsy = -gkx, -gky
            dux_k = invA[...,0,0]*rhsx + invA[...,0,1]*rhsy
            duy_k = invA[...,1,0]*rhsx + invA[...,1,1]*rhsy
            dux_k[(np.abs(KX)<1e-14)&(np.abs(KY)<1e-14)] = 0.0
            duy_k[(np.abs(KX)<1e-14)&(np.abs(KY)<1e-14)] = 0.0
            Ux += omega*np.real(ifftn(dux_k))
            Uy += omega*np.real(ifftn(duy_k))
            res = np.sqrt(np.mean(np.abs(gkx)**2 + np.abs(gky)**2))/g0
            if res < tol:
                break
        Exx,Eyy,Exy = strain_from_u(Ux,Uy,KX,KY)
        dExx=Exx-E0xx; dEyy=Eyy-E0yy; dExy=Exy
        tr=dExx+dEyy
        f_el = 0.5*(2*muF*(dExx**2 + dEyy**2 + 2*dExy**2) + lamF*(tr**2))
        sxx,syy,sxy = stress(Exx,Eyy,Exy)
        mu_el = -(sxx*Eps0[0,0] + syy*Eps0[1,1] + 2.0*sxy*Eps0[0,1])
        return mu_el, f_el, lamF, muF

    if mech_mode == 'hetero':
        mu_el, f_el, lamF, muF = solve_hetero_mu_el(c)
    else:
        mu_el, f_el = solve_uniform_mu_el(c)
        lamF = lam_p_d*np.ones_like(c); muF = mu_p_d*np.ones_like(c)

    mu = dfdc_chem(c) - laplace(c, K2) + mu_el

    # gradient energy density
    ck = fftn(c)
    dcx = np.real(ifftn(1j*KX*ck)); dcy = np.real(ifftn(1j*KY*ck))
    f_grad = 0.5*(dcx*dcx + dcy*dcy)

    # ---- plots ----
    extent = [0, Lx, 0, Ly]
    fig,axs = plt.subplots(2,3,figsize=(11,7))
    im = axs[0,0].imshow(c.T, origin='lower', extent=extent); axs[0,0].set_title('c')
    plt.colorbar(im, ax=axs[0,0]); axs[0,0].contour(H.T, levels=[0.5], colors='red', linewidths=0.8, extent=extent)

    im = axs[0,1].imshow(mu.T, origin='lower', extent=extent); axs[0,1].set_title('μ')
    plt.colorbar(im, ax=axs[0,1])

    im = axs[0,2].imshow(mu_el.T, origin='lower', extent=extent); axs[0,2].set_title('μ_el')
    plt.colorbar(im, ax=axs[0,2])

    im = axs[1,0].imshow(f_el.T, origin='lower', extent=extent); axs[1,0].set_title('f_el')
    plt.colorbar(im, ax=axs[1,0])

    im = axs[1,1].imshow(f_chem(c).T, origin='lower', extent=extent); axs[1,1].set_title('f_chem')
    plt.colorbar(im, ax=axs[1,1])

    im = axs[1,2].imshow(f_grad.T, origin='lower', extent=extent); axs[1,2].set_title('f_grad')
    plt.colorbar(im, ax=axs[1,2])

    for ax in axs.ravel():
        ax.set_xlabel('x [Wc]'); ax.set_ylabel('y [Wc]')
    plt.tight_layout()
    plt.savefig('analysis_fields_ch.png', dpi=150)
    print('Saved: analysis_fields_ch.png')

    if mech_mode == 'hetero':
        fig2,axs2 = plt.subplots(1,2,figsize=(9,3.6))
        im = axs2[0].imshow(lamF.T, origin='lower', extent=extent)
        axs2[0].set_title('λ(x) (dimless)'); plt.colorbar(im, ax=axs2[0])
        im = axs2[1].imshow(muF.T, origin='lower', extent=extent)
        axs2[1].set_title('μ(x) (dimless)'); plt.colorbar(im, ax=axs2[1])
        for ax in axs2:
            ax.contour(H.T, levels=[0.5], colors='red', linewidths=0.8, extent=extent)
            ax.set_xlabel('x [Wc]'); ax.set_ylabel('y [Wc]')
        plt.tight_layout(); plt.savefig('stiffness_fields.png', dpi=150)
        print('Saved: stiffness_fields.png')

if __name__ == '__main__':
    main()

Using checkpoint: lfp_spectral_mech_checkpoint.npz
Using config   : config_mech.py
Saved: analysis_fields_ch.png


In [88]:
import numpy as np
import matplotlib.pyplot as plt

def wrap_coord(val, L): return val % L

def line_profile_x(c, y0, Ly):
    Nx, Ny = c.shape
    y0w = wrap_coord(y0, Ly)
    jy = (y0w/Ly) * Ny
    j0 = int(np.floor(jy)); j1 = (j0 + 1) % Ny
    t = float(jy - j0)
    return (1.0 - t) * c[:, j0] + t * c[:, j1]

def line_profile_y(c, x0, Lx):
    Nx, Ny = c.shape
    x0w = wrap_coord(x0, Lx)
    ix = (x0w/Lx) * Nx
    i0 = int(np.floor(ix)); i1 = (i0 + 1) % Nx
    s = float(ix - i0)
    return (1.0 - s) * c[i0, :] + s * c[i1, :]

# Choose positions (Wc units)
y_positions = [0.5*Ly] #[0.25*Ly, 0.5*Ly, 0.75*Ly]
x_positions = [0.5*Lx] #[0.33*Lx, 0.66*Lx]

x = np.linspace(0, Lx, Nx, endpoint=False)
y = np.linspace(0, Ly, Ny, endpoint=False)


# c(x, y0)
plt.figure(figsize=(6.4, 3.8))
for y0 in y_positions:
    plt.plot(x, line_profile_x(c, y0, Ly), label=f"y={y0:.3f}")
plt.xlabel("x [Wc]"); plt.ylabel("c(x, y0)"); plt.grid(True, alpha=0.3)
plt.legend(); plt.tight_layout(); #plt.show()
plt.savefig('lineprofile_x.png', dpi=150)
print('Saved: lineprofile_x.png')
# c(x0, y)
plt.figure(figsize=(6.4, 3.8))
for x0 in x_positions:
    plt.plot(y, line_profile_y(c, x0, Lx), label=f"x={x0:.3f}")
plt.xlabel("y [Wc]"); plt.ylabel("c(x0, y)"); plt.grid(True, alpha=0.3)
plt.legend(); plt.tight_layout(); #plt.show()
plt.savefig('lineprofile_y.png', dpi=150)
print('Saved: lineprofile_y.png')

Saved: lineprofile_x.png
Saved: lineprofile_y.png


# adding PFC

In [89]:
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.spatial import Voronoi
from dataclasses import dataclass
import numpy as np
from scipy.spatial import cKDTree
from scipy.ndimage import distance_transform_edt
# -----------------------
# Parameters
# -----------------------

@dataclass
class Params:
    nx: int = Nx
    ny: int = Ny
    Lx: float = Lx   # domain size in arbitrary units
    Ly: float = Ly
    dx: float = 1.0
#    r_electrode: float = 80.0  # electrode radius (in grid units)
    kappa: float = 1.0
    gamma: float = 1.0
    RT: float = 1.0
    Om: float = 2.0
    xi: float = 1.0
    # PFC control parameter r(x)
    r_inside: float = -0.2
    r_outside: float = 0.2
    # Add r values for FP (c=0) and LFP (c=1)
    r_FP: float = -0.2 # r value for FePO4 (c=0), favors crystal
    r_LFP: float = -0.3 # r value for LiFePO4 (c=1), favors crystal
    # Lattice parameters from Table I (normalized)
    a0: float = 4.788
    a1_fp: float = 9.821
    a3_fp: float = 4.788
    a1_lfp: float = 10.334
    a3_lfp: float = 4.693

    @property
    def alpha_fp(self):
        return self.a1_fp / self.a0
    @property
    def beta_fp(self):
        return self.a3_fp / self.a0
    @property
    def alpha_lfp(self):
        return self.a1_lfp / self.a0
    @property
    def beta_lfp(self):
        return self.a3_lfp / self.a0


def init_poly_voronoi_simple(
    p, mask, n_grains=16, amp=0.30, k=1.0,
    lattice='hex',   # 'hex' (3-mode) or 'rect' (2-mode)
    band_px=1,       # width of taper band along grain boundaries (0 = off)
    seed=0
):
    """
    Minimal polycrystal initializer with random Voronoi grains.

    Returns
    -------
    psi0   : (nx, ny) float   analytic seed
    labels : (nx, ny) int     -1 outside mask, 0..G-1 inside
    seeds  : (G, 2) int       (i, j) seed coords
    thetas : (G,) float       random grain orientations in [0, π)
    """
    rng = np.random.default_rng(seed)

    # 1) pick random seeds inside mask
    ii, jj = np.nonzero(mask)
    if len(ii) == 0:
        raise ValueError("Mask is empty; adjust r_electrode or mask generation.")
    G = int(n_grains)
    if G > len(ii):
        G = len(ii)
    idx = rng.choice(len(ii), size=G, replace=False)
    seeds = np.c_[ii[idx], jj[idx]]  # (G, 2) in (i, j)

    # 2) Voronoi tessellation via nearest seed
    labels = -np.ones(mask.shape, dtype=int)
    kd = cKDTree(seeds.astype(float))
    pts = np.c_[ii, jj].astype(float)
    _, lab = kd.query(pts)
    labels[mask] = lab

    # 3) random orientations
    thetas = rng.uniform(0.0, np.pi, size=G)

    # 4) build analytic lattice per grain
    X, Y = np.meshgrid(np.arange(p.nx), np.arange(p.ny), indexing='ij')
    psi0 = np.zeros(mask.shape, dtype=float)

    if lattice == 'hex':
        # 3 modes at 60°: θ, θ+π/3, θ+2π/3
        for g in range(G):
            gi = (labels == g)
            if not gi.any():
                continue
            th = float(thetas[g])
            a = th + np.array([0.0, np.pi/3.0, 2.0*np.pi/3.0])
            field = (
                np.cos(k*(np.cos(a[0])*X + np.sin(a[0])*Y)) +
                np.cos(k*(np.cos(a[1])*X + np.sin(a[1])*Y)) +
                np.cos(k*(np.cos(a[2])*X + np.sin(a[2])*Y))
            ) / 3.0
            psi0[gi] = amp * field[gi]
    else:
        # rectangular: 2 modes (orthogonal)
        for g in range(G):
            gi = (labels == g)
            if not gi.any():
                continue
            th = float(thetas[g])
            field = (
                np.cos(k*( np.cos(th)*X + np.sin(th)*Y)) +
                np.cos(k*(-np.sin(th)*X + np.cos(th)*Y))
            ) / 2.0
            psi0[gi] = amp * field[gi]

    # zero in reservoir
    psi0[~mask] = 0.0

    # 5) (optional) narrow taper band along grain boundaries
    if band_px and band_px > 0:
        L = labels
        edges = (
            ((np.roll(L, 1, 0) != L) |
             (np.roll(L,-1, 0) != L) |
             (np.roll(L, 1, 1) != L) |
             (np.roll(L,-1, 1) != L)) &
            (L >= 0)
        )
        band = (distance_transform_edt(~edges) <= float(band_px)) & mask
        psi0[band] *= 0.6  # small taper

    return psi0, labels, seeds, thetas

def apply_G2(u, c, p):
    L  = anisotropic_laplacian_c
    Lu  = L(u,  c, p.alpha_fp, p.beta_fp, p.alpha_lfp, p.beta_lfp, p.dx, p.xi)
    L2u = L(Lu, c, p.alpha_fp, p.beta_fp, p.alpha_lfp, p.beta_lfp, p.dx, p.xi)
    return u + 2.0*Lu + L2u

def free_energy_pfc(psi, c, r_field, p):
    G2psi = apply_G2(psi, c, p)
    # integrate per pixel area; with dx=dy=1 this is just sum()
    term_quad = 0.5 * (r_field * psi**2 + psi * G2psi)
    term_quar = 0.25 * psi**4
    # Note: In some formulations, gamma multiplies the entire free energy density,
    # not just the integral. Adjust if needed based on the specific model.
    return float(p.gamma * np.sum(term_quad + term_quar))


# Define the pfc_step_spectral function directly in the notebook
# This version implements the spectral solver logic
def pfc_step_spectral(psi, c, r_field, p, cache, mask, dt, psi_clip=1.8):
    """Performs one spectral step for the PFC equation.

    Args:
        psi: The phase field.
        c: The concentration field.
        r_field: The spatially varying r parameter.
        p: The parameters object.
        cache: The FFTCache object.
        mask: The mask defining the active region.
        dt: The time step size.
        psi_clip: Value to clip psi to.

    Returns:
        The updated psi field.
    """
    # Ensure psi is float64 and C-contiguous
    psi = np.asarray(psi, dtype=np.float64, order='C').copy()

    # Transform psi to Fourier space
    psik = fftn(psi)

    # Calculate the right-hand side in Fourier space
    # PFC equation: d(psi)/dt = (r - (1-q^2)^2) psi - psi^3
    # In Fourier space: d(psik)/dt = fft(r*psi) - (1+K2)**2 * psik - fft(psi^3)

    psi_cubed = psi**3
    psi_cubed_k = fftn(psi_cubed)

    # r_field is spatially varying, so r*psi needs to be calculated in real space and then FFT'd
    r_psi_k = fftn(r_field * psi)

    # The spectral operator (1+K^2)^2 comes from (1 - Laplace)^2.
    # If using an anisotropic Laplacian L_c, the operator is (1 - L_c)^2 in k-space.
    # The current implementation uses an isotropic placeholder for L_c, resulting in (1+K2)**2.
    # If anisotropic_laplacian_c were fully implemented with k-space operator K_c(k),
    # this line would be: spectral_operator = (1.0 - K_c(k, c_field))**2  -- but K_c(k, c) is complex.
    # For now, stick to the isotropic spectral operator based on K2.
    spectral_operator = (1.0 + cache.K2)**2


    # Linear term in Fourier space: fft(r*psi) - (1+K2)**2 * psik
    linear_term_k = r_psi_k - spectral_operator * psik

    # Nonlinear term in Fourier space: -fft(psi^3)
    nonlinear_term_k = -fft(psi_cubed) # Use fft from numpy.fft

    # Forward Euler time integration in Fourier space
    psik_new = psik + dt * (linear_term_k + nonlinear_term_k)

    # Transform back to real space
    psi_new = np.real(ifftn(psik_new))

    # Apply mask and clipping
    # Note: boolean_mask should be used here consistently, not the smoothed mask 'mask'
    if mask is not None: # Assuming 'mask' here is the boolean mask
        psi_new[~mask] = 0.0 # Set to zero outside the mask

    # Apply clipping
    np.clip(psi_new, -psi_clip, psi_clip, out=psi_new)

    # Optional: Handle enforce_amorphous if needed. This is complex in spectral space.
    # For this subtask, we will skip the enforce_amorphous logic in the spectral step.

    if not np.isfinite(psi_new).all():
        raise FloatingPointError('PFC spectral step produced NaN/Inf; lower dt.')

    return psi_new

In [90]:
import numpy as np

# Load the checkpoint file
checkpoint_data = np.load('lfp_spectral_mech_checkpoint.npz', allow_pickle=True)

# Print the names of the variables saved in the checkpoint
print("Variables in the checkpoint file:", checkpoint_data.files)

# You can access individual variables like this:
c = checkpoint_data['c']
mask = checkpoint_data['H'] # 'H' seems to be the mask in this checkpoint
# print("\nShape of 'c':", c.shape)
# print("Shape of 'H':", H.shape)

# If you were expecting 'psi' and 'r_field', they might not be in this specific checkpoint.
# The plot_fields.py script reconstructs some fields or expects a different checkpoint format.

Variables in the checkpoint file: ['c', 'Nx', 'Ny', 'Lx', 'Ly', 'RTv', 'Om', 'Dm', 'lam_p_d', 'mu_p_d', 'lam_r_d', 'mu_r_d', 'H', 'delta_Gamma', 'mech_mode']


# Task
Explain the error in the selected code, and if possible, fix the error and incorporate the changes into the existing code. Otherwise, try to diagnose the error. The goal is to couple the Cahn-Hilliard and Phase Field Crystal models, writing the configuration to the file "config_mech.py". The code should also be able to read the variables c, psi, r_field, and mask from a checkpoint file.

## Define pfc functions

### Subtask:
Add the Python code that defines the `pfc_step_spectral` function and any other required helper functions for the PFC model (e.g., `plot_fields` if it's part of the PFC implementation, `build_fft_cache_from_params`, `pfc_relax_stable`).


**Reasoning**:
The subtask requires defining several helper functions for the PFC model that are missing in the current notebook state, specifically `anisotropic_laplacian_c`, `build_fft_cache_from_params`, `plot_fields`, and `pfc_relax_stable`. I will define these functions in a code block.



In [91]:
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.spatial import Voronoi
from dataclasses import dataclass
import numpy as np
from scipy.spatial import cKDTree
from scipy.ndimage import distance_transform_edt
from numpy.fft import fftn, ifftn, fftfreq

# Define anisotropic_laplacian_c based on the structure implied by apply_G2
# This is a placeholder/simplified version based on common PFC structures.
# The actual implementation might need more details from the original source.
def anisotropic_laplacian_c(u, c, alpha_fp, beta_fp, alpha_lfp, beta_lfp, dx, xi):
    """
    Placeholder for anisotropic Laplacian dependent on concentration c.
    Needs actual implementation based on the specific PFC model being used.
    This simplified version assumes a spatially varying but isotropic Laplacian for now.
    """
    # Example: Simple Laplacian scaled by a concentration-dependent factor
    # In a real PFC model, this would involve the reciprocal lattice vectors
    # and their dependence on the lattice parameter which changes with c.
    # For a proper anisotropic Laplacian, one would need to implement
    # K_c(k) = \sum_i \alpha_i(c) * exp(i k . a_i(c)) as the spectral filter.
    # For now, let's use a simple diffusion-like term.
    uk = fftn(u)
    # This is a placeholder, needs the actual k-space operator for anisotropic Laplacian
    # based on the alpha/beta parameters and concentration 'c'.
    # Since the full anisotropic logic is complex and not provided, we'll use
    # an isotropic placeholder based on K2. The coefficients (alpha, beta)
    # would typically define the k-space operator.
    # A proper implementation requires understanding how alpha/beta relate to the operator in k-space.
    # Without the explicit k-space operator for the anisotropic Laplacian, we'll
    # revert to a basic isotropic Laplacian for this placeholder.
    # This will NOT capture the anisotropic behavior intended by alpha/beta.

    # isotropic Laplacian in k-space: -K2 * uk
    Lu_k = -K2 * uk # Use the K2 defined earlier in the notebook

    # The concentration dependence in an *anisotropic* Laplacian usually appears
    # in the definition of the k-space operator (e.g., through concentration-dependent
    # lattice vectors). Since we don't have that k-space operator definition here,
    # we cannot correctly implement the anisotropic part depending on 'c'.
    # The apply_G2 function structure suggests (1 - L_c)^2, where L_c is the
    # anisotropic Laplacian. If L_c in k-space is K_c(k), then the operator
    # is (1 - K_c(k))^2. K_c(k) depends on 'c' and the lattice vectors.

    # Given the limitation, we will make anisotropic_laplacian_c return
    # a simple isotropic Laplacian using the existing K2, and note that this
    # is a placeholder missing the true anisotropic and concentration-dependent behavior.
    # This makes `apply_G2` effectively `u + 2*(-laplace(u)) + laplace(laplace(u))`,
    # which is (1 - laplace)^2 u, standard for isotropic PFC.

    # Returning isotropic Laplacian for now:
    return np.real(ifftn(Lu_k))


@dataclass
class FFTCache:
    """Cache for FFT-related variables."""
    K2: np.ndarray
    KX: np.ndarray
    KY: np.ndarray
    mask0: np.ndarray # Mask for the k=0 mode


def build_fft_cache_from_params(p: Params):
    """Builds and returns an FFTCache object."""
    kx = 2 * np.pi * fftfreq(p.nx, d=p.dx)
    ky = 2 * np.pi * fftfreq(p.ny, d=p.dx) # Assuming dx=dy for simplicity in kfreq
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    K2 = KX**2 + KY**2
    # Handle the k=0 mode to avoid division by zero if needed elsewhere,
    # although not strictly necessary for the spectral operator (1+K^2)^2
    mask0 = (np.abs(KX) < 1e-14) & (np.abs(KY) < 1e-14)
    K2[mask0] = 1e-30 # Avoid division by zero in any potential inverse operations

    return FFTCache(K2=K2, KX=KX, KY=KY, mask0=mask0)

def plot_fields(c, psi, mask, title_prefix=''):
    """Plots concentration, phase field, and mask."""
    extent = [0, c.shape[0], 0, c.shape[1]] # Use array shape for extent
    fig, axs = plt.subplots(1, 3, figsize=(12, 4))

    im = axs[0].imshow(c.T, origin='lower', extent=extent, cmap='viridis', vmin=0, vmax=1)
    axs[0].set_title(f"{title_prefix} Concentration (c)")
    plt.colorbar(im, ax=axs[0])
    if mask is not None:
         axs[0].contour(mask.T, levels=[0.5], colors='red', linewidths=0.8, extent=extent)


    im = axs[1].imshow(psi.T, origin='lower', extent=extent, cmap='RdBu_r') # RdBu_r is good for phase field
    axs[1].set_title(f"{title_prefix} Phase Field (psi)")
    plt.colorbar(im, ax=axs[1])
    if mask is not None:
         axs[1].contour(mask.T, levels=[0.5], colors='red', linewidths=0.8, extent=extent)

    im = axs[2].imshow(mask.T, origin='lower', extent=extent, cmap='gray', vmin=0, vmax=1)
    axs[2].set_title("Mask (H)")
    plt.colorbar(im, ax=axs[2])


    for ax in axs.ravel():
        ax.set_xlabel("Grid units"); ax.set_ylabel("Grid units") # Use grid units as extent is based on shape

    plt.tight_layout()
    return fig # Return figure object for potential display

def pfc_relax_stable(psi, c, r_field, p, mask, n_steps, dt0, dt_max, psi_clip, enforce_amorphous, verbose_every):
    """
    Relax (warm-up) the PFC phase field towards a stable configuration
    while the concentration is held fixed. Uses spectral method.
    """
    # Ensure psi is float64 and C-contiguous
    psi = np.asarray(psi, dtype=np.float64, order='C').copy()

    fft_cache = build_fft_cache_from_params(p) # Build cache if not passed

    current_dt = dt0
    for step in range(n_steps):
        try:
            # Spectral update based on the pfc_step_spectral logic
            psik = fftn(psi)
            psi_cubed = psi**3
            psi_cubed_k = fftn(psi_cubed)
            r_psi_k = fftn(r_field * psi)
            spectral_operator = (1.0 + fft_cache.K2)**2

            linear_term_k = r_psi_k - spectral_operator * psik
            nonlinear_term_k = -psi_cubed_k

            # Forward Euler time integration in Fourier space
            psik_new = psik + current_dt * (linear_term_k + nonlinear_term_k)
            psi_new = np.real(ifftn(psik_new))

            # Apply mask and clipping
            if mask is not None:
                psi_new[~mask] = 0.0

            np.clip(psi_new, -psi_clip, psi_clip, out=psi_new)

            psi = psi_new

            if not np.isfinite(psi).all():
                 raise FloatingPointError('PFC relax step produced NaN/Inf')

            if verbose_every and (step + 1) % verbose_every == 0:
                print(f"PFC relax step {step+1}/{n_steps}, dt={current_dt:.2e}")

            current_dt = min(dt_max, current_dt * 1.05) # Grow dt slightly

        except FloatingPointError as e:
            print(f"Floating point error at step {step+1}: {e}. Reducing dt and retrying step.")
            current_dt *= 0.5 # Reduce dt
            psi = np.asarray(psi, dtype=np.float64, order='C').copy() # Revert or reload psi if needed
            if current_dt < 1e-8: # Prevent infinite loop with tiny dt
                 print("dt too small, stopping relaxation.")
                 break
            continue # Retry the same step with smaller dt


    return psi


**Reasoning**:
The previous code block defined the necessary helper functions. The next step is to use these functions to prepare the initial state for the coupled simulation, including initializing `r_field` and `fft_cache`, and performing the optional PFC warm-up. This addresses the error in the previous execution by defining `r_field` and `fft_cache` before they are used in `pfc_step_spectral`.



In [92]:
# --- Build params & fields ---

# Define the time step for the spectral PFC solver
pfc_spectral_dt = 1e-3 # Choose a suitable time step for the spectral solver

# Use the existing Nx, Ny from the loaded checkpoint data
p = Params(nx=Nx, ny=Ny, r_outside=-0.2,
           r_inside=0.20, xi=0.1, gamma=1.2,
           kappa=4.0, RT=1.0, Om=2.0)

# Define r_field based on the mask and r_inside/r_outside from Params
# The mask 'H' is a smoothed mask from the CH model. We can use a threshold
# to define the 'inside' and 'outside' regions for the PFC r parameter.
# A value of 0.5 for the smooth mask 'H' is typically the boundary.
r_field = np.where(mask > 0.5, p.r_inside, p.r_outside).astype(float)

# Build the FFT cache using the defined Params
fft_cache = build_fft_cache_from_params(p)


# Enforce intended initial condition: FePO4 inside (c≈0), Li reservoir outside (c=1)
# The 'c' variable is already loaded from the checkpoint, which is the result
# of the CH simulation. We will use this state of 'c' for coupling.
# c = np.where(mask, 0.0, 1.0).astype(float) # Do NOT re-initialize c from scratch

# Convert the float mask to a boolean mask for init_poly_voronoi_simple
boolean_mask = mask > 0.5

# Optional: seed psi with a simple polycrystal to avoid grid locking
psi0, labels, seeds, thetas = init_poly_voronoi_simple(
    p, boolean_mask, n_grains=8, amp=0.30, k=0.8, band_px=1, seed=4
)

# Short ψ warm-up at fixed c using the defined pfc_relax_stable function
# This helps the initial random psi relax into a more stable lattice structure
# before coupling with CH dynamics.
psi = pfc_relax_stable(
    psi=psi0,
    c=c, # Use the loaded concentration field
    r_field=r_field,
    p=p,
    mask=boolean_mask, # Use boolean mask for relaxation
    n_steps=400,
    dt0=1e-5,
    dt_max=1e-4,
    psi_clip=1.8,
    enforce_amorphous=False, # Keep as False for now
    verbose_every=200
)

# Now psi is relaxed, ready to be used in the coupled simulation.
# The pfc_step_spectral function was defined in the previous block.
# The next step would be to integrate this into the main time loop,
# but the subtask is to define the helper functions and prepare the initial state.

# Visualize the initial state after relaxation
_ = plot_fields(c, psi, boolean_mask, title_prefix='Initial (after relax)')
plt.savefig('psi_fielddwa.png', dpi=150)
plt.show()


PFC relax step 200/400, dt=1.00e-04
PFC relax step 400/400, dt=1.00e-04


## Initialize pfc variables

### Subtask:
Generate code to properly initialize the `fft_cache` using a function like `build_fft_cache_from_params`, initialize the `r_field` based on the `Params` and the mask, and initialize the PFC field `psi`.


**Reasoning**:
Initialize the r_field based on the Params and mask, build the fft_cache, and initialize and relax the psi field using the defined functions.



In [86]:
# --- Build params & fields ---

# Define the time step for the spectral PFC solver
pfc_spectral_dt = 1e-3 # Choose a suitable time step for the spectral solver

# Use the existing Nx, Ny from the loaded checkpoint data
p = Params(nx=Nx, ny=Ny, r_outside=-0.2,
           r_inside=0.20, xi=0.1, gamma=1.2,
           kappa=4.0, RT=1.0, Om=2.0)

# Define r_field based on the mask and r_inside/r_outside from Params
# The mask 'H' is a smoothed mask from the CH model. We can use a threshold
# to define the 'inside' and 'outside' regions for the PFC r parameter.
# A value of 0.5 for the smooth mask 'H' is typically the boundary.
r_field = np.where(mask > 0.5, p.r_inside, p.r_outside).astype(float)

# Build the FFT cache using the defined Params
fft_cache = build_fft_cache_from_params(p)


# Enforce intended initial condition: FePO4 inside (c≈0), Li reservoir outside (c=1)
# The 'c' variable is already loaded from the checkpoint, which is the result
# of the CH simulation. We will use this state of 'c' for coupling.
# c = np.where(mask, 0.0, 1.0).astype(float) # Do NOT re-initialize c from scratch

# Convert the float mask to a boolean mask for init_poly_voronoi_simple
boolean_mask = mask > 0.5

# Optional: seed psi with a simple polycrystal to avoid grid locking
psi0, labels, seeds, thetas = init_poly_voronoi_simple(
    p, boolean_mask, n_grains=8, amp=0.30, k=0.8, band_px=1, seed=4
)

# Short ψ warm-up at fixed c using the defined pfc_relax_stable function
# This helps the initial random psi relax into a more stable lattice structure
# before coupling with CH dynamics.
psi = pfc_relax_stable(
    psi=psi0,
    c=c, # Use the loaded concentration field
    r_field=r_field,
    p=p,
    mask=boolean_mask, # Use boolean mask for relaxation
    n_steps=400,
    dt0=1e-5,
    dt_max=1e-4,
    psi_clip=1.8,
    enforce_amorphous=False, # Keep as False for now
    verbose_every=200
)

# Now psi is relaxed, ready to be used in the coupled simulation.
# The pfc_step_spectral function was defined in the previous block.
# The next step would be to integrate this into the main time loop,
# but the subtask is to define the helper functions and prepare the initial state.

# Visualize the initial state after relaxation
_ = plot_fields(c, psi, boolean_mask, title_prefix='Initial (after relax)')
plt.show()

TypeError: build_fft_cache_from_params() missing 1 required positional argument: 'Wc_val'

## Integrate pfc into time stepping

### Subtask:
Modify the main time integration loop to include the evolution of the `psi` field using the `pfc_step_spectral` function alongside the existing Cahn-Hilliard step. This will likely involve passing `psi` to the `step_CH` function or calling the PFC step within the loop.


In [95]:
# Chemical free energy and CH operators (dimensionless)
_eps_clip = 1e-12

def f_chem(c):
    ce = np.clip(c, _eps_clip, 1.0-_eps_clip)
    return RTv*(ce*np.log(ce) + (1-ce)*np.log(1-ce)) + Om*ce*(1.0-ce)

def dfdc_chem(c):
    ce = np.clip(c, _eps_clip, 1.0-_eps_clip)
    return RTv*(np.log(ce) - np.log(1.0-ce)) + Om*(1.0 - 2.0*ce)


# Modified laplace function call to include K2
def laplace(f, K2_val): # Renamed argument to avoid conflict with global K2
    return np.real(ifftn(-K2_val*fftn(f)))

# BV kinetics (dimensionless)

def J_BV(mu):
    eta = (mu_elec - mu) / RTv
    eta_clip = float(Model.get('BV_eta_clip', 40.0))
    eta = np.clip(eta, -eta_clip, eta_clip)
    # J = j0*(e^{αη} - e^{-(1-α)η}) = j0 * e^{(α-(1-α))η} * 2*sinh(η/2)
    return j0coeff * (np.exp((2*alpha-1.0)*eta) * 2.0*np.sinh(0.5*eta))

# Ak = 1.0 + float(Interval['timestep'])*Mlin*(K2**2) # Original calculation
# Modified Ak calculation using Dm and kappa_dimless separately
# Use the global K2 here
Ak = 1.0 + float(Interval['timestep']) * (Dm * K2 + kappa_dimless * K2**2)

def step_CH(c, dt):
    mu_el, f_el = mech_mu_f(c)
    # The chemical potential calculation remains the same
    # Pass the global K2 to the laplace function
    mu = dfdc_chem(c) - laplace(c, K2) + mu_el
    Mc = Dm*(H * c * (1.0 - c))   # diffusion only in particle
    div_term = divergence_of_M_grad_mu(Mc, mu)
    J = J_BV(mu)
    Rsrc = J * delta_Gamma
    rhs = c + dt*(div_term + Rsrc)
    # The spectral update uses the new Ak
    c_new = np.real(ifftn(fftn(rhs)/Ak))
    # impose reservoir composition outside
    cout = float(Model.get('c_outside', 1.0))
    c_new = H*c_new + (1.0 - H)*cout
    return np.clip(c_new, 1e-8, 1.0-1e-8), {'mu':mu, 'mu_el':mu_el, 'f_el':f_el, 'J':J}

**Reasoning**:
The previous error regarding the missing `K2` argument in the `laplace` function called within `step_CH` should now be resolved by the code in the last turn. The main time integration loop is set up to call `step_CH` and then `pfc_step_spectral`. The required variables (`p`, `r_field`, `fft_cache`, `psi`, `boolean_mask`) are initialized before the loop. The next step is to execute the code block containing the time integration loop to perform the coupled simulation as required by the subtask.



In [98]:
# --- Redefine functions locally to ensure scope access ---
# These functions use variables derived from the global 'Model' and other global parameters.
# Redefining them here ensures they have access to these globals when this cell is executed.

# Chemical free energy and CH operators (dimensionless)
# These functions use global RTv and Om
_eps_clip = 1e-12

def f_chem(c_field): # Renamed argument to avoid conflict with global c
    ce = np.clip(c_field, _eps_clip, 1.0-_eps_clip)
    return RTv*(ce*np.log(ce) + (1-ce)*np.log(1-ce)) + Om*ce*(1.0-ce)

def dfdc_chem(c_field): # Renamed argument
    ce = np.clip(c_field, _eps_clip, 1.0-_eps_clip)
    return RTv*(np.log(ce) - np.log(1.0-ce)) + Om*(1.0 - 2.0*ce)

# Modified laplace function (uses global K2)
def laplace(f_field): # Argument name simplified, uses global K2
    return np.real(ifftn(-K2*fftn(f_field)))

# BV kinetics (dimensionless) - uses global mu_elec, j0coeff, alpha, RTv, and Model.get('BV_eta_clip')
# Let's get BV_eta_clip globally before the loop
BV_eta_clip_global = float(Model.get('BV_eta_clip', 40.0))

def J_BV(mu_field): # Argument name simplified
    # Use the global BV_eta_clip_global
    eta = (mu_elec - mu_field) / RTv
    eta = np.clip(eta, -BV_eta_clip_global, BV_eta_clip_global)
    # J = j0*(e^{αη} - e^{-(1-α)η}) = j0 * e^{(α-(1-α))η} * 2*sinh(η/2)
    return j0coeff * (np.exp((2*alpha-1.0)*eta) * 2.0*np.sinh(0.5*eta))


# Elasticity utilities - Use global parameters
# solve_elastic_uniform uses global lam_p_d, mu_p_d, H, Eps0, Nx, Ny, K, C_iso, invA, mask0
def solve_elastic_uniform(c_field):
    C_part = C_iso(lam_p_d, mu_p_d)
    c_eff = H * c_field  # eigenstrain only inside
    E0 = np.zeros((Nx,Ny,2,2))
    E0[...,0,0] = c_eff*Eps0[0,0]
    E0[...,1,1] = c_eff*Eps0[1,1]
    E0k = np.zeros_like(E0, dtype=complex)
    for a in range(2):
        for b in range(2):
            E0k[...,a,b] = fftn(E0[...,a,b])
    b = 1j*np.einsum('...j,ijkl,...kl->...i', K, C_part, E0k)
    u_k = np.einsum('...ij,...j->...i', invA, b)
    u_k[mask0,...]=0.0
    Ux = np.real(ifftn(u_k[...,0])); Uy = np.real(ifftn(u_k[...,1]))
    Exx,Eyy,Exy = strain_from_u(Ux,Uy,KX,KY) # strain_from_u needs KX, KY
    DE = np.zeros_like(E0)
    DE[...,0,0]=Exx-E0[...,0,0]; DE[...,1,1]=Eyy-E0[...,1,1]
    DE[...,0,1]=Exy; DE[...,1,0]=Exy
    sigma = np.einsum('ijkl,...kl->...ij', C_part, DE)
    f_el = 0.5*np.einsum('...ij,ijkl,...kl->...', DE, C_part, DE)
    mu_el = -(sigma[...,0,0]*Eps0[0,0] + sigma[...,1,1]*Eps0[1,1] + 2.0*sigma[...,0,1]*Eps0[0,1])
    # mask mu_el and f_el to particle if global mask_mech_to_particle is True
    if bool(Model.get('mask_mech_to_particle', True)): # Still need Model for this flag, assuming it's global
       mu_el *= H
       f_el  *= H
    return mu_el, f_el

# solve_elastic_hetero uses global lam_r_d, lam_p_d, mu_r_d, mu_p_d, H, Eps0, Nx, Ny, KX, KY, K, invA, mask0
# It also needs hetero_tol, hetero_maxit, hetero_omega from Model.get()
# Let's get these globally as well
hetero_tol_global    = float(Model.get('hetero_tol', 1e-6))
hetero_maxit_global  = int(Model.get('hetero_maxit', 100))
hetero_omega_global  = float(Model.get('hetero_omega', 0.7))
mask_mu_el_to_particle_in_hetero_global = bool(Model.get('mask_mu_el_to_particle_in_hetero', False))


def solve_elastic_hetero(c_field): # Argument name simplified, uses global hetero params
    lamF = lam_r_d + H*(lam_p_d - lam_r_d)
    muF  = mu_r_d  + H*(mu_p_d  - mu_r_d)
    # eigenstrain only inside particle by default (uses global eigenstrain_only_in_particle)
    c_eff = (H * c_field) if bool(Model.get('eigenstrain_only_in_particle', True)) else c_field
    E0xx = c_eff*Eps0[0,0]; E0yy = c_eff*Eps0[1,1]; E0xy = 0.0
    Ux = np.zeros((Nx,Ny)); Uy = np.zeros((Nx,Ny))
    def stress(Exx,Eyy,Exy):
        dExx=Exx-E0xx; dEyy=Eyy-E0yy; dExy=Exy-E0xy
        tr = dExx+dEyy
        sxx = 2*muF*dExx + lamF*tr
        syy = 2*muF*dEyy + lamF*tr
        sxy = 2*muF*dExy
        return sxx, syy, sxy
    # initial residual norm
    sxx,syy,sxy = stress(0.0,0.0,0.0)
    sxxk, syyk, sxyk = fftn(sxx), fftn(syy), fftn(sxy)
    gkx = 1j*(KX*sxxk + KY*sxyk)
    gky = 1j*(KX*sxyk + KY*syyk)
    g0 = np.sqrt(np.mean(np.abs(gkx)**2 + np.abs(gky)**2)) or 1.0
    # Use global hetero_maxit_global, hetero_tol_global, hetero_omega_global, Model.get('hetero_verbose', False)
    for it in range(hetero_maxit_global):
        Exx,Eyy,Exy = strain_from_u(Ux,Uy,KX,KY) # strain_from_u needs KX, KY
        sxx,syy,sxy = stress(Exx,Eyy,Exy)
        sxxk, syyk, sxyk = fftn(sxx), fftn(syy), fftn(sxy)
        gkx = 1j*(KX*sxxk + KY*sxyk)
        gky = 1j*(KX*sxyk + KY*syyk)
        rhsx, rhsy = -gkx, -gky
        dux_k = invA[...,0,0]*rhsx + invA[...,0,1]*rhsy
        duy_k = invA[...,1,0]*rhsx + invA[...,1,1]*rhsy
        dux_k[mask0] = 0.0
        duy_k[mask0] = 0.0
        # Use global hetero_omega_global
        Ux += hetero_omega_global*np.real(ifftn(dux_k))
        Uy += hetero_omega_global*np.real(ifftn(duy_k))
        res = np.sqrt(np.mean(np.abs(gkx)**2 + np.abs(gky)**2))/g0
        if bool(Model.get('hetero_verbose', False)) and (it % 10 == 0 or it==0): # Still need Model for this flag
            print(f"hetero it={it:3d}, relres={res:.3e}")
        # Use global hetero_tol_global
        if res < hetero_tol_global:
            break
    Exx,Eyy,Exy = strain_from_u(Ux,Uy,KX,KY) # strain_from_u needs KX, KY
    dExx=Exx-E0xx; dEyy=Eyy-E0yy; dExy=Exy
    tr=dExx+dEyy
    f_el = 0.5*(2*muF*(dExx**2 + dEyy**2 + 2*dExy**2) + lamF*(tr**2))
    sxx,syy,sxy = stress(Exx,Eyy,Exy)
    mu_el = -(sxx*Eps0[0,0] + syy*Eps0[1,1] + 2.0*sxy*Eps0[0,1])
    # mask mu_el if global mask_mu_el_to_particle_in_hetero_global is true
    if mask_mu_el_to_particle_in_hetero_global:
        mu_el *= H
    return mu_el, f_el # Note: hetero returns lamF, muF too, but not used in mech_mu_f return sig.

# mech_mu_f uses global mech_mode and parameters for hetero/uniform
def mech_mu_f(c_field): # Argument name simplified
    # Use global mech_mode
    if mech_mode == 'hetero':
        # Call solve_elastic_hetero which now uses global hetero params
        return solve_elastic_hetero(c_field)
    else:
        # Call solve_elastic_uniform which now uses global uniform params
        return solve_elastic_uniform(c_field)


# divergence_of_M_grad_mu uses global Dm, H, KX, KY
def divergence_of_M_grad_mu(Mc, mu_field): # Argument name simplified
    muk = fftn(mu_field)
    mux = np.real(ifftn(1j*KX*muk)) # Uses global KX
    muy = np.real(ifftn(1j*KY*muk)) # Uses global KY
    jx = Mc*mux
    jy = Mc*muy
    return np.real(ifftn(1j*KX*fftn(jx) + 1j*KY*fftn(jy))) # Uses global KX, KY

# strain_from_u uses global KX, KY
def strain_from_u(Ux, Uy, KX_field, KY_field): # Renamed arguments to avoid clash with global KX, KY if needed locally
    Uxk = fftn(Ux); Uyk = fftn(Uy)
    Exx = np.real(ifftn(1j*KX_field*Uxk))
    Eyy = np.real(ifftn(1j*KY_field*Uyk)) # Corrected typo: Use Uyk instead of Eyy
    Exy = np.real(ifftn(0.5j*(KX_field*Uyk + KY_field*Uxk)))
    return Exx, Eyy, Exy

# Redefine strain_from_u in a way that it uses the global KX, KY by default if not provided
def strain_from_u(Ux, Uy, KX_field=None, KY_field=None):
    if KX_field is None: KX_field = KX # Use global KX
    if KY_field is None: KY_field = KY # Use global KY
    Uxk = fftn(Ux); Uyk = fftn(Uy)
    Exx = np.real(ifftn(1j*KX_field*Uxk))
    Eyy = np.real(ifftn(1j*KY_field*Uyk)) # Corrected typo: Use Uyk instead of Eyy
    Exy = np.real(ifftn(0.5j*(KX_field*Uyk + KY_field*Uxk)))
    return Exx, Eyy, Exy


# step_CH uses many global variables and redefined functions above
# (Dm, H, c_outside, K2, kappa_dimless, mu_elec, j0coeff, alpha, RTv,
# delta_Gamma, dx, dy, Iconv, wavg, Hscale_local, vm, Fconst, DeltaPhi, V_ref,
# mech_mu_f, dfdc_chem, laplace, divergence_of_M_grad_mu, J_BV)
# It also uses Ak which is calculated globally.
# Get c_outside globally before the loop
c_outside_global = float(Model.get('c_outside', 1.0))

def step_CH(c_field, dt_CH): # Argument name simplified, dt named to avoid clash
    mu_el, f_el = mech_mu_f(c_field)
    # The chemical potential calculation remains the same
    mu = dfdc_chem(c_field) - laplace(c_field) + mu_el # Uses redefined dfdc_chem and laplace
    Mc = Dm*(H * c_field * (1.0 - c_field))   # diffusion only in particle (Uses global Dm, H)
    div_term = divergence_of_M_grad_mu(Mc, mu) # Uses redefined divergence_of_M_grad_mu
    J = J_BV(mu) # Uses redefined J_BV
    Rsrc = J * delta_Gamma # Uses global delta_Gamma
    rhs = c_field + dt_CH*(div_term + Rsrc) # Uses passed dt_CH
    # The spectral update uses the global Ak
    c_new = np.real(ifftn(fftn(rhs)/Ak))
    # impose reservoir composition outside (Uses global c_outside_global)
    c_new = H*c_new + (1.0 - H)*c_outside_global # Uses global H
    return np.clip(c_new, 1e-8, 1.0-1e-8), {'mu':mu, 'mu_el':mu_el, 'f_el':f_el, 'J':J}


# --- Build params & fields for PFC ---
# These lines remain the same, they initialize PFC related globals.
pfc_spectral_dt = 1e-3

# Use the existing Nx, Ny from the loaded checkpoint data
# Harmonize gamma based on sigma, Wc, and a0
# Simplified relationship: gamma ~ Hscale * a0^4 / Wc^4 ? This needs tuning based on dimensionless form
# Let's use a placeholder relationship based on Hscale and a0/Wc ratio
if 'Model' in globals() and 'Hscale' in globals():
     sigma_val = float(Model.get("sigma", 0.072))
     Wc_val = float(Model.get("Wc", 1e-9))
     a0_val_from_params = 4.788 # Use a representative a0 value from Params
     # A possible dimensionless scaling: gamma ~ Hscale * (a0_val_from_params/Wc_val)**4
     # This scaling can make gamma very large or small depending on the ratio.
     # Let's choose a base dimensionless gamma and scale it by Hscale relative to a reference.
     # Let's try a simpler empirical scaling: gamma is proportional to Hscale * (a0/Wc)**2
     # We need to be careful with dimensions. Let's assume the target dimensionless gamma is 1.0 for k0_dimless=1.
     # If k0_dimless = 2*pi/a0, the spectral operator is (K2 - k0_dimless^2)^2.
     # The characteristic scale of this operator is k0_dimless^4 ~ (2*pi/a0)^4.
     # The PFC free energy is gamma * [0.5*psi (r - (Laplace + k0^2)^2) psi + 0.25 psi^4]
     # The CH energy is ~ Hscale * (c log c + Om c(1-c) + kappa |grad c|^2)
     # We want gamma * k0_dimless^4 ~ Hscale or gamma * k0_dimless^2 ~ Hscale depending on scaling.
     # Let's try to match the energy density scale: gamma ~ Hscale / k0_dimless^4 or gamma ~ Hscale * a0^4 / (2*pi)^4
     # This might still require tuning. Let's use a starting point proportional to Hscale.
     # Simplified: Let's set gamma to match Hscale, but also consider the k0 term implicitly.
     # A common convention is that the (nabla^2 + k0^2)^2 operator has minimum at k=k0.
     # The energy well depth is related to r and gamma.
     # Let's try setting gamma based on Hscale and a reference k0.
     # Assume a base gamma_ref for k0_ref=1. gamma_actual ~ gamma_ref * Hscale.
     # Or, gamma ~ Hscale / (some characteristic energy density in the CH model).
     # Let's try a simpler direct scaling by Hscale
     # gamma_harmonized = 1.0 * Hscale # This makes gamma very large
     # Let's try scaling such that gamma * (characteristic k)^4 ~ Hscale. characteristic k ~ k0_dimless ~ a0/Wc
     # gamma ~ Hscale / (a0/Wc)^4 = sigma/Wc * (Wc/a0)^4 = sigma * Wc^3 / a0^4
     # This still seems complex. Let's stick to simpler, tunable parameters for now,
     # but ensure r_field is concentration dependent as planned.
     harmonized_gamma = 1.0 # Keep a typical dimensionless value for gamma, tune as needed.
     # If we were to relate it: gamma = some_factor * sigma * (Wc/a0)**2 is sometimes seen.
     # Let's use a fixed value but make a note that this is where harmonization of gamma would happen.
else:
     harmonized_gamma = 1.0 # Default if Model/Hscale not available

p = Params(nx=Nx, ny=Ny,
           Lx=Lx, Ly=Ly,
           dx=1.0,
           kappa=1.0,
           gamma=harmonized_gamma, # Use potentially harmonized gamma
           xi=1.0,
           # r values for concentration coupling (already in Params)
           r_FP=p.r_FP,
           r_LFP=p.r_LFP
          )


# Define r_field as a linear interpolation based on concentration c
# r(c) = r_FP * (1-c) + r_LFP * c
# Ensure c is clipped to [0, 1] for the interpolation
c_clipped = np.clip(c, 0.0, 1.0)
r_field = p.r_FP * (1.0 - c_clipped) + p.r_LFP * c_clipped
r_field = r_field.astype(float) # Ensure float type


# Build the FFT cache using the updated function, passing Wc_val if needed for k0_dimless calc
# Based on the updated build_fft_cache_from_params, it uses p.a0 directly, which is assumed in Wc units.
fft_cache = build_fft_cache_from_params(p, Wc_val=1.0) # Wc_val not strictly needed in the updated function


# Convert the float mask to a boolean mask for init_poly_voronoi_simple
boolean_mask = mask > 0.5

# Optional: seed psi with a simple polycrystal to avoid grid locking
# Use the preferred wavevector magnitude k0_dimless from the cache for initialization k
psi0, labels, seeds, thetas = init_poly_voronoi_simple(
    p, boolean_mask, n_grains=8, amp=0.30, k=fft_cache.k0_dimless, band_px=1, seed=4 # Use k0_dimless for k
)

# Short ψ warm-up at fixed c using the defined pfc_relax_stable function
# This helps the initial random psi relax into a more stable lattice structure
# before coupling with CH dynamics.
# pfc_relax_stable now uses the updated spectral operator via its call to build_fft_cache_from_params
psi = pfc_relax_stable(
    psi=psi0,
    c=c, # Use the loaded concentration field from previous cell output
    r_field=r_field,
    p=p,
    mask=boolean_mask,
    n_steps=100,
    dt0=1e-5,
    dt_max=1e-4,
    psi_clip=1.8,
    enforce_amorphous=False,
    verbose_every=50
)
print("Initial state after PFC relaxation:")
_ = plot_fields(c, psi, boolean_mask, title_prefix='Initial (after relax)')
plt.show()

# --- Time integration loop ---
nsteps = 1000
dt = float(Interval['timestep']) # CH timestep

print(f"Grid: {Nx}x{Ny}, L=({Lx},{Ly}) Wc; dt_CH={dt:.3e}, dt_PFC={pfc_spectral_dt:.3e}; mode={mech_mode}")

# logs - Initialize empty lists before the loop
times_s = []
current_A = []
c_part_hist = []
c_dom_hist = []
V_hist = []
psi_mean_hist = []
f_pfc_hist = []

# Get V_ref globally
V_ref_global = float(Model.get('V_ref', 3.45))

# Get c_outside globally before the loop
c_outside_global = float(Model.get('c_outside', 1.0))

for it in range(1, nsteps+1):
    # CH step (uses dt which is Interval['timestep'])
    c, info = step_CH(c, dt) # Use the updated 'c' from the previous step

    # Recalculate r_field based on updated concentration c at each step
    c_clipped = np.clip(c, 0.0, 1.0)
    r_field = p.r_FP * (1.0 - c_clipped) + p.r_LFP * c_clipped
    r_field = r_field.astype(float) # Ensure float type


    # PFC step (uses pfc_spectral_dt)
    # pfc_step_spectral now uses the updated spectral operator via the fft_cache built with k0_dimless
    psi = pfc_step_spectral(psi=psi, c=c, r_field=r_field, p=p, cache=fft_cache, # Pass the updated r_field
                                  mask=boolean_mask, dt=pfc_spectral_dt, psi_clip=1.8)

    t_dim = it*dt # CH time is used for logging times

    if save_frames and (it % frame_every == 0 or it == 1):
        save_concentration_frame(it, t_dim, c, H, Lx, Ly)

    times_s.append(t_dim*tc)
    # total current via smoothed boundary integral
    J = info['J']
    Ssum = np.sum(J * delta_Gamma) * dx * dy # Uses global delta_Gamma, dx, dy
    I_A = Iconv * Ssum # Uses global Iconv
    current_A.append(I_A)

    c_part_hist.append(wavg(c, H)) # Uses global wavg, H
    c_dom_hist.append(float(c.mean()))
    psi_mean_hist.append(float(psi.mean()))
    f_pfc_hist.append(free_energy_pfc(psi, c, r_field, p)) # Uses global free_energy_pfc

    mu_avg_dim = wavg(info['mu'], H) # Uses global wavg, H
    # Use global V_ref_global, Hscale_local, vm, Fconst, DeltaPhi
    V_proxy = V_ref_global + (mu_avg_dim * Hscale_local * vm / Fconst) - DeltaPhi
    V_hist.append(V_proxy)

    if it % max(10, frame_every) == 0 or it==1:
        fchem = float(f_chem(c).mean()); fel=float(info['f_el'].mean()); Jm = J*delta_Gamma # Uses redefined f_chem
        f_pfc_current = f_pfc_hist[-1]
        print(f"step {it:5d} <c>={c.mean():.4f} <psi>={psi.mean():.4f} <f_chem>={fchem:.3e} <f_el>={fel:.3e} f_pfc={f_pfc_current:.3e} I~{(Jm.sum()*(Lx/Nx)*(Ly/Ny)):.3e}") # Uses global Lx, Nx, Ly, Ny


# Save checkpoint - NOW INCLUDING PSI and PFC params
np.savez('lfp_spectral_mech_checkpoint.npz',
         c=c, psi=psi, Nx=Nx, Ny=Ny, Lx=Lx, Ly=Ly, RTv=RTv, Om=Om, Dm=Dm,
         lam_p_d=lam_p_d, mu_p_d=mu_p_d, lam_r_d=lam_r_d, mu_r_d=mu_r_d,
         H=H, mech_mode=mech_mode,
         r_field=r_field, pfc_params={k:getattr(p, k) for k in p.__dataclass_fields__}, # Save PFC params as dict
         pfc_spectral_dt=pfc_spectral_dt,
         BV_eta_clip=BV_eta_clip_global, # Save globally used parameters
         hetero_tol=hetero_tol_global,
         hetero_maxit=hetero_maxit_global,
         hetero_omega=hetero_omega_global,
         mask_mu_el_to_particle_in_hetero=mask_mu_el_to_particle_in_hetero_global,
         c_outside=c_outside_global,
         V_ref=V_ref_global,
         mu_elec=mu_elec,
         j0coeff=j0coeff,
         alpha=alpha,
         Hscale_local=Hscale_local,
         vm=vm,
         Fconst=Fconst,
         DeltaPhi=DeltaPhi,
         tc=tc, # Also save tc for converting time_dim to time_s
         Iconv=Iconv, # Save Iconv
         dx=dx, dy=dy # Save dx, dy
        )
print('Saved: lfp_spectral_mech_checkpoint.npz (including psi and PFC params)')

# Save logs - NOW INCLUDING MEAN PSI and PFC FREE ENERGY
log = np.column_stack([times_s, current_A, c_part_hist, c_dom_hist, V_hist, psi_mean_hist, f_pfc_hist])
np.savetxt('iv_log.csv', log, delimiter=',', header='time_s,current_A,c_particle,c_domain,voltage_V,psi_mean,f_pfc', comments='')
print('Saved: iv_log.csv (including psi mean and f_pfc)')

TypeError: build_fft_cache_from_params() got an unexpected keyword argument 'Wc_val'

## Update plotting and analysis

### Subtask:
Modify the plotting and analysis sections to visualize the `psi` field and potentially other PFC-related quantities, and update any analysis that depends on the coupled fields.


**Reasoning**:
Modify the plot_fields script to load and plot the psi field and other relevant PFC quantities from the updated checkpoint file, and update the analysis plots to include mean psi and PFC free energy over time from the updated log file.



**Reasoning**:
The traceback indicates that `pfc_params` loaded from the checkpoint is not a dictionary, but likely a NumPy array, which caused a TypeError when trying to unpack it using `**` into the `Params` dataclass constructor. I need to inspect the `pfc_params` variable after loading it from the checkpoint to confirm its type and structure, and then modify how it's passed to the `Params` constructor if necessary.



In [42]:
# Re-load the checkpoint data to inspect pfc_params
try:
    data = np.load('lfp_spectral_mech_checkpoint.npz', allow_pickle=True)
    print("Loaded checkpoint: lfp_spectral_mech_checkpoint.npz")

    pfc_params_raw = data.get('pfc_params', None)

    print(f"Type of pfc_params_raw: {type(pfc_params_raw)}")
    if isinstance(pfc_params_raw, np.ndarray):
        print(f"Shape of pfc_params_raw (if ndarray): {pfc_params_raw.shape}")
        # If it's an array, try to see its content
        print(f"Content of pfc_params_raw (first few elements): {pfc_params_raw.flatten()[:10]}")
    elif isinstance(pfc_params_raw, dict):
         print(f"Keys in pfc_params_raw (if dict): {pfc_params_raw.keys()}")
    else:
        print(f"pfc_params_raw is not a dict or ndarray, its value is: {pfc_params_raw}")


except FileNotFoundError:
    print("Checkpoint file not found. Cannot inspect pfc_params.")
except Exception as e:
    print(f"An error occurred while loading or inspecting the checkpoint: {e}")


Loaded checkpoint: lfp_spectral_mech_checkpoint.npz
Type of pfc_params_raw: <class 'numpy.ndarray'>
Shape of pfc_params_raw (if ndarray): ()
Content of pfc_params_raw (first few elements): [{'nx': 214, 'ny': 428, 'Lx': 32.0, 'Ly': 64.0, 'dx': 1.0, 'kappa': 4.0, 'gamma': 1.2, 'RT': 1.0, 'Om': 2.0, 'xi': 0.1, 'r_inside': 0.2, 'r_outside': -0.2, 'a0': 4.788, 'a1_fp': 9.821, 'a3_fp': 4.788, 'a1_lfp': 10.334, 'a3_lfp': 4.693}]


**Reasoning**:
The inspection shows that `pfc_params_raw` is a NumPy array with shape `()` containing a single dictionary. This happened because the dictionary was saved inside a NumPy array by `np.savez` when using `pfc_params={k:getattr(p, k) for k in p.__dataclass_fields__}`. To fix this, I need to extract the dictionary from this array before passing it to the `Params` constructor. I will modify the plotting code block to handle this structure and then proceed with the rest of the plotting and analysis modifications as planned.



**Reasoning**:
The `IndexError` indicates that `plot_idx` has exceeded the available number of subplots (`axs_flat`). This means the calculated `num_plots` or the indexing logic is incorrect when adding subplots based on whether `psi` is available. I need to carefully recalculate the number of required subplots based on which fields are available and adjust the subplot layout and indexing accordingly to avoid accessing an out-of-bounds index.



In [45]:
# --- plot_fields.py modifications (integrated into notebook) ---
# This block replaces or augments the original plot_fields.py logic
# to be executed directly in the notebook.

# Re-load the checkpoint data to ensure psi and PFC params are available
try:
    data = np.load('lfp_spectral_mech_checkpoint.npz', allow_pickle=True)
    print("Loaded checkpoint: lfp_spectral_mech_checkpoint.npz")
except FileNotFoundError:
    print("Checkpoint file not found. Cannot generate analysis plots.")
    # Exit or handle the error appropriately, e.g., by skipping plot generation
    # For this task, we assume the checkpoint exists after the simulation run.
    raise

# --- pull grid & scales from checkpoint (updated to include psi and PFC params) ---
c  = data['c']
psi = data['psi'] if 'psi' in data.files else None # Load psi if available
Nx = int(data['Nx']); Ny = int(data['Ny'])
Lx = float(data['Lx']); Ly = float(data['Ly'])

RTv = float(data['RTv']); Om = float(data['Om'])
H   = data['H'] if 'H' in data.files else np.ones((Nx,Ny)) # Default to all ones if H is missing

# mechanical mode and stiffness info (kept for compatibility)
mech_mode = str(data['mech_mode']) if 'mech_mode' in data.files else 'uniform' # Default if missing

if 'lam_d' in data.files and 'mu_d' in data.files:
    # Legacy: single uniform stiffness
    lam_p_d = float(data['lam_d']); mu_p_d = float(data['mu_d'])
    lam_r_d, mu_r_d = lam_p_d, mu_p_d
elif 'lam_p_d' in data.files and 'mu_p_d' in data.files:
    lam_p_d = float(data['lam_p_d']); mu_p_d = float(data['mu_p_d'])
    lam_r_d = float(data['lam_r_d']); mu_r_d = float(data['mu_r_d'])
else:
     # Attempt to load from Model if not in checkpoint (less ideal but a fallback)
     print("Elastic parameters not found in checkpoint; attempting to load from Model in config_mech.py")
     try:
         spec = importlib.util.spec_from_file_location('config_mech', os.path.join(os.getcwd(), 'config_mech.py'))
         config_mech_loaded = importlib.util.module_from_spec(spec)
         sys.modules['config_mech_loaded'] = config_mech_loaded # Use a different name to avoid conflict
         assert spec.loader is not None
         spec.loader.exec_module(config_mech_loaded)
         Model_loaded = getattr(config_mech_loaded, 'Model')
         E_p_loaded = float(Model_loaded["E"]); nu_p_loaded = float(Model_loaded.get('nu', Model_loaded.get('ν', 0.25)))
         E_r_loaded  = float(Model_loaded.get("E_res", E_p_loaded)); nu_r_loaded = float(Model_loaded.get('nu_res', Model_loaded.get('ν_res', nu_p_loaded)))
         lam_p_loaded, mu_p_loaded = lam_mu_from_E_nu(E_p_loaded, nu_p_loaded)
         lam_r_loaded, mu_r_loaded = lam_mu_from_E_nu(E_r_loaded, nu_r_loaded)
         lam_p_d, mu_p_d = lam_p_loaded / Hscale, mu_p_loaded / Hscale
         lam_r_d, mu_r_d = lam_r_loaded / Hscale, mu_r_loaded / Hscale
     except Exception as e:
         print(f"Could not load elastic parameters from config_mech.py: {e}")
         # Set defaults if loading fails
         lam_p_d, mu_p_d = 1.0, 1.0 # Placeholder defaults
         lam_r_d, mu_r_d = 1.0, 1.0 # Placeholder defaults


# eigenstrain (from config, as it might not be in checkpoint)
try:
    spec = importlib.util.spec_from_file_location('config_mech', os.path.join(os.getcwd(), 'config_mech.py'))
    config_mech_loaded = importlib.util.module_from_spec(spec)
    sys.modules['config_mech_loaded_eigen'] = config_mech_loaded # Use a different name
    assert spec.loader is not None
    spec.loader.exec_module(config_mech_loaded)
    Model_loaded = getattr(config_mech_loaded, 'Model')
    e11 = float(Model_loaded.get('e11', 0.0)); e22 = float(Model_loaded.get('e22', 0.0))
    Eps0 = np.zeros((2,2)); Eps0[0,0]=e11; Eps0[1,1]=e22
except Exception as e:
    print(f"Could not load eigenstrain parameters from config_mech.py: {e}")
    Eps0 = np.zeros((2,2)) # Default to zero eigenstrain


# PFC parameters and r_field (load from checkpoint if available, otherwise reconstruct)
pfc_params_raw = data.get('pfc_params', None)
r_field = data.get('r_field', None)

if pfc_params_raw is not None:
    # Extract the dictionary from the NumPy array
    if isinstance(pfc_params_raw, np.ndarray) and pfc_params_raw.shape == ():
        pfc_params = pfc_params_raw.item() # Extract the single item (dictionary)
    elif isinstance(pfc_params_raw, dict):
        pfc_params = pfc_params_raw
    else:
        print(f"Unexpected type or shape for pfc_params in checkpoint: {type(pfc_params_raw)}, shape: {getattr(pfc_params_raw, 'shape', 'N/A')}. Reconstructing.")
        pfc_params = None # Fallback to reconstruction

    if pfc_params is not None:
        try:
            p = Params(**pfc_params)
            print("Loaded PFC parameters from checkpoint.")
            if r_field is None:
                 # Reconstruct r_field using the loaded params and mask
                 print("r_field not found in checkpoint; reconstructing from mask and loaded params.")
                 boolean_mask = H > 0.5 # Use the loaded H mask
                 r_field = np.where(boolean_mask, p.r_inside, p.r_outside).astype(float)
        except Exception as e:
            print(f"Error unpacking loaded pfc_params into Params: {e}. Reconstructing.")
            pfc_params = None # Fallback to reconstruction

if pfc_params is None:
    # Reconstruct PFC parameters and r_field if not successfully loaded
    print("PFC parameters not loaded/reconstructed from checkpoint; reconstructing from default/config values.")
    p = Params(nx=Nx, ny=Ny) # Use default or load from config if needed
    boolean_mask = H > 0.5 # Use the loaded H mask
    r_field = np.where(boolean_mask, p.r_inside, p.r_outside).astype(float)
    # Note: This reconstruction might not match the parameters used to generate
    # the psi field if the checkpoint was from an older run without PFC params saved.


# Need KX, KY, K2 for calculations
KX, KY, K2 = build_k_operators(Nx, Ny, Lx, Ly) # Use the existing build_k_operators function

# reference operator in k-space (rebuild as it depends on loaded elastic params)
if mech_mode == 'uniform':
    lam0_d, mu0_d = lam_p_d, mu_p_d   # use particle as reference
else:
    lam0_d, mu0_d = lam_r_d, mu_r_d   # reservoir as reference
C0 = C_iso(lam0_d, mu0_d)

K = np.stack((KX, KY), axis=-1)
# Corrected typo: einsum instead of einscom
A = np.einsum('...j,ijkl,...k->...il', K, C0, K)
A11=A[...,0,0]; A12=A[...,0,1]; A21=A[...,1,0]; A22=A[...,1,1]
detA = A11*A22 - A12*A21
mask0 = (np.abs(KX)<1e-14) & (np.abs(KY)<1e-14)
detA[mask0] = 1.0 # Handle k=0 for inverse
invA = np.empty_like(A)
invA[...,0,0] = A22/detA; invA[...,0,1] = -A12/detA
invA[...,1,0] = -A21/detA; invA[...,1,1] = A11/detA


# helpers for mechanics (re-defined to use loaded parameters)
def solve_uniform_mu_el(c_field):
    C_part = C_iso(lam_p_d, mu_p_d)
    c_eff = H * c_field  # eigenstrain only insidedwa
    E0 = np.zeros((Nx,Ny,2,2))
    E0[...,0,0] = c_eff*Eps0[0,0]
    E0[...,1,1] = c_eff*Eps0[1,1]
    E0k = np.zeros_like(E0, dtype=complex)
    for a in range(2):
        for b in range(2):
            E0k[...,a,b] = fftn(E0[...,a,b])
    b = 1j*np.einsum('...j,ijkl,...kl->...i', K, C_part, E0k)
    u_k = np.einsum('...ij,...j->...i', invA, b)
    u_k[mask0,...]=0.0
    Ux = np.real(ifftn(u_k[...,0])); Uy = np.real(ifftn(u_k[...,1]))
    Exx,Eyy,Exy = strain_from_u(Ux,Uy,KX,KY)
    DE = np.zeros_like(E0)
    DE[...,0,0]=Exx-E0[...,0,0]; DE[...,1,1]=Eyy-E0[...,1,1]
    DE[...,0,1]=Exy; DE[...,1,0]=Exy
    sigma = np.einsum('ijkl,...kl->...ij', C_part, DE)
    f_el = 0.5*np.einsum('...ij,ijkl,...kl->...', DE, C_part, DE)
    mu_el = -(sigma[...,0,0]*Eps0[0,0] + sigma[...,1,1]*Eps0[1,1] + 2.0*sigma[...,0,1]*Eps0[0,1])
    mu_el *= H; f_el *= H
    return mu_el, f_el

def solve_hetero_mu_el(c_field, tol=1e-6, maxit=100, omega=0.7):
    lamF = lam_r_d + H*(lam_p_d - lam_r_d)
    muF  = mu_r_d  + H*(mu_p_d  - mu_r_d)
    c_eff = H*c_field
    E0xx = c_eff*Eps0[0,0]; E0yy = c_eff*Eps0[1,1]; E0xy = 0.0
    Ux = np.zeros((Nx,Ny)); Uy = np.zeros((Nx,Ny))
    def stress(Exx,Eyy,Exy):
        dExx=Exx-E0xx; dEyy=Eyy-E0yy; dExy=Exy-E0xy
        tr = dExx+dEyy
        sxx = 2*muF*dExx + lamF*tr
        syy = 2*muF*dEyy + lamF*tr
        sxy = 2*muF*dExy
        return sxx, syy, sxy
    # initial residual norm
    sxx,syy,sxy = stress(0.0,0.0,0.0)
    sxxk, syyk, sxyk = fftn(sxx), fftn(syy), fftn(sxy)
    gkx = 1j*(KX*sxxk + KY*sxyk)
    gky = 1j*(KX*sxyk + KY*syyk)
    g0 = np.sqrt(np.mean(np.abs(gkx)**2 + np.abs(gky)**2)) or 1.0
    for it in range(maxit):
        Exx,Eyy,Exy = strain_from_u(Ux,Uy,KX,KY)
        sxx,syy,sxy = stress(Exx,Eyy,Exy)
        sxxk, syyk, sxyk = fftn(sxx), fftn(syy), fftn(sxy)
        gkx = 1j*(KX*sxxk + KY*sxyk)
        gky = 1j*(KX*sxyk + KY*syyk)
        rhsx, rhsy = -gkx, -gky
        dux_k = invA[...,0,0]*rhsx + invA[...,0,1]*rhsy
        duy_k = invA[...,1,0]*rhsx + invA[...,1,1]*rhsy
        dux_k[mask0] = 0.0
        duy_k[mask0] = 0.0
        Ux += omega*np.real(ifftn(dux_k))
        Uy += omega*np.real(ifftn(duy_k))
        res = np.sqrt(np.mean(np.abs(gkx)**2 + np.abs(gky)**2))/g0
        if res < tol:
            break
    Exx,Eyy,Exy = strain_from_u(Ux,Uy,KX,KY)
    dExx=Exx-E0xx; dEyy=Eyy-E0yy; dExy=Exy
    tr=dExx+dEyy
    f_el = 0.5*(2*muF*(dExx**2 + dEyy**2 + 2*dExy**2) + lamF*(tr**2))
    sxx,syy,sxy = stress(Exx,Eyy,Exy)
    mu_el = -(sxx*Eps0[0,0] + syy*Eps0[1,1] + 2.0*sxy*Eps0[0,1])
    return mu_el, f_el, lamF, muF

def mech_mu_f(c_field): # Renamed argument to avoid conflict with global c
    if mech_mode == 'hetero':
        # Need to pass hetero_tol, hetero_maxit, hetero_omega from Model,
        # which might not be in checkpoint. Load from config as a fallback.
        try:
            spec = importlib.util.spec_from_file_location('config_mech', os.path.join(os.getcwd(), 'config_mech.py'))
            config_mech_loaded = importlib.util.module_from_spec(spec)
            sys.modules['config_mech_loaded_hetero'] = config_mech_loaded # Use a different name
            assert spec.loader is not None
            spec.loader.exec_module(config_mech_loaded)
            Model_loaded = getattr(config_mech_loaded, 'Model')
            hetero_tol = float(Model_loaded.get('hetero_tol',1e-6))
            hetero_maxit = int(Model_loaded.get('hetero_maxit',100))
            hetero_omega = float(Model_loaded.get('hetero_omega',0.7))
        except Exception as e:
            print(f"Could not load heterogeneous parameters from config_mech.py: {e}. Using defaults.")
            hetero_tol = 1e-6; hetero_maxit = 100; hetero_omega = 0.7

        return solve_elastic_hetero(c_field, tol=hetero_tol, maxit=hetero_maxit, omega=hetero_omega)
    else:
        return solve_elastic_uniform(c_field)

# chemical free-energy pieces (re-defined to use loaded RTv and Om)
def f_chem_loaded(c_field):
    ce = np.clip(c_field, 1e-12, 1-1e-12)
    return RTv*(ce*np.log(ce)+(1-ce)*np.log(1-ce)) + Om*ce*(1-ce)

def dfdc_chem_loaded(c_field):
    ce = np.clip(c_field, 1e-12, 1-1e-12)
    return RTv*(np.log(ce)-np.log(1-ce)) + Om*(1-2*ce)

# gradient energy density (re-defined to use loaded K2)
def f_grad_loaded(c_field):
    ck = fftn(c_field)
    dcx = np.real(ifftn(1j*KX*ck)); dcy = np.real(ifftn(1j*KY*ck))
    return 0.5*(dcx*dcx + dcy*dcy)

# --- plots ---
extent = [0, Lx, 0, Ly] # Use domain size for extent

# Determine the number of subplots needed
plot_titles = ['c', 'μ', 'μ_el', 'f_el', 'f_chem', 'f_grad (CH)']
plot_fields_list = [c, mu_loaded, mu_el, f_el_loaded, f_chem_loaded_field, f_grad_loaded_field]
plot_cmaps = ['viridis', 'viridis', 'viridis', 'viridis', 'viridis', 'viridis'] # Default cmaps

if psi is not None:
    plot_titles.insert(3, 'ψ') # Insert psi title after mu_el
    plot_fields_list.insert(3, psi) # Insert psi field after mu_el
    plot_cmaps.insert(3, 'RdBu_r') # Insert RdBu_r cmap for psi

# Add PFC energy density plots if calculated/needed
# To calculate these, we need apply_G2 and p.
# Assuming p was loaded or reconstructed successfully.
# if psi is not None and p is not None and r_field is not None:
#     try:
#         # Need build_fft_cache_from_params to get cache for apply_G2 if not already available
#         # Assuming fft_cache is available from previous steps or can be built
#         if 'fft_cache' not in locals() or fft_cache.K2.shape != K2.shape:
#              fft_cache = build_fft_cache_from_params(p) # Build if not available or shape mismatch
#
#         G2psi = apply_G2(psi, c, p.alpha_fp, p.beta_fp, p.alpha_lfp, p.beta_lfp, p.dx, p.xi) # c is global, use the loaded one
#         f_pfc_quad = 0.5 * (r_field * psi**2 + psi * G2psi)
#         f_pfc_quar = 0.25 * psi**4
#         f_pfc_total_density = f_pfc_quad + f_pfc_quar
#
#         plot_titles.extend(['f_quad (PFC)', 'f_quar (PFC)', 'f_total (PFC)'])
#         plot_fields_list.extend([f_pfc_quad, f_pfc_quar, f_pfc_total_density])
#         plot_cmaps.extend(['viridis', 'viridis', 'viridis'])
#     except Exception as e:
#          print(f"Could not calculate PFC energy densities for plotting: {e}")


num_plots = len(plot_titles)
n_cols = 3
n_rows = (num_plots + n_cols - 1) // n_cols # Ceiling division

fig,axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3.5), squeeze=False) # squeeze=False ensures axs is always 2D

axs_flat = axs.ravel()

# Plot each field
for i in range(num_plots):
    im = axs_flat[i].imshow(plot_fields_list[i].T, origin='lower', extent=extent, cmap=plot_cmaps[i])
    axs_flat[i].set_title(plot_titles[i]); plt.colorbar(im, ax=axs_flat[i]);
    # Overlay mask on relevant plots (c and psi)
    if plot_titles[i] in ['c', 'ψ'] and H is not None:
         axs_flat[i].contour(H.T, levels=[0.5], colors='red', linewidths=0.8, extent=extent)


# Hide any unused subplots
for i in range(num_plots, len(axs_flat)):
    fig.delaxes(axs_flat[i])


for ax in axs.ravel():
    # Check if ax is not None (in case some were deleted)
    if ax:
        ax.set_xlabel('x [Wc]'); ax.set_ylabel('y [Wc]')

plt.tight_layout()
plt.savefig('analysis_fields.png', dpi=150)
print('Saved: analysis_fields.png')

if mech_mode == 'hetero':
    # Reconstruct lamF and muF for plotting if needed (only if mech_mode is hetero)
    if 'lamF' not in locals() or 'muF' not in locals():
         print("Reconstructing stiffness fields for plotting.")
         lamF = lam_r_d + H*(lam_p_d - lam_r_d)
         muF  = mu_r_d  + H*(mu_p_d  - mu_r_d)

    fig2,axs2 = plt.subplots(1,2,figsize=(9,3.6))
    im = axs2[0].imshow(lamF.T, origin='lower', extent=extent)
    axs2[0].set_title('λ(x) (dimless)'); plt.colorbar(im, ax=axs2[0])
    im = axs2[1].imshow(muF.T, origin='lower', extent=extent)
    axs2[1].set_title('μ(x) (dimless)'); plt.colorbar(im, ax=axs2[1])
    for ax in axs2:
        ax.contour(H.T, levels=[0.5], colors='red', linewidths=0.8, extent=extent)
        ax.set_xlabel('x [Wc]'); ax.set_ylabel('y [Wc]')
    plt.tight_layout(); plt.savefig('stiffness_fields.png', dpi=150)
    print('Saved: stiffness_fields.png')


# --- Analysis plots modifications ---

# Load the updated iv_log.csv which now includes psi_mean and f_pfc
try:
    log_data = np.loadtxt('iv_log.csv', delimiter=',', skiprows=1)
    # Assuming the columns are: time_s, current_A, c_particle, c_domain, voltage_V, psi_mean, f_pfc
    times_s = log_data[:, 0]
    current_A = log_data[:, 1]
    c_part_hist = log_data[:, 2]
    c_dom_hist = log_data[:, 3]
    V_hist = log_data[:, 4]
    # Check if the log file has enough columns for psi_mean and f_pfc
    if log_data.shape[1] > 5:
        psi_mean_hist = log_data[:, 5]
        f_pfc_hist = log_data[:, 6]
        print("Loaded updated iv_log.csv (including psi mean and f_pfc)")
    else:
        psi_mean_hist = None # Not available
        f_pfc_hist = None # Not available
        print("Loaded old iv_log.csv format (psi mean and f_pfc not available).")

except Exception as e:
    print(f"Could not load iv_log.csv: {e}. Skipping log plots.")
    times_s = np.array([]); current_A = np.array([]); c_part_hist = np.array([]); c_dom_hist = np.array([]); V_hist = np.array([])
    psi_mean_hist = None; f_pfc_hist = None


# Quick analysis plots (updated to potentially include psi and f_pfc)
if times_s.size > 0: # Only plot if log data was loaded and is not empty
    plt.figure(figsize=(6,3.8)); plt.plot(times_s, current_A, 'k-'); plt.xlabel('time [s]'); plt.ylabel('current I [A]'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('current_vs_time.png', dpi=150)
    print('Saved: current_vs_time.png')

    plt.figure(figsize=(6,3.8)); plt.plot(c_part_hist, V_hist, 'b.-'); plt.xlabel('particle-avg conc'); plt.ylabel('voltage [V]'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('voltage_vs_concentration.png', dpi=150)
    print('Saved: voltage_vs_concentration.png')

    plt.figure(figsize=(6,3.8)); plt.plot(c_part_hist, current_A, 'r.-'); plt.xlabel('particle-avg conc'); plt.ylabel('current [A]'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('current_vs_concentration.png', dpi=150)
    print('Saved: current_vs_concentration.png')

    # Add new plots for psi and f_pfc if available
    if psi_mean_hist is not None and f_pfc_hist is not None:
        plt.figure(figsize=(6,3.8)); plt.plot(times_s, psi_mean_hist, 'g-'); plt.xlabel('time [s]'); plt.ylabel('<psi>'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('psi_mean_vs_time.png', dpi=150)
        print('Saved: psi_mean_vs_time.png')

        plt.figure(figsize=(6,3.8)); plt.plot(times_s, f_pfc_hist, 'm-'); plt.xlabel('time [s]'); plt.ylabel('Total PFC Free Energy'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('f_pfc_vs_time.png', dpi=150)
        print('Saved: f_pfc_vs_time.png')

        # Optional: voltage vs. mean psi
        plt.figure(figsize=(6,3.8)); plt.plot(psi_mean_hist, V_hist, 'c.-'); plt.xlabel('<psi>'); plt.ylabel('voltage [V]'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('voltage_vs_psi_mean.png', dpi=150)
        print('Saved: voltage_vs_psi_mean.png')

# Update the final print statement to list all potential saved plot files
print('Analysis plots generated: analysis_fields.png')
if mech_mode == 'hetero':
    print('                      : stiffness_fields.png')
if times_s.size > 0:
    print('                      : current_vs_time.png')
    print('                      : voltage_vs_concentration.png')
    print('                      : current_vs_concentration.png')
    if psi_mean_hist is not None:
         print('                      : psi_mean_vs_time.png')
         print('                      : f_pfc_vs_time.png')
         print('                      : voltage_vs_psi_mean.png')


Loaded checkpoint: lfp_spectral_mech_checkpoint.npz
Loaded PFC parameters from checkpoint.
Saved: analysis_fields.png
Loaded updated iv_log.csv (including psi mean and f_pfc)
Saved: current_vs_time.png
Saved: voltage_vs_concentration.png
Saved: current_vs_concentration.png
Saved: psi_mean_vs_time.png
Saved: f_pfc_vs_time.png
Saved: voltage_vs_psi_mean.png
Analysis plots generated: analysis_fields.png
                      : current_vs_time.png
                      : voltage_vs_concentration.png
                      : current_vs_concentration.png
                      : psi_mean_vs_time.png
                      : f_pfc_vs_time.png
                      : voltage_vs_psi_mean.png


## Summary:

### Data Analysis Key Findings

*   The Phase Field Crystal (PFC) helper functions, including `anisotropic_laplacian_c`, `build_fft_cache_from_params`, `plot_fields`, and `pfc_relax_stable`, were successfully defined.
*   A placeholder was used for the `anisotropic_laplacian_c` function due to the absence of specific details regarding its concentration dependence and anisotropic k-space operator.
*   Initial conditions for the coupled simulation were set up by initializing PFC parameters, creating the `r_field` from the mask, building the `fft_cache`, and relaxing the initial `psi` field using the loaded concentration field `c`.
*   The `pfc_relax_stable` function successfully performed the warm-up relaxation of the `psi` field.
*   The main time integration loop was modified to include the spectral evolution of the `psi` field using `pfc_step_spectral` alongside the existing Cahn-Hilliard step.
*   A `TypeError` in the `laplace` function call within `step_CH` was diagnosed and fixed by passing the global `K2` variable.
*   An `AttributeError` (`numpy.einscom`) was identified as a typo and corrected to `np.einsum`.
*   An `IndexError` related to plotting subplots was resolved by dynamically determining the number of plots based on available fields (including `psi`) and adjusting the plotting loop and subplot indexing accordingly.
*   The plotting and analysis sections were updated to load the `psi` field from the checkpoint, include a plot of the `psi` field in `analysis_fields.png`, and generate new plots for the time evolution of mean `psi` and total PFC free energy using data from the updated `iv_log.csv`.
*   The code now saves the `psi` field, PFC parameters, and PFC timestep to the checkpoint file and logs mean `psi` and total PFC free energy to the `iv_log.csv` file during the simulation.

### Insights or Next Steps

*   The placeholder implementation for `anisotropic_laplacian_c` means the current PFC model does not include concentration-dependent anisotropy. A next step could be to obtain or implement the correct k-space operator for the anisotropic Laplacian to fully realize the intended model coupling.
*   The current implementation uses a fixed timestep for the PFC model (`pfc_spectral_dt`), which may not be optimal for stability or efficiency. Future work could explore adaptive timestepping or subcycling the PFC step relative to the Cahn-Hilliard step.


## Define pfc functions

### Subtask:
Add the Python code that defines the `pfc_step_spectral` function and any other required helper functions for the PFC model (e.g., `plot_fields` if it's part of the PFC implementation, `build_fft_cache_from_params`, `pfc_relax_stable`).

**Reasoning**:
The subtask requires defining several helper functions for the PFC model that are missing in the current notebook state, specifically `anisotropic_laplacian_c`, `build_fft_cache_from_params`, `plot_fields`, and `pfc_relax_stable`. I will define these functions in a code block.

In [63]:
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.spatial import Voronoi
from dataclasses import dataclass
import numpy as np
from scipy.spatial import cKDTree
from scipy.ndimage import distance_transform_edt
from numpy.fft import fftn, ifftn, fftfreq

# Define anisotropic_laplacian_c based on the structure implied by apply_G2
# This is a placeholder/simplified version based on common PFC structures.
# The actual implementation might need more details from the original source.
def anisotropic_laplacian_c(u, c, alpha_fp, beta_fp, alpha_lfp, beta_lfp, dx, xi):
    """
    Placeholder for anisotropic Laplacian dependent on concentration c.
    Needs actual implementation based on the specific PFC model being used.
    This simplified version assumes a spatially varying but isotropic Laplacian for now.
    """
    # Example: Simple Laplacian scaled by a concentration-dependent factor
    # In a real PFC model, this would involve the reciprocal lattice vectors
    # and their dependence on the lattice parameter which changes with c.
    # For a proper anisotropic Laplacian, one would need to implement
    # K_c(k) = \sum_i \alpha_i(c) * exp(i k . a_i(c)) as the spectral filter.
    # For now, let's use a simple diffusion-like term.
    uk = fftn(u)
    # This is a placeholder, needs the actual k-space operator for anisotropic Laplacian
    # based on the alpha/beta parameters and concentration 'c'.
    # Since the full anisotropic logic is complex and not provided, we'll use
    # an isotropic placeholder based on K2. The coefficients (alpha, beta)
    # would typically define the k-space operator.
    # A proper implementation requires understanding how alpha/beta relate to the operator in k-space.
    # Without the explicit k-space operator for the anisotropic Laplacian, we'll
    # revert to a basic isotropic Laplacian for this placeholder.
    # This will NOT capture the anisotropic behavior intended by alpha/beta.

    # isotropic Laplacian in k-space: -K2 * uk
    Lu_k = -K2 * uk # Use the K2 defined earlier in the notebook

    # The concentration dependence in an *anisotropic* Laplacian usually appears
    # in the definition of the k-space operator (e.g., through concentration-dependent
    # lattice vectors). Since we don't have that k-space operator definition here,
    # we cannot correctly implement the anisotropic part depending on 'c'.
    # The apply_G2 function structure suggests (1 - L_c)^2, where L_c is the
    # anisotropic Laplacian. If L_c in k-space is K_c(k), then the operator
    # is (1 - K_c(k))^2. K_c(k) depends on 'c' and the lattice vectors.

    # Given the limitation, we will make anisotropic_laplacian_c return
    # a simple isotropic Laplacian using the existing K2, and note that this
    # is a placeholder missing the true anisotropic and concentration-dependent behavior.
    # This makes `apply_G2` effectively `u + 2*(-laplace(u)) + laplace(laplace(u))`,
    # which is (1 - laplace)^2 u, standard for isotropic PFC.

    # Returning isotropic Laplacian for now:
    return np.real(ifftn(Lu_k))


@dataclass
class FFTCache:
    """Cache for FFT-related variables including preferred wavevector."""
    K2: np.ndarray
    KX: np.ndarray
    KY: np.ndarray
    mask0: np.ndarray # Mask for the k=0 mode
    k0_dimless: float # Dimensionless preferred wavevector magnitude


def build_fft_cache_from_params(p: Params, Wc_val: float):
    """Builds and returns an FFTCache object, calculating k0_dimless."""
    kx = 2 * np.pi * fftfreq(p.nx, d=Lx/p.nx) # Use Lx/Nx for dx in Wc units
    ky = 2 * np.pi * fftfreq(p.ny, d=Ly/p.ny) # Use Ly/Ny for dy in Wc units
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    K2 = KX**2 + KY**2
    mask0 = (np.abs(KX) < 1e-14) & (np.abs(KY) < 1e-14)
    K2[mask0] = 1e-30

    # Calculate dimensionless preferred wavevector magnitude
    # Assuming p.a0 is the preferred real-space wavelength (lattice spacing) in Wc units
    k0_dimless = 2.0 * np.pi / p.a0

    return FFTCache(K2=K2, KX=KX, KY=KY, mask0=mask0, k0_dimless=k0_dimless)

def plot_fields(c, psi, mask, title_prefix=''):
    """Plots concentration, phase field, and mask."""
    extent = [0, c.shape[0], 0, c.shape[1]] # Use array shape for extent
    fig, axs = plt.subplots(1, 3, figsize=(12, 4))

    im = axs[0].imshow(c.T, origin='lower', extent=extent, cmap='viridis', vmin=0, vmax=1)
    axs[0].set_title(f"{title_prefix} Concentration (c)")
    plt.colorbar(im, ax=axs[0])
    if mask is not None:
         axs[0].contour(mask.T, levels=[0.5], colors='red', linewidths=0.8, extent=extent)


    im = axs[1].imshow(psi.T, origin='lower', extent=extent, cmap='RdBu_r') # RdBu_r is good for phase field
    axs[1].set_title(f"{title_prefix} Phase Field (psi)")
    plt.colorbar(im, ax=axs[1])
    if mask is not None:
         axs[1].contour(mask.T, levels=[0.5], colors='red', linewidths=0.8, extent=extent)

    im = axs[2].imshow(mask.T, origin='lower', extent=extent, cmap='gray', vmin=0, vmax=1)
    axs[2].set_title("Mask (H)")
    plt.colorbar(im, ax=axs[2])


    for ax in axs.ravel():
        ax.set_xlabel("Grid units"); ax.set_ylabel("Grid units") # Use grid units as extent is based on shape

    plt.tight_layout()
    return fig # Return figure object for potential display

def pfc_relax_stable(psi, c, r_field, p, mask, n_steps, dt0, dt_max, psi_clip, enforce_amorphous, verbose_every):
    """
    Relax (warm-up) the PFC phase field towards a stable configuration
    while the concentration is held fixed. Uses spectral method.
    """
    # Ensure psi is float64 and C-contiguous
    psi = np.asarray(psi, dtype=np.float64, order='C').copy()

    # Need to build fft_cache with the updated definition that includes k0_dimless
    # Assuming p has the necessary a0 value.
    fft_cache = build_fft_cache_from_params(p, Wc_val=1.0) # Wc_val might not be needed depending on build_fft_cache_from_params

    current_dt = dt0
    for step in range(n_steps):
        try:
            # Spectral update based on the pfc_step_spectral logic
            psik = fftn(psi)
            psi_cubed = psi**3
            psi_cubed_k = fftn(psi_cubed)
            r_psi_k = fftn(r_field * psi)

            # UPDATED spectral operator using k0_dimless
            spectral_operator = (fft_cache.K2 - fft_cache.k0_dimless**2)**2

            linear_term_k = r_psi_k - spectral_operator * psik
            nonlinear_term_k = -psi_cubed_k

            # Forward Euler time integration in Fourier space
            psik_new = psik + current_dt * (linear_term_k + nonlinear_term_k)
            psi_new = np.real(ifftn(psik_new))

            # Apply mask and clipping
            if mask is not None:
                psi_new[~mask] = 0.0 # Set to zero outside the mask

            # Apply clipping
            np.clip(psi_new, -psi_clip, psi_clip, out=psi_new)

            psi = psi_new

            if not np.isfinite(psi_new).all():
                 raise FloatingPointError('PFC relax step produced NaN/Inf; lower dt.')

            if verbose_every and (step + 1) % verbose_every == 0:
                print(f"PFC relax step {step+1}/{n_steps}, dt={current_dt:.2e}")

            current_dt = min(dt_max, current_dt * 1.05) # Grow dt slightly

        except FloatingPointError as e:
            print(f"Floating point error at step {step+1}: {e}. Reducing dt and retrying step.")
            current_dt *= 0.5 # Reduce dt
            psi = np.asarray(psi, dtype=np.float64, order='C').copy() # Revert or reload psi if needed
            if current_dt < 1e-8: # Prevent infinite loop with tiny dt
                 print("dt too small, stopping relaxation.")
                 break
            continue # Retry the same step with smaller dt


    return psi

# Define the pfc_step_spectral function directly in the notebook
# This version implements the spectral solver logic
def pfc_step_spectral(psi, c, r_field, p, cache, mask, dt, psi_clip=1.8):
    """Performs one spectral step for the PFC equation.

    Args:
        psi: The phase field.
        c: The concentration field.
        r_field: The spatially varying r parameter.
        p: The parameters object.
        cache: The FFTCache object (must include k0_dimless).
        mask: The mask defining the active region.
        dt: The time step size.
        psi_clip: Value to clip psi to.

    Returns:
        The updated psi field.
    """
    # Ensure psi is float64 and C-contiguous
    psi = np.asarray(psi, dtype=np.float64, order='C').copy()

    # Transform psi to Fourier space
    psik = fftn(psi)

    # Calculate the right-hand side in Fourier space
    # PFC equation: d(psi)/dt = (r - (nabla^2 + k0^2)^2) psi - psi^3
    # In Fourier space: d(psik)/dt = fft(r*psi) - (K2 - k0_dimless^2)^2 * psik - fft(psi^3)

    psi_cubed = psi**3
    psi_cubed_k = fftn(psi_cubed)

    # r_field is spatially varying, so r*psi needs to be calculated in real space and then FFT'd
    r_psi_k = fftn(r_field * psi)

    # UPDATED spectral operator using k0_dimless
    spectral_operator = (cache.K2 - cache.k0_dimless**2)**2

    # Linear term in Fourier space: fft(r*psi) - (K2 - k0_dimless^2)^2 * psik
    linear_term_k = r_psi_k - spectral_operator * psik

    # Nonlinear term in Fourier space: -fft(psi^3)
    nonlinear_term_k = -psi_cubed_k

    # Forward Euler time integration in Fourier space
    psik_new = psik + dt * (linear_term_k + nonlinear_term_k)

    # Transform back to real space
    psi_new = np.real(ifftn(psik_new))

    # Apply mask and clipping
    if mask is not None:
        psi_new[~mask] = 0.0 # Set to zero outside the mask

    # Apply clipping
    np.clip(psi_new, -psi_clip, psi_clip, out=psi_new)

    # Optional: Handle enforce_amorphous if needed. This is complex in spectral space.
    # For this subtask, we will skip the enforce_amorphous logic in the spectral step.

    if not np.isfinite(psi_new).all():
        raise FloatingPointError('PFC spectral step produced NaN/Inf; lower dt.')

    return psi_new

## Initialize pfc variables

### Subtask:
Generate code to properly initialize the `fft_cache` using a function like `build_fft_cache_from_params`, initialize the `r_field` based on the `Params` and the mask, and initialize the PFC field `psi`.

**Reasoning**:
Initialize the r_field based on the Params and mask, build the fft_cache, and initialize and relax the psi field using the defined functions.

In [62]:
# --- Build params & fields ---

# Define the time step for the spectral PFC solver
pfc_spectral_dt = 1e-3 # Choose a suitable time step for the spectral solver

# Use the existing Nx, Ny from the loaded checkpoint data
# Harmonize gamma based on sigma, Wc, and a0
# Simplified relationship: gamma ~ sigma * a0^2 / Wc
# We need a0 from Params, and sigma, Wc from Model
# Let's set a baseline a0 in Params and use Model's sigma and Wc
# Note: This is a simplified scaling. A rigorous harmonization requires
# deriving the relationship from the free energy functionals.
a0_val = 4.788 # Use the baseline a0 value from Params definition

# Calculate a harmonized gamma
# We need sigma and Wc from the global Model variable
if 'Model' in globals():
    sigma_val = float(Model.get("sigma", 0.072))
    Wc_val = float(Model.get("Wc", 1e-9))
    # Use a scaling factor to get gamma in the right dimensionless range
    # This scaling factor might need tuning. Let's use Hscale as a reference.
    # Hscale is sigma/Wc. So gamma ~ Hscale * a0^2 / Wc^2 ? Or just scale by Hscale?
    # Let's try scaling gamma by Hscale and relate it to a0.
    # A common PFC wavelength is 2*pi/k0. If k0 ~ 1/a0, then k0^2 ~ 1/a0^2.
    # The (1+k^2)^2 term in PFC has a characteristic scale of 1.
    # Maybe gamma should scale with Hscale * (Wc/a0)^2 to be consistent?
    # Let's try a simpler approach: scale gamma by Hscale and adjust the base value.
    # The base PFC energy density is ~ gamma * (psi^2 + psi (1-Laplace)^2 psi).
    # If psi ~ 1, this is ~ gamma. CH energy density is ~ Hscale * c(1-c).
    # So maybe gamma should be roughly proportional to Hscale?
    # Let's try setting gamma relative to Hscale and the square of the preferred wavevector magnitude.
    # The preferred wavevector is often set to 1 in dimensionless PFC, corresponding to a0.
    # If a0 is the characteristic length, k0 ~ 2*pi/a0.
    # The dimensionless k in our code is scaled by Wc. So the dimensionless preferred k is ~ a0/Wc.
    # The spectral operator is (1 + k_dimless^2)^2. If k_dimless_preferred ~ a0/Wc,
    # then the characteristic scale of (1+k^2)^2 is (1 + (a0/Wc)^2)^2.
    # This is getting complicated without a clear derivation.
    # Let's use a simpler empirical approach based on typical dimensionless PFC values and Hscale.
    # We know Hscale is sigma/Wc. Let's try to set gamma such that the energy scales are similar.
    # A typical dimensionless gamma might be around 1. Let's scale it by Hscale.
    # This might make gamma very large. Let's rethink.

    # The dimensionless parameters in the original CH part are scaled by Hscale = sigma/Wc.
    # RTv = (R * To / vm) / Hscale
    # Om = (Omega / vm) / Hscale
    # Dm = 1.0 / RTv
    # kappa_dimless = kappa_dim / (Hscale * Wc**2)

    # Let's try to define dimensionless PFC parameters in a similar spirit.
    # The PFC free energy density is f_pfc = gamma * [0.5*psi(r - (1-nabla^2)^2)psi + 0.25*psi^4]
    # We want f_pfc to be in units of Hscale.
    # So maybe the dimensionless gamma should be related to the dimensional gamma / Hscale?
    # Or perhaps the dimensional PFC parameters should be defined first, then non-dimensionalized.

    # Let's stick to the current dimensionless framework where CH parameters are scaled by Hscale.
    # We need dimensionless PFC parameters: gamma, kappa, xi, r.
    # The current Params class already defines these. The issue is their values.
    # Let's assume the values in Params are *already* dimensionless and need to be consistent
    # with the dimensionless CH parameters (RTv, Om, Dm, kappa_dimless) and length scales (Lx, Ly, dx, dy).

    # Harmonization Strategy:
    # 1. Keep the dimensionless CH parameters (RTv, Om, Dm, kappa_dimless) as calculated.
    # 2. Keep the dimensionless length scales (Lx, Ly, dx, dy).
    # 3. Set dimensionless PFC parameters (gamma, kappa, xi) based on typical PFC values
    #    and potentially some relation to the CH length/energy scales if a clear one exists.
    #    For simplicity, let's use typical dimensionless PFC values for gamma, kappa, xi for now,
    #    as deriving them from CH parameters is complex without a specific theoretical model.
    # 4. Make the dimensionless r parameter field (r_field) depend on the dimensionless concentration c.

    # Let's revise the Params initialization to reflect this:
    p = Params(nx=Nx, ny=Ny,
               Lx=Lx, Ly=Ly, # Use domain size for Params
               dx=1.0, # Assuming dx=dy=1.0 in PFC grid units for simplicity in Params
               # Typical dimensionless PFC values (these might need tuning)
               kappa=1.0,
               gamma=1.0,
               xi=1.0,
               # r values for concentration coupling
               r_FP=-0.2,  # r at c=0 (favors crystal)
               r_LFP=-0.3  # r at c=1 (favors crystal, potentially different lattice constant)
              )
    # Note: The dx=1.0 in Params means the PFC grid is assumed to have unit spacing
    # in its own dimensionless units. The relationship between these units and the
    # CH Wc units is implicitly handled by the domain size (Lx, Ly) and grid points (Nx, Ny).
    # The k in the PFC operator (1+k^2)^2 is in units of 1/dx_pfc.
    # The k in the CH operator (K2) is in units of 1/Wc.
    # So, there's a potential mismatch in how k-spaces are defined if dx_pfc != Wc.
    # In this spectral code, the grid points Nx, Ny and domain size Lx, Ly in Wc units
    # define the k-space (KX, KY, K2). The PFC spectral operator (1+K2)^2 uses this K2.
    # This implies the characteristic length scale of the PFC model (related to 1/sqrt(kappa) or 1/k0)
    # is being implicitly defined in Wc units.
    # If the preferred PFC wavelength is a0, and we are using K2 in 1/Wc units,
    # then the preferred dimensionless wavevector magnitude should be k0_dimless ~ a0/Wc.
    # The isotropic PFC operator (1+k^2)^2 has a minimum at k=1. So if we want the minimum
    # to be at k0_dimless, the operator should be (k^2 - k0_dimless^2)^2 or similar.
    # The current (1+K2)^2 implies a preferred wavevector magnitude of 1 in Wc units.
    # This means the characteristic PFC length scale is implicitly 1 Wc.
    # This is a significant point for harmonization! If the physical lattice spacing a0 is
    # different from Wc, the (1+K2)^2 operator is not correctly representing the PFC physics.

    # Let's adjust the PFC spectral operator to have a minimum at a wavevector magnitude
    # corresponding to the dimensionless preferred wavelength (related to a0).
    # Preferred dimensionless wavevector magnitude k0_dimless = (2*pi/a0_dimensional) * Wc
    # Assuming a0_dimensional is p.a0 * Wc (if p.a0 is a dimensionless ratio to Wc)
    # Or if p.a0 is already in Wc units, then k0_dimless = 2*pi / p.a0
    # Let's assume p.a0 is in Wc units. Then k0_dimless = 2*pi / p.a0
    # The PFC operator should be (K2 - k0_dimless^2)^2 or similar.
    # A more standard form is (nabla^2 + k0^2)^2. In k-space: (-k^2 + k0^2)^2 = (k^2 - k0^2)^2.
    # So the spectral operator should be (K2 - k0_dimless**2)**2.

    # Calculate k0_dimless
    k0_dimless = 2.0 * np.pi / p.a0 # Assuming p.a0 is in Wc units

    # Modify build_fft_cache_from_params to store k0_dimless and update pfc_step_spectral
    # to use (cache.K2 - cache.k0_dimless**2)**2 as the spectral operator.

    # Let's update the FFTCache and build_fft_cache_from_params
    @dataclass
    class FFTCache:
        """Cache for FFT-related variables including preferred wavevector."""
        K2: np.ndarray
        KX: np.ndarray
        KY: np.ndarray
        mask0: np.ndarray # Mask for the k=0 mode
        k0_dimless: float # Dimensionless preferred wavevector magnitude

    def build_fft_cache_from_params(p: Params, Wc_val: float):
        """Builds and returns an FFTCache object, calculating k0_dimless."""
        kx = 2 * np.pi * fftfreq(p.nx, d=Lx/p.nx) # Use Lx/Nx for dx in Wc units
        ky = 2 * np.pi * fftfreq(p.ny, d=Ly/p.ny) # Use Ly/Ny for dy in Wc units
        KX, KY = np.meshgrid(kx, ky, indexing='ij')
        K2 = KX**2 + KY**2
        mask0 = (np.abs(KX) < 1e-14) & (np.abs(KY) < 1e-14)
        K2[mask0] = 1e-30

        # Calculate dimensionless preferred wavevector magnitude
        # Assuming p.a0 is the preferred real-space wavelength (lattice spacing) in Wc units
        k0_dimless = 2.0 * np.pi / p.a0

        return FFTCache(K2=K2, KX=KX, KY=KY, mask0=mask0, k0_dimless=k0_dimless)

    # Now, initialize p and fft_cache using the updated definition and function
    p = Params(nx=Nx, ny=Ny,
               Lx=Lx, Ly=Ly,
               dx=1.0, # This dx in Params is likely a nominal value for PFC internal units,
                       # but the spectral solver uses the grid defined by Nx,Ny,Lx,Ly.
                       # Let's keep it but note its role is unclear in the spectral code.
               # Set gamma based on Hscale (sigma/Wc) and the preferred wavevector k0_dimless
               # A simple scaling might be gamma ~ Hscale / k0_dimless^4 or Hscale * a0^4
               # Let's try setting gamma to a typical dimensionless value, but note
               # that a rigorous link to Hscale is missing in this simplified harmonization.
               gamma=1.0, # Keep as a typical dimensionless value for now
               kappa=1.0, # Keep as a typical dimensionless value for now
               xi=1.0,    # Keep as a typical dimensionless value for now
               # r values for concentration coupling
               r_FP=-0.2,
               r_LFP=-0.3
              )

    # Build the FFT cache using the updated function, passing Wc_val if needed for k0_dimless calc
    # Based on the updated build_fft_cache_from_params, it uses p.a0 directly, which is assumed in Wc units.
    fft_cache = build_fft_cache_from_params(p, Wc_val=1.0) # Wc_val not strictly needed in the updated function

    # Define r_field as a linear interpolation based on concentration c
    # r(c) = r_FP * (1-c) + r_LFP * c
    # Ensure c is clipped to [0, 1] for the interpolation
    c_clipped = np.clip(c, 0.0, 1.0)
    r_field = p.r_FP * (1.0 - c_clipped) + p.r_LFP * c_clipped
    r_field = r_field.astype(float) # Ensure float type

    # Convert the float mask to a boolean mask for init_poly_voronoi_simple
    boolean_mask = mask > 0.5

    # Optional: seed psi with a simple polycrystal to avoid grid locking
    # Use the preferred wavevector magnitude k0_dimless from the cache for initialization k
    psi0, labels, seeds, thetas = init_poly_voronoi_simple(
        p, boolean_mask, n_grains=8, amp=0.30, k=fft_cache.k0_dimless, band_px=1, seed=4 # Use k0_dimless for k
    )

    # Short ψ warm-up at fixed c using the defined pfc_relax_stable function
    # This helps the initial random psi relax into a more stable lattice structure
    # before coupling with CH dynamics.
    # Need to update pfc_relax_stable to use the new FFTCache structure
    # Need to update pfc_step_spectral to use (cache.K2 - cache.k0_dimless**2)**2

    # Let's update pfc_step_spectral and pfc_relax_stable in the previous cell (2a9181c3) first.
    # Then, we can run this cell.

    # For now, let's just run the initialization of p, r_field, fft_cache, psi0
    # and the initial plot, assuming the functions will be updated next.

    # Short ψ warm-up with the OLD pfc_relax_stable (will fail or be wrong)
    # We need to update the functions first.

    # Let's pause here and wait for the next turn to update the functions
    # before executing the relaxation and time loop.

    print("PFC parameters and r_field initialized based on simplified harmonization.")
    print(f"Calculated k0_dimless: {fft_cache.k0_dimless:.3f}")
    print("Next step: Update pfc_step_spectral and pfc_relax_stable to use the new FFTCache and spectral operator.")

    # Keep the plot_fields call commented out until functions are updated
    # _ = plot_fields(c, psi, boolean_mask, title_prefix='Initial (after relax)')
    # plt.show()

PFC parameters and r_field initialized based on simplified harmonization.
Calculated k0_dimless: 1.312
Next step: Update pfc_step_spectral and pfc_relax_stable to use the new FFTCache and spectral operator.


## Integrate pfc into time stepping

### Subtask:
Modify the main time integration loop to include the evolution of the `psi` field using the `pfc_step_spectral` function alongside the existing Cahn-Hilliard step. This will likely involve passing `psi` to the `step_CH` function or calling the PFC step within the loop.

In [48]:
# Chemical free energy and CH operators (dimensionless)
_eps_clip = 1e-12

def f_chem(c):
    ce = np.clip(c, _eps_clip, 1.0-_eps_clip)
    return RTv*(ce*np.log(ce) + (1-ce)*np.log(1-ce)) + Om*ce*(1.0-ce)

def dfdc_chem(c):
    ce = np.clip(c, _eps_clip, 1.0-_eps_clip)
    return RTv*(np.log(ce) - np.log(1.0-ce)) + Om*(1.0 - 2.0*ce)


# Modified laplace function call to include K2
def laplace(f, K2_val): # Renamed argument to avoid conflict with global K2
    return np.real(ifftn(-K2_val*fftn(f)))

# BV kinetics (dimensionless)

def J_BV(mu):
    eta = (mu_elec - mu) / RTv
    eta_clip = float(Model.get('BV_eta_clip', 40.0))
    eta = np.clip(eta, -eta_clip, eta_clip)
    # J = j0*(e^{αη} - e^{-(1-α)η}) = j0 * e^{(α-(1-α))η} * 2*sinh(η/2)
    return j0coeff * (np.exp((2*alpha-1.0)*eta) * 2.0*np.sinh(0.5*eta))

# Ak = 1.0 + float(Interval['timestep'])*Mlin*(K2**2) # Original calculation
# Modified Ak calculation using Dm and kappa_dimless separately
# Use the global K2 here
Ak = 1.0 + float(Interval['timestep']) * (Dm * K2 + kappa_dimless * K2**2)

def step_CH(c, dt):
    mu_el, f_el = mech_mu_f(c)
    # The chemical potential calculation remains the same
    # Pass the global K2 to the laplace function
    mu = dfdc_chem(c) - laplace(c, K2) + mu_el
    Mc = Dm*(H * c * (1.0 - c))   # diffusion only in particle
    div_term = divergence_of_M_grad_mu(Mc, mu)
    J = J_BV(mu)
    Rsrc = J * delta_Gamma
    rhs = c + dt*(div_term + Rsrc)
    # The spectral update uses the new Ak
    c_new = np.real(ifftn(fftn(rhs)/Ak))
    # impose reservoir composition outside
    cout = float(Model.get('c_outside', 1.0))
    c_new = H*c_new + (1.0 - H)*cout
    return np.clip(c_new, 1e-8, 1.0-1e-8), {'mu':mu, 'mu_el':mu_el, 'f_el':f_el, 'J':J}

## Update plotting and analysis

### Subtask:
Modify the plotting and analysis sections to visualize the `psi` field and potentially other PFC-related quantities, and update any analysis that depends on the coupled fields.

**Reasoning**:
Modify the plot_fields script to load and plot the psi field and other relevant PFC quantities from the updated checkpoint file, and update the analysis plots to include mean psi and PFC free energy over time from the updated log file.

In [68]:
# --- plot_fields.py modifications (integrated into notebook) ---
# This block replaces or augments the original plot_fields.py logic
# to be executed directly in the notebook.

# Re-load the checkpoint data to ensure psi and PFC params are available
try:
    data = np.load('lfp_spectral_mech_checkpoint.npz', allow_pickle=True)
    print("Loaded checkpoint: lfp_spectral_mech_checkpoint.npz")
except FileNotFoundError:
    print("Checkpoint file not found. Cannot generate analysis plots.")
    # Exit or handle the error appropriately, e.g., by skipping plot generation
    # For this task, we assume the checkpoint exists after the simulation run.
    raise

# --- pull grid & scales from checkpoint (updated to include psi and PFC params) ---
c  = data['c']
psi = data['psi'] if 'psi' in data.files else None # Load psi if available
Nx = int(data['Nx']); Ny = int(data['Ny'])
Lx = float(data['Lx']); Ly = float(data['Ly'])

RTv = float(data['RTv']); Om = float(data['Om'])
H   = data['H'] if 'H' in data.files else np.ones((Nx,Ny)) # Default to all ones if H is missing

# mechanical mode and stiffness info (kept for compatibility)
mech_mode = str(data['mech_mode']) if 'mech_mode' in data.files else 'uniform' # Default if missing

if 'lam_d' in data.files and 'mu_d' in data.files:
    # Legacy: single uniform stiffness
    lam_p_d = float(data['lam_d']); mu_p_d = float(data['mu_d'])
    lam_r_d, mu_r_d = lam_p_d, mu_p_d
elif 'lam_p_d' in data.files and 'mu_p_d' in data.files:
    lam_p_d = float(data['lam_p_d']); mu_p_d = float(data['mu_p_d'])
    lam_r_d = float(data['lam_r_d']); mu_r_d = float(data['mu_r_d'])
else:
     # Attempt to load from Model if not in checkpoint (less ideal but a fallback)
     print("Elastic parameters not found in checkpoint; attempting to load from Model in config_mech.py")
     try:
         spec = importlib.util.spec_from_file_location('config_mech', os.path.join(os.getcwd(), 'config_mech.py'))
         config_mech_loaded = importlib.util.module_from_spec(spec)
         sys.modules['config_mech_loaded'] = config_mech_loaded # Use a different name to avoid conflict
         assert spec.loader is not None
         spec.loader.exec_module(config_mech_loaded)
         Model_loaded = getattr(config_mech_loaded, 'Model')
         E_p_loaded = float(Model_loaded["E"]); nu_p_loaded = float(Model_loaded.get('nu', Model_loaded.get('ν', 0.25)))
         E_r_loaded  = float(Model_loaded.get("E_res", E_p_loaded)); nu_r_loaded = float(Model_loaded.get('nu_res', Model_loaded.get('ν_res', nu_p_loaded)))
         lam_p_loaded, mu_p_loaded = lam_mu_from_E_nu(E_p_loaded, nu_p_loaded)
         lam_r_loaded, mu_r_loaded = lam_mu_from_E_nu(E_r_loaded, nu_r_loaded)
         lam_p_d, mu_p_d = lam_p_loaded / Hscale, mu_p_loaded / Hscale
         lam_r_d, mu_r_d = lam_r_loaded / Hscale, mu_r_loaded / Hscale
     except Exception as e:
         print(f"Could not load elastic parameters from config_mech.py: {e}")
         # Set defaults if loading fails
         lam_p_d, mu_p_d = 1.0, 1.0 # Placeholder defaults
         lam_r_d, mu_r_d = 1.0, 1.0 # Placeholder defaults


# eigenstrain (from config, as it might not be in checkpoint)
try:
    spec = importlib.util.spec_from_file_location('config_mech', os.path.join(os.getcwd(), 'config_mech.py'))
    config_mech_loaded = importlib.util.module_from_spec(spec)
    sys.modules['config_mech_loaded_eigen'] = config_mech_loaded # Use a different name
    assert spec.loader is not None
    spec.loader.exec_module(config_mech_loaded)
    Model_loaded = getattr(config_mech_loaded, 'Model')
    e11 = float(Model_loaded.get('e11', 0.0)); e22 = float(Model_loaded.get('e22', 0.0))
    Eps0 = np.zeros((2,2)); Eps0[0,0]=e11; Eps0[1,1]=e22
except Exception as e:
    print(f"Could not load eigenstrain parameters from config_mech.py: {e}")
    Eps0 = np.zeros((2,2)) # Default to zero eigenstrain


# PFC parameters and r_field (load from checkpoint if available, otherwise reconstruct)
pfc_params_raw = data.get('pfc_params', None)
r_field = data.get('r_field', None)

if pfc_params_raw is not None:
    # Extract the dictionary from the NumPy array
    if isinstance(pfc_params_raw, np.ndarray) and pfc_params_raw.shape == ():
        pfc_params = pfc_params_raw.item() # Extract the single item (dictionary)
    elif isinstance(pfc_params_raw, dict):
        pfc_params = pfc_params_raw
    else:
        print(f"Unexpected type or shape for pfc_params in checkpoint: {type(pfc_params_raw)}, shape: {getattr(pfc_params_raw, 'shape', 'N/A')}. Reconstructing.")
        pfc_params = None # Fallback to reconstruction

    if pfc_params is not None:
        try:
            p = Params(**pfc_params)
            print("Loaded PFC parameters from checkpoint.")
            if r_field is None:
                 # Reconstruct r_field using the loaded params and mask
                 print("r_field not found in checkpoint; reconstructing from mask and loaded params.")
                 boolean_mask = H > 0.5 # Use the loaded H mask
                 r_field = np.where(boolean_mask, p.r_inside, p.r_outside).astype(float)
        except Exception as e:
            print(f"Error unpacking loaded pfc_params into Params: {e}. Reconstructing.")
            pfc_params = None # Fallback to reconstruction

if pfc_params is None:
    # Reconstruct PFC parameters and r_field if not successfully loaded
    print("PFC parameters not loaded/reconstructed from checkpoint; reconstructing from default/config values.")
    p = Params(nx=Nx, ny=Ny) # Use default or load from config if needed
    boolean_mask = H > 0.5 # Use the loaded H mask
    r_field = np.where(boolean_mask, p.r_inside, p.r_outside).astype(float)
    # Note: This reconstruction might not match the parameters used to generate
    # the psi field if the checkpoint was from an older run without PFC params saved.


# Need KX, KY, K2 for calculations
KX, KY, K2 = build_k_operators(Nx, Ny, Lx, Ly) # Use the existing build_k_operators function

# reference operator in k-space (rebuild as it depends on loaded elastic params)
if mech_mode == 'uniform':
    lam0_d, mu0_d = lam_p_d, mu_p_d   # use particle as reference
else:
    lam0_d, mu0_d = lam_r_d, mu_r_d   # reservoir as reference
C0 = C_iso(lam0_d, mu0_d)

K = np.stack((KX, KY), axis=-1)
# Corrected typo: einsum instead of einscom
A = np.einsum('...j,ijkl,...k->...il', K, C0, K)
A11=A[...,0,0]; A12=A[...,0,1]; A21=A[...,1,0]; A22=A[...,1,1]
detA = A11*A22 - A12*A21
mask0 = (np.abs(KX)<1e-14) & (np.abs(KY)<1e-14)
detA[mask0] = 1.0 # Handle k=0 for inverse
invA = np.empty_like(A)
invA[...,0,0] = A22/detA; invA[...,0,1] = -A12/detA
invA[...,1,0] = -A21/detA; invA[...,1,1] = A11/detA


# helpers for mechanics (re-defined to use loaded parameters)
def solve_uniform_mu_el(c_field):
    C_part = C_iso(lam_p_d, mu_p_d)
    c_eff = H * c_field  # eigenstrain only inside
    E0 = np.zeros((Nx,Ny,2,2))
    E0[...,0,0] = c_eff*Eps0[0,0]
    E0[...,1,1] = c_eff*Eps0[1,1]
    E0k = np.zeros_like(E0, dtype=complex)
    for a in range(2):
        for b in range(2):
            E0k[...,a,b] = fftn(E0[...,a,b])
    b = 1j*np.einsum('...j,ijkl,...kl->...i', K, C_part, E0k)
    u_k = np.einsum('...ij,...j->...i', invA, b)
    u_k[mask0,...]=0.0
    Ux = np.real(ifftn(u_k[...,0])); Uy = np.real(ifftn(u_k[...,1]))
    Exx,Eyy,Exy = strain_from_u(Ux,Uy,KX,KY)
    DE = np.zeros_like(E0)
    DE[...,0,0]=Exx-E0[...,0,0]; DE[...,1,1]=Eyy-E0[...,1,1]
    DE[...,0,1]=Exy; DE[...,1,0]=Exy
    sigma = np.einsum('ijkl,...kl->...ij', C_part, DE)
    f_el = 0.5*np.einsum('...ij,ijkl,...kl->...', DE, C_part, DE)
    mu_el = -(sigma[...,0,0]*Eps0[0,0] + sigma[...,1,1]*Eps0[1,1] + 2.0*sigma[...,0,1]*Eps0[0,1])
    mu_el *= H; f_el *= H
    return mu_el, f_el

def solve_hetero_mu_el(c_field, tol=1e-6, maxit=100, omega=0.7):
    lamF = lam_r_d + H*(lam_p_d - lam_r_d)
    muF  = mu_r_d  + H*(mu_p_d  - mu_r_d)
    c_eff = H*c_field
    E0xx = c_eff*Eps0[0,0]; E0yy = c_eff*Eps0[1,1]; E0xy = 0.0
    Ux = np.zeros((Nx,Ny)); Uy = np.zeros((Nx,Ny))
    def stress(Exx,Eyy,Exy):
        dExx=Exx-E0xx; dEyy=Eyy-E0yy; dExy=Exy-E0xy
        tr = dExx+dEyy
        sxx = 2*muF*dExx + lamF*tr
        syy = 2*muF*dEyy + lamF*tr
        sxy = 2*muF*dExy
        return sxx, syy, sxy
    # initial residual norm
    sxx,syy,sxy = stress(0.0,0.0,0.0)
    sxxk, syyk, sxyk = fftn(sxx), fftn(syy), fftn(sxy)
    gkx = 1j*(KX*sxxk + KY*sxyk)
    gky = 1j*(KX*sxyk + KY*syyk)
    g0 = np.sqrt(np.mean(np.abs(gkx)**2 + np.abs(gky)**2)) or 1.0
    for it in range(maxit):
        Exx,Eyy,Exy = strain_from_u(Ux,Uy,KX,KY)
        sxx,syy,sxy = stress(Exx,Eyy,Exy)
        sxxk, syyk, sxyk = fftn(sxx), fftn(syy), fftn(sxy)
        gkx = 1j*(KX*sxxk + KY*sxyk)
        gky = 1j*(KX*sxyk + KY*syyk)
        rhsx, rhsy = -gkx, -gky
        dux_k = invA[...,0,0]*rhsx + invA[...,0,1]*rhsy
        duy_k = invA[...,1,0]*rhsx + invA[...,1,1]*rhsy
        dux_k[mask0] = 0.0
        duy_k[mask0] = 0.0
        Ux += omega*np.real(ifftn(dux_k))
        Uy += omega*np.real(ifftn(duy_k))
        res = np.sqrt(np.mean(np.abs(gkx)**2 + np.abs(gky)**2))/g0
        if res < tol:
            break
    Exx,Eyy,Exy = strain_from_u(Ux,Uy,KX,KY)
    dExx=Exx-E0xx; dEyy=Eyy-E0yy; dExy=Exy
    tr=dExx+dEyy
    f_el = 0.5*(2*muF*(dExx**2 + dEyy**2 + 2*dExy**2) + lamF*(tr**2))
    sxx,syy,sxy = stress(Exx,Eyy,Exy)
    mu_el = -(sxx*Eps0[0,0] + syy*Eps0[1,1] + 2.0*sxy*Eps0[0,1])
    return mu_el, f_el, lamF, muF

def mech_mu_f(c_field): # Renamed argument to avoid conflict with global c
    if mech_mode == 'hetero':
        # Need to pass hetero_tol, hetero_maxit, hetero_omega from Model,
        # which might not be in checkpoint. Load from config as a fallback.
        try:
            spec = importlib.util.spec_from_file_location('config_mech', os.path.join(os.getcwd(), 'config_mech.py'))
            config_mech_loaded = importlib.util.module_from_spec(spec)
            sys.modules['config_mech_loaded_hetero'] = config_mech_loaded # Use a different name
            assert spec.loader is not None
            spec.loader.exec_module(config_mech_loaded)
            Model_loaded = getattr(config_mech_loaded, 'Model')
            hetero_tol = float(Model_loaded.get('hetero_tol',1e-6))
            hetero_maxit = int(Model_loaded.get('hetero_maxit',100))
            hetero_omega = float(Model_loaded.get('hetero_omega',0.7))
        except Exception as e:
            print(f"Could not load heterogeneous parameters from config_mech.py: {e}. Using defaults.")
            hetero_tol = 1e-6; hetero_maxit = 100; hetero_omega = 0.7

        return solve_elastic_hetero(c_field, tol=hetero_tol, maxit=hetero_maxit, omega=hetero_omega)
    else:
        return solve_elastic_uniform(c_field)

# chemical free-energy pieces (re-defined to use loaded RTv and Om)
def f_chem_loaded(c_field):
    ce = np.clip(c_field, 1e-12, 1-1e-12)
    return RTv*(ce*np.log(ce)+(1-ce)*np.log(1-ce)) + Om*ce*(1-ce)

def dfdc_chem_loaded(c_field):
    ce = np.clip(c_field, 1e-12, 1-1e-12)
    return RTv*(np.log(ce)-np.log(1-ce)) + Om*(1-2*ce)

# gradient energy density (re-defined to use loaded K2)
def f_grad_loaded(c_field):
    ck = fftn(c_field)
    dcx = np.real(ifftn(1j*KX*ck)); dcy = np.real(ifftn(1j*KY*ck))
    return 0.5*(dcx*dcx + dcy*dcy)

# --- plots ---
extent = [0, Lx, 0, Ly] # Use domain size for extent

# Determine the number of subplots needed
plot_titles = ['c', 'μ', 'μ_el', 'f_el', 'f_chem', 'f_grad (CH)']
plot_fields_list = [c, mu_loaded, mu_el, f_el_loaded, f_chem_loaded_field, f_grad_loaded_field]
plot_cmaps = ['viridis', 'viridis', 'viridis', 'viridis', 'viridis', 'viridis'] # Default cmaps

if psi is not None:
    plot_titles.insert(3, 'ψ') # Insert psi title after mu_el
    plot_fields_list.insert(3, psi) # Insert psi field after mu_el
    plot_cmaps.insert(3, 'RdBu_r') # Insert RdBu_r cmap for psi

# Add PFC energy density plots if calculated/needed
# To calculate these, we need apply_G2 and p.
# Assuming p was loaded or reconstructed successfully.
# if psi is not None and p is not None and r_field is not None:
#     try:
#         # Need build_fft_cache_from_params to get cache for apply_G2 if not already available
#         # Assuming fft_cache is available from previous steps or can be built
#         if 'fft_cache' not in locals() or fft_cache.K2.shape != K2.shape:
#              fft_cache = build_fft_cache_from_params(p) # Build if not available or shape mismatch
#
#         G2psi = apply_G2(psi, c, p.alpha_fp, p.beta_fp, p.alpha_lfp, p.beta_lfp, p.dx, p.xi) # c is global, use the loaded one
#         f_pfc_quad = 0.5 * (r_field * psi**2 + psi * G2psi)
#         f_pfc_quar = 0.25 * psi**4
#         f_pfc_total_density = f_pfc_quad + f_pfc_quar
#
#         plot_titles.extend(['f_quad (PFC)', 'f_quar (PFC)', 'f_total (PFC)'])
#         plot_fields_list.extend([f_pfc_quad, f_pfc_quar, f_pfc_total_density])
#         plot_cmaps.extend(['viridis', 'viridis', 'viridis'])
#     except Exception as e:
#          print(f"Could not calculate PFC energy densities for plotting: {e}")


num_plots = len(plot_titles)
n_cols = 3
n_rows = (num_plots + n_cols - 1) // n_cols # Ceiling division

fig,axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3.5), squeeze=False) # squeeze=False ensures axs is always 2D

axs_flat = axs.ravel()

# Plot each field
for i in range(num_plots):
    im = axs_flat[i].imshow(plot_fields_list[i].T, origin='lower', extent=extent, cmap=plot_cmaps[i])
    axs_flat[i].set_title(plot_titles[i]); plt.colorbar(im, ax=axs_flat[i]);
    # Overlay mask on relevant plots (c and psi)
    if plot_titles[i] in ['c', 'ψ'] and H is not None:
         axs_flat[i].contour(H.T, levels=[0.5], colors='red', linewidths=0.8, extent=extent)


# Hide any unused subplots
for i in range(num_plots, len(axs_flat)):
    fig.delaxes(axs_flat[i])


for ax in axs.ravel():
    # Check if ax is not None (in case some were deleted)
    if ax:
        ax.set_xlabel('x [Wc]'); ax.set_ylabel('y [Wc]')

plt.tight_layout()
plt.savefig('analysis_fields.png', dpi=150)
print('Saved: analysis_fields.png')

if mech_mode == 'hetero':
    # Reconstruct lamF and muF for plotting if needed (only if mech_mode is hetero)
    if 'lamF' not in locals() or 'muF' not in locals():
         print("Reconstructing stiffness fields for plotting.")
         lamF = lam_r_d + H*(lam_p_d - lam_r_d)
         muF  = mu_r_d  + H*(mu_p_d  - mu_r_d)

    fig2,axs2 = plt.subplots(1,2,figsize=(9,3.6))
    im = axs2[0].imshow(lamF.T, origin='lower', extent=extent)
    axs2[0].set_title('λ(x) (dimless)'); plt.colorbar(im, ax=axs2[0])
    im = axs2[1].imshow(muF.T, origin='lower', extent=extent)
    axs2[1].set_title('μ(x) (dimless)'); plt.colorbar(im, ax=axs2[1])
    for ax in axs2:
        ax.contour(H.T, levels=[0.5], colors='red', linewidths=0.8, extent=extent)
        ax.set_xlabel('x [Wc]'); ax.set_ylabel('y [Wc]')
    plt.tight_layout(); plt.savefig('stiffness_fields.png', dpi=150)
    print('Saved: stiffness_fields.png')


# --- Analysis plots modifications ---

# Load the updated iv_log.csv which now includes psi_mean and f_pfc
try:
    log_data = np.loadtxt('iv_log.csv', delimiter=',', skiprows=1)
    # Assuming the columns are: time_s, current_A, c_particle, c_domain, voltage_V, psi_mean, f_pfc
    times_s = log_data[:, 0]
    current_A = log_data[:, 1]
    c_part_hist = log_data[:, 2]
    c_dom_hist = log_data[:, 3]
    V_hist = log_data[:, 4]
    # Check if the log file has enough columns for psi_mean and f_pfc
    if log_data.shape[1] > 5:
        psi_mean_hist = log_data[:, 5]
        f_pfc_hist = log_data[:, 6]
        print("Loaded updated iv_log.csv (including psi mean and f_pfc)")
    else:
        psi_mean_hist = None # Not available
        f_pfc_hist = None # Not available
        print("Loaded old iv_log.csv format (psi mean and f_pfc not available).")

except Exception as e:
    print(f"Could not load iv_log.csv: {e}. Skipping log plots.")
    times_s = np.array([]); current_A = np.array([]); c_part_hist = np.array([]); c_dom_hist = np.array([]); V_hist = np.array([])
    psi_mean_hist = None; f_pfc_hist = None


# Quick analysis plots (updated to potentially include psi and f_pfc)
if times_s.size > 0: # Only plot if log data was loaded and is not empty
    plt.figure(figsize=(6,3.8)); plt.plot(times_s, current_A, 'k-'); plt.xlabel('time [s]'); plt.ylabel('current I [A]'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('current_vs_time.png', dpi=150)
    print('Saved: current_vs_time.png')

    plt.figure(figsize=(6,3.8)); plt.plot(c_part_hist, V_hist, 'b.-'); plt.xlabel('particle-avg conc'); plt.ylabel('voltage [V]'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('voltage_vs_concentration.png', dpi=150)
    print('Saved: voltage_vs_concentration.png')

    plt.figure(figsize=(6,3.8)); plt.plot(c_part_hist, current_A, 'r.-'); plt.xlabel('particle-avg conc'); plt.ylabel('current [A]'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('current_vs_concentration.png', dpi=150)
    print('Saved: current_vs_concentration.png')

    # Add new plots for psi and f_pfc if available
    if psi_mean_hist is not None and f_pfc_hist is not None:
        plt.figure(figsize=(6,3.8)); plt.plot(times_s, psi_mean_hist, 'g-'); plt.xlabel('time [s]'); plt.ylabel('<psi>'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('psi_mean_vs_time.png', dpi=150)
        print('Saved: psi_mean_vs_time.png')

        plt.figure(figsize=(6,3.8)); plt.plot(times_s, f_pfc_hist, 'm-'); plt.xlabel('time [s]'); plt.ylabel('Total PFC Free Energy'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('f_pfc_vs_time.png', dpi=150)
        print('Saved: f_pfc_vs_time.png')

        # Optional: voltage vs. mean psi
        plt.figure(figsize=(6,3.8)); plt.plot(psi_mean_hist, V_hist, 'c.-'); plt.xlabel('<psi>'); plt.ylabel('voltage [V]'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('voltage_vs_psi_mean.png', dpi=150)
        print('Saved: voltage_vs_psi_mean.png')

# Update the final print statement to list all potential saved plot files
print('Analysis plots generated: analysis_fields.png')
if mech_mode == 'hetero':
    print('                      : stiffness_fields.png')
if times_s.size > 0:
    print('                      : current_vs_time.png')
    print('                      : voltage_vs_concentration.png')
    print('                      : current_vs_concentration.png')
    if psi_mean_hist is not None:
         print('                      : psi_mean_vs_time.png')
         print('                      : f_pfc_vs_time.png')
         print('                      : voltage_vs_psi_mean.png')

Loaded checkpoint: lfp_spectral_mech_checkpoint.npz
Loaded PFC parameters from checkpoint.
Saved: analysis_fields.png
Loaded updated iv_log.csv (including psi mean and f_pfc)
Saved: current_vs_time.png
Saved: voltage_vs_concentration.png
Saved: current_vs_concentration.png
Saved: psi_mean_vs_time.png
Saved: f_pfc_vs_time.png
Saved: voltage_vs_psi_mean.png
Analysis plots generated: analysis_fields.png
                      : current_vs_time.png
                      : voltage_vs_concentration.png
                      : current_vs_concentration.png
                      : psi_mean_vs_time.png
                      : f_pfc_vs_time.png
                      : voltage_vs_psi_mean.png


In [69]:
import numpy as np

# Load the checkpoint data
try:
    data = np.load('lfp_spectral_mech_checkpoint.npz', allow_pickle=True)
    print("Loaded checkpoint: lfp_spectral_mech_checkpoint.npz")

    c = data['c'] # Need c to calculate f_el using the functions
    H = data['H'] # Need mask for elastic calculations
    Nx = int(data['Nx']); Ny = int(data['Ny'])
    Lx = float(data['Lx']); Ly = float(data['Ly'])

    # Load parameters needed for elastic calculation from checkpoint or global scope
    # Assuming these are available globally or in checkpoint after previous steps
    lam_p_d = float(data.get('lam_p_d', lam_p_d))
    mu_p_d = float(data.get('mu_p_d', mu_p_d))
    lam_r_d = float(data.get('lam_r_d', lam_r_d))
    mu_r_d = float(data.get('mu_r_d', mu_r_d))
    mech_mode = str(data.get('mech_mode', mech_mode))
    Eps0 = np.zeros((2,2)); # Reconstruct Eps0 - assume it's consistent with Model
    # Try to load e11, e22 from checkpoint or use globals if they exist
    e11 = float(data.get('e11', Model.get('e11', 0.0))) if 'Model' in globals() else float(data.get('e11', 0.0))
    e22 = float(data.get('e22', Model.get('e22', 0.0))) if 'Model' in globals() else float(data.get('e22', 0.0))
    Eps0[0,0] = e11
    Eps0[1,1] = e22

    # Need KX, KY, K2 for spectral calculations
    kx = 2*np.pi*fftfreq(Nx, d=Lx/Nx)
    ky = 2*np.pi*fftfreq(Ny, d=Ly/Ny)
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    K2 = KX**2 + KY**2
    mask0 = (np.abs(KX)<1e-14) & (np.abs(KY)<1e-14)
    K2[mask0] = 1e-30 # Avoid division by zero

    # Rebuild necessary spectral operators for elasticity if not globally available or mismatched
    # reference operator in k-space (rebuild as it depends on loaded elastic params)
    if mech_mode == 'uniform':
        lam0_d, mu0_d = lam_p_d, mu_p_d   # use particle as reference
    else:
        lam0_d, mu0_d = lam_r_d, mu_r_d   # reservoir as reference
    C0 = C_iso(lam0_d, mu0_d) # C_iso needs to be defined or imported

    K = np.stack((KX, KY), axis=-1)
    A = np.einsum('...j,ijkl,...k->...il', K, C0, K)
    A11=A[...,0,0]; A12=A[...,0,1]; A21=A[...,1,0]; A22=A[...,1,1]
    detA = A11*A22 - A12*A21
    detA[mask0] = 1.0 # Handle k=0 for inverse
    invA = np.empty_like(A)
    invA[...,0,0] = A22/detA; invA[...,0,1] = -A12/detA
    invA[...,1,0] = -A21/detA; invA[...,1,1] = A11/detA

    # Redefine solve_elastic_uniform and solve_elastic_hetero locally to ensure scope
    # These functions are needed to calculate mu_el and f_el
    def strain_from_u_local(Ux, Uy, KX_field=None, KY_field=None):
        if KX_field is None: KX_field = KX # Use local KX
        if KY_field is None: KY_field = KY # Use local KY
        Uxk = fftn(Ux); Uyk = fftn(Uy)
        Exx = np.real(ifftn(1j*KX_field*Uxk))
        Eyy = np.real(ifftn(1j*KY_field*Uyk))
        Exy = np.real(ifftn(0.5j*(KX_field*Uyk + KY_field*Uxk)))
        return Exx, Eyy, Exy

    def C_iso_local(lam, mu):
        C = np.zeros((2,2,2,2))
        for i in range(2):
            for j in range(2):
                for k in range(2):
                    for l in range(2):
                        C[i,j,k,l] = lam*(i==j)*(k==l) + mu*((i==k)*(j==l) + (i==l)*(j==k))
        return C

    def solve_elastic_uniform_local(c_field):
        C_part = C_iso_local(lam_p_d, mu_p_d)
        # Use eigenstrain_only_in_particle flag if available globally or reconstruct from Model
        eigenstrain_only_in_particle_local = bool(data.get('eigenstrain_only_in_particle', Model.get('eigenstrain_only_in_particle', True)) if 'Model' in globals() else data.get('eigenstrain_only_in_particle', True))
        c_eff = (H * c_field) if eigenstrain_only_in_particle_local else c_field # Use local H, c_field
        E0 = np.zeros((Nx,Ny,2,2))
        E0[...,0,0] = c_eff*Eps0[0,0] # Use local Eps0
        E0[...,1,1] = c_eff*Eps0[1,1]
        E0k = np.zeros_like(E0, dtype=complex)
        for a in range(2):
            for b in range(2):
                E0k[...,a,b] = fftn(E0[...,a,b])
        b = 1j*np.einsum('...j,ijkl,...kl->...i', K, C_part, E0k) # Use local K, C_part
        u_k = np.einsum('...ij,...j->...i', invA, b) # Use local invA
        u_k[mask0,...]=0.0 # Use local mask0
        Ux = np.real(ifftn(u_k[...,0])); Uy = np.real(ifftn(u_k[...,1]))
        Exx,Eyy,Exy = strain_from_u_local(Ux,Uy,KX,KY) # Use local strain_from_u_local, KX, KY
        DE = np.zeros_like(E0)
        DE[...,0,0]=Exx-E0[...,0,0]; DE[...,1,1]=Eyy-E0[...,1,1]
        DE[...,0,1]=Exy; DE[...,1,0]=Exy
        sigma = np.einsum('ijkl,...kl->...ij', C_part, DE)
        f_el = 0.5*np.einsum('...ij,ijkl,...kl->...', DE, C_part, DE)
        mu_el = -(sigma[...,0,0]*Eps0[0,0] + sigma[...,1,1]*Eps0[1,1] + 2.0*sigma[...,0,1]*Eps0[0,1])
        # Use mask_mech_to_particle flag if available globally or reconstruct from Model
        mask_mech_to_particle_local = bool(data.get('mask_mech_to_particle', Model.get('mask_mech_to_particle', True)) if 'Model' in globals() else data.get('mask_mech_to_particle', True))
        if mask_mech_to_particle_local:
           mu_el *= H # Use local H
           f_el  *= H # Use local H
        return mu_el, f_el

    def solve_hetero_mu_el(c_field, tol=1e-6, maxit=100, omega=0.7):
        lamF = lam_r_d + H*(lam_p_d - lam_r_d) # Use local H
        muF  = mu_r_d  + H*(mu_p_d  - mu_r_d) # Use local H
        # Use eigenstrain_only_in_particle flag if available globally or reconstruct from Model
        eigenstrain_only_in_particle_local = bool(data.get('eigenstrain_only_in_particle', Model.get('eigenstrain_only_in_particle', True)) if 'Model' in globals() else data.get('eigenstrain_only_in_particle', True))
        c_eff = (H*c_field) if eigenstrain_only_in_particle_local else c_field # Use local H, c_field
        E0xx = c_eff*Eps0[0,0]; E0yy = c_eff*Eps0[1,1]; E0xy = 0.0 # Use local Eps0
        Ux = np.zeros((Nx,Ny)); Uy = np.zeros((Nx,Ny)) # Use local Nx, Ny
        def stress(Exx,Eyy,Exy):
            dExx=Exx-E0xx; dEyy=Eyy-E0yy; dExy=Exy-E0xy
            tr = dExx+dEyy
            sxx = 2*muF*dExx + lamF*tr
            syy = 2*muF*dEyy + lamF*tr
            sxy = 2*muF*dExy
            return sxx, syy, sxy
        # initial residual norm
        sxx,syy,sxy = stress(0.0,0.0,0.0)
        sxxk, syyk, sxyk = fftn(sxx), fftn(syy), fftn(sxy)
        gkx = 1j*(KX*sxxk + KY*sxyk) # Use local KX, KY
        gky = 1j*(KX*sxyk + KY*syyk) # Use local KX, KY
        g0 = np.sqrt(np.mean(np.abs(gkx)**2 + np.abs(gky)**2)) or 1.0
        # Load hetero params from checkpoint or use defaults
        hetero_tol_local = float(data.get('hetero_tol', 1e-6))
        hetero_maxit_local = int(data.get('hetero_maxit', 100))
        hetero_omega_local = float(data.get('hetero_omega', 0.7))
        hetero_verbose_local = bool(data.get('hetero_verbose', False)) # Not used in return, but for logging
        mask_mu_el_to_particle_in_hetero_local = bool(data.get('mask_mu_el_to_particle_in_hetero', False)) # Not used in return
        for it in range(hetero_maxit_local): # Use local hetero_maxit_local
            Exx,Eyy,Exy = strain_from_u_local(Ux,Uy,KX,KY) # Use local strain_from_u_local, KX, KY
            sxx,syy,sxy = stress(Exx,Eyy,Exy)
            sxxk, syyk, sxyk = fftn(sxx), fftn(syy), fftn(sxy)
            gkx = 1j*(KX*sxxk + KY*sxyk) # Use local KX, KY
            gky = 1j*(KX*sxyk + KY*syyk) # Use local KX, KY
            rhsx, rhsy = -gkx, -gky
            dux_k = invA[...,0,0]*rhsx + invA[...,0,1]*rhsy # Use local invA
            duy_k = invA[...,1,0]*rhsx + invA[...,1,1]*rhsy # Use local invA
            dux_k[mask0] = 0.0 # Use local mask0
            duy_k[mask0] = 0.0
            Ux += hetero_omega_local*np.real(ifftn(dux_k)) # Use local hetero_omega_local
            Uy += hetero_omega_local*np.real(ifftn(duy_k))
            res = np.sqrt(np.mean(np.abs(gkx)**2 + np.abs(gky)**2))/g0
            if hetero_verbose_local and (it % 10 == 0 or it==0):
                print(f"hetero it={it:3d}, relres={res:.3e}")
            if res < hetero_tol_local: # Use local hetero_tol_local
                break
        Exx,Eyy,Exy = strain_from_u_local(Ux,Uy,KX,KY) # Use local strain_from_u_local, KX, KY
        dExx=Exx-E0xx; dEyy=Eyy-E0yy; dExy=Exy
        tr=dExx+dEyy
        f_el = 0.5*(2*muF*(dExx**2 + dEyy**2 + 2*dExy**2) + lamF*(tr**2))
        sxx,syy,sxy = stress(Exx,Eyy,Exy)
        mu_el = -(sxx*Eps0[0,0] + syy*Eps0[1,1] + 2.0*sxy*Eps0[0,1])
        if mask_mu_el_to_particle_in_hetero_local: # Use local mask_mu_el_to_particle_in_hetero_local
            mu_el *= H
        return mu_el, f_el # Note: hetero returns lamF, muF too, but not used in mech_mu_f return sig.

    def mech_mu_f_local(c_field): # Renamed function to use local elastic solvers
        # Use local mech_mode
        if mech_mode == 'hetero':
            return solve_elastic_hetero_local(c_field) # Call local hetero solver
        else:
            return solve_elastic_uniform_local(c_field) # Call local uniform solver


    # Calculate mu_el and f_el for the final state
    mu_el_final, f_el_final = mech_mu_f_local(c) # Use the loaded c and local mech_mu_f

    # Calculate the mean elastic energy density across the domain
    mean_f_el_domain = np.mean(f_el_final)

    # Optionally, calculate the mean within the particle mask
    mean_f_el_particle = np.sum(f_el_final * H) / np.sum(H) # Use local H

    print(f"Mean elastic energy density (whole domain): {mean_f_el_domain:.3e}")
    print(f"Mean elastic energy density (within particle mask): {mean_f_el_particle:.3e}")

except FileNotFoundError:
    print("Checkpoint file not found. Cannot perform quantitative analysis of elastic energy.")
except Exception as e:
    print(f"An error occurred during quantitative analysis of elastic energy: {e}")

# Quantitative analysis of mean psi over time is already captured in the psi_mean_vs_time.png plot
# You can examine this plot to see how the average phase field value evolved,
# which can indicate transitions or overall changes in the structural order.

# You can also load the log file and analyze the time series data for f_pfc
try:
    log_data = np.loadtxt('iv_log.csv', delimiter=',', skiprows=1)
    # Assuming columns: time_s, current_A, c_particle, c_domain, voltage_V, psi_mean, f_pfc
    if log_data.shape[1] > 6: # Check if f_pfc column exists
        times_s = log_data[:, 0]
        f_pfc_hist = log_data[:, 6]
        print("\nTotal PFC Free Energy over time (from iv_log.csv):")
        # Print first few and last few values
        print(f_pfc_hist[:5])
        if len(f_pfc_hist) > 10:
             print("...")
             print(f_pfc_hist[-5:])
        else:
             print(f_pfc_hist[5:])
        print(f"Final Total PFC Free Energy: {f_pfc_hist[-1]:.3e}")
    else:
        print("\nTotal PFC Free Energy data not found in iv_log.csv.")
except FileNotFoundError:
    print("\niv_log.csv not found. Cannot analyze PFC free energy over time.")
except Exception as e:
    print(f"\nAn error occurred while loading or analyzing iv_log.csv for PFC free energy: {e}")

Loaded checkpoint: lfp_spectral_mech_checkpoint.npz
Mean elastic energy density (whole domain): 4.224e-01
Mean elastic energy density (within particle mask): 8.105e-01

Total PFC Free Energy over time (from iv_log.csv):
[5.06924276e+10 5.06924276e+10 5.06924276e+10 5.06924276e+10
 5.06924276e+10]
...
[5.06924276e+10 5.06924276e+10 5.06924276e+10 5.06924276e+10
 5.06924276e+10]
Final Total PFC Free Energy: 5.069e+10


1.  **Load checkpoint**: Load the `psi` field and necessary grid information (Nx, Ny, Lx, Ly) from the `lfp_spectral_mech_checkpoint.npz` file.

In [55]:
import numpy as np

# Load the checkpoint data
try:
    data = np.load('lfp_spectral_mech_checkpoint.npz', allow_pickle=True)
    print("Loaded checkpoint: lfp_spectral_mech_checkpoint.npz")

    # Extract psi and grid information
    psi = data.get('psi', None)
    Nx = int(data.get('Nx', -1))
    Ny = int(data.get('Ny', -1))
    Lx = float(data.get('Lx', -1.0))
    Ly = float(data.get('Ly', -1.0))

    if psi is None or Nx == -1 or Ny == -1 or Lx == -1.0 or Ly == -1.0:
        print("Error: Could not load all necessary data from the checkpoint.")
    else:
        print(f"Successfully loaded psi field with shape {psi.shape}")
        print(f"Grid dimensions: Nx={Nx}, Ny={Ny}, Lx={Lx}, Ly={Ly}")

except FileNotFoundError:
    print("Error: Checkpoint file 'lfp_spectral_mech_checkpoint.npz' not found.")
except Exception as e:
    print(f"An error occurred while loading the checkpoint: {e}")

Loaded checkpoint: lfp_spectral_mech_checkpoint.npz
Successfully loaded psi field with shape (214, 428)
Grid dimensions: Nx=214, Ny=428, Lx=32.0, Ly=64.0


2.  **Compute FFT of psi**: Calculate the 2D Fast Fourier Transform (FFT) of the `psi` field.

In [56]:
import numpy.fft as fft

# Ensure psi was loaded in the previous step
if 'psi' not in locals() or psi is None:
    print("Error: psi field not loaded. Please run the previous cell first.")
else:
    # Compute the 2D FFT of the psi field
    psi_k = fft.fftn(psi)

    print(f"Computed FFT of psi field. Result shape: {psi_k.shape}")
    # Display the magnitude of the k=0 component (average value)
    print(f"Magnitude of k=0 component (|psi_k[0,0]|): {np.abs(psi_k[0,0]):.3e}")

Computed FFT of psi field. Result shape: (214, 428)
Magnitude of k=0 component (|psi_k[0,0]|): 5.387e+00


3.  **Compute Power Spectrum**: Compute the power spectrum, which is typically the magnitude squared of the FFT, and often averaged radially or displayed in 2D k-space.

In [57]:
import numpy as np

# Ensure psi_k was computed in the previous step
if 'psi_k' not in locals():
    print("Error: psi_k not computed. Please run the previous cell first.")
else:
    # Compute the power spectrum (magnitude squared of the FFT)
    power_spectrum_2d = np.abs(psi_k)**2

    print(f"Computed 2D Power Spectrum. Result shape: {power_spectrum_2d.shape}")
    # The k=0 component is the square of the sum/mean of psi, often very large
    print(f"Magnitude of k=0 component in Power Spectrum: {power_spectrum_2d[0,0]:.3e}")

    # Shift the zero-frequency component to the center for visualization
    power_spectrum_2d_shifted = np.fft.fftshift(power_spectrum_2d)

    print("Shifted 2D Power Spectrum for visualization.")

Computed 2D Power Spectrum. Result shape: (214, 428)
Magnitude of k=0 component in Power Spectrum: 2.902e+01
Shifted 2D Power Spectrum for visualization.


4.  **Visualize Power Spectrum**: Plot the 2D power spectrum in k-space. This plot will show the characteristic peaks corresponding to the crystal structure.

In [58]:
import matplotlib.pyplot as plt
import numpy as np

# Ensure power_spectrum_2d_shifted was computed in the previous step
if 'power_spectrum_2d_shifted' not in locals():
    print("Error: Shifted power spectrum not computed. Please run the previous cells first.")
else:
    # Create k-space axes for plotting
    # Use the kx and ky from the global scope if available, otherwise recompute
    if 'kx' not in globals() or 'ky' not in globals():
         print("Recomputing kx and ky for plotting axes.")
         # Need Nx, Ny, Lx, Ly which should be available from checkpoint loading step
         if 'Nx' not in globals() or 'Ny' not in globals() or 'Lx' not in globals() or 'Ly' not in globals():
              print("Error: Grid dimensions not available for recomputing kx and ky.")
         else:
              kx = 2*np.pi*np.fft.fftfreq(Nx, d=Lx/Nx)
              ky = 2*np.pi*np.fft.fftfreq(Ny, d=Ly/Ny)


    if 'kx' in globals() and 'ky' in globals():
         # Create extent for the k-space plot
         extent_k = [kx.min(), kx.max(), ky.min(), ky.max()]

         plt.figure(figsize=(7, 6))
         # Use log scale for the power spectrum for better visibility of peaks
         # Clip values to avoid log(0)
         power_spectrum_log = np.log10(np.clip(power_spectrum_2d_shifted, 1e-12, None))
         im = plt.imshow(power_spectrum_log.T, origin='lower', cmap='viridis',
                         extent=extent_k,
                         vmax=np.max(power_spectrum_log)*0.8) # Adjust vmax for better contrast if needed
         plt.colorbar(im, label='log10(Power)')
         plt.title('2D Power Spectrum of ψ (shifted)')
         plt.xlabel('$k_x$')
         plt.ylabel('$k_y$')
         plt.grid(True, alpha=0.3)
         plt.tight_layout()
         plt.savefig('power_spectrum_2d.png', dpi=150)
         print('Saved: power_spectrum_2d.png')
         plt.show()
    else:
        print("Error: kx and ky could not be determined for plotting.")

Saved: power_spectrum_2d.png


In [59]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fftn, ifftn

# Load the checkpoint data
try:
    data = np.load('lfp_spectral_mech_checkpoint.npz', allow_pickle=True)
    print("Loaded checkpoint: lfp_spectral_mech_checkpoint.npz")

    psi = data.get('psi', None)
    Nx = int(data.get('Nx', -1))
    Ny = int(data.get('Ny', -1))
    Lx = float(data.get('Lx', -1.0))
    Ly = float(data.get('Ly', -1.0))
    H = data.get('H', np.ones((Nx, Ny))) # Load mask as well

    if psi is None or Nx == -1 or Ny == -1 or Lx == -1.0 or Ly == -1.0:
        print("Error: Could not load all necessary data from the checkpoint.")
except FileNotFoundError:
    print("Error: Checkpoint file 'lfp_spectral_mech_checkpoint.npz' not found.")
except Exception as e:
    print(f"An error occurred while loading the checkpoint: {e}")

# Ensure data was loaded successfully before proceeding
if 'psi' in locals() and psi is not None and Nx != -1:
    # Need KX, KY for gradient calculation
    # Use the global KX, KY if available, otherwise recompute
    if 'KX' not in globals() or 'KY' not in globals() or KX.shape != psi.shape:
         print("Recomputing KX and KY for gradient calculation.")
         kx = 2*np.pi*np.fft.fftfreq(Nx, d=Lx/Nx)
         ky = 2*np.pi*np.fft.fftfreq(Ny, d=Ly/Ny)
         KX, KY = np.meshgrid(kx, ky, indexing='ij')


    # Compute the gradient of psi using FFTs
    psi_k = fftn(psi)
    dpsi_dx = np.real(ifftn(1j * KX * psi_k))
    dpsi_dy = np.real(ifftn(1j * KY * psi_k))

    # Compute the magnitude of the gradient
    grad_psi_magnitude = np.sqrt(dpsi_dx**2 + dpsi_dy**2)

    # Plot the magnitude of the gradient in real space
    extent = [0, Lx, 0, Ly]
    plt.figure(figsize=(7, 6))
    im = plt.imshow(grad_psi_magnitude.T, origin='lower', extent=extent, cmap='hot_r') # hot_r or viridis can work well
    plt.colorbar(im, label='|∇ψ|')
    plt.title('Magnitude of Gradient of ψ in Real Space')
    plt.xlabel('x [Wc]'); plt.ylabel('y [Wc]')

    # Optionally, overlay the particle boundary
    if H is not None:
        plt.contour(H.T, levels=[0.5], colors='cyan', linewidths=0.8, extent=extent)

    plt.tight_layout()
    plt.savefig('grad_psi_magnitude.png', dpi=150)
    print('Saved: grad_psi_magnitude.png')
    plt.show()

else:
    print("Could not proceed with plotting as necessary data was not loaded.")

Loaded checkpoint: lfp_spectral_mech_checkpoint.npz
Saved: grad_psi_magnitude.png


In [83]:
import numpy as np
import pandas as pd

# Load the log file from the CH-only simulation
try:
    log_ch_only = pd.read_csv('iv_log_ch.csv')
    print("Loaded iv_log_ch.csv (CH-only simulation):")
    display(log_ch_only.head())
    print("\nColumns in CH-only log:", log_ch_only.columns.tolist())

except FileNotFoundError:
    print("Error: iv_log_ch.csv not found. Cannot compare logs.")
    log_ch_only = None
except Exception as e:
    print(f"An error occurred while loading iv_log_ch.csv: {e}")
    log_ch_only = None

print("-" * 30)

# Load the log file from the latest coupled simulation
try:
    log_coupled = pd.read_csv('iv_log.csv')
    print("Loaded iv_log.csv (Coupled simulation):")
    display(log_coupled.head())
    print("\nColumns in Coupled log:", log_coupled.columns.tolist())

except FileNotFoundError:
    print("Error: iv_log.csv not found. Cannot compare logs.")
    log_coupled = None
except Exception as e:
    print(f"An error occurred while loading iv_log.csv: {e}")
    log_coupled = None

if log_ch_only is not None and log_coupled is not None:
    print("\nLog files loaded. We can now compare specific columns like 'current_A' and 'voltage_V'.")

Loaded iv_log_ch.csv (CH-only simulation):


,time_s,current_A,c_particle,c_domain,voltage_V
0,1.000000e-07,-4.897861e-12,0.120686,0.555931,3.434437
1,2.000000e-07,-1.281660e-11,0.128663,0.560556,3.436704
2,3.000000e-07,-2.331438e-11,0.134530,0.563785,3.438726
3,4.000000e-07,-3.281367e-11,0.139080,0.566231,3.440278
4,5.000000e-07,-4.074645e-11,0.142748,0.568174,3.441504



Columns in CH-only log: ['time_s', 'current_A', 'c_particle', 'c_domain', 'voltage_V']
------------------------------
Loaded iv_log.csv (Coupled simulation):


,time_s,current_A,c_particle,c_domain,voltage_V,psi_mean,f_pfc
0,1.000000e-07,-9.638587e-11,0.161607,0.578043,3.350502,-0.000786,5.069243e+10
1,2.000000e-07,-9.638587e-11,0.161607,0.578043,3.350502,0.000786,5.069243e+10
2,3.000000e-07,-9.638587e-11,0.161607,0.578043,3.350502,-0.000786,5.069243e+10
3,4.000000e-07,-9.638587e-11,0.161607,0.578043,3.350502,0.000786,5.069243e+10
4,5.000000e-07,-9.638587e-11,0.161607,0.578043,3.350502,-0.000786,5.069243e+10



Columns in Coupled log: ['time_s', 'current_A', 'c_particle', 'c_domain', 'voltage_V', 'psi_mean', 'f_pfc']

Log files loaded. We can now compare specific columns like 'current_A' and 'voltage_V'.


In [85]:
import matplotlib.pyplot as plt
import pandas as pd

# Ensure log files were loaded in the previous step
if 'log_ch_only' not in locals() or log_ch_only is None or 'log_coupled' not in locals() or log_coupled is None:
    print("Error: Both log files must be loaded to compare. Please run the previous cell.")
else:
    # Plot Current vs Time
    plt.figure(figsize=(8, 4))
    plt.plot(log_ch_only['time_s'], log_ch_only['current_A'], label='CH-only', color='blue', linestyle='-')
    plt.plot(log_coupled['time_s'], log_coupled['current_A'], label='Coupled (CH+PFC)', color='red', linestyle='--')
    plt.xlabel('time [s]')
    plt.ylabel('current I [A]')
    plt.title('Current vs Time Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('current_vs_time_comparison.png', dpi=150)
    print('Saved: current_vs_time_comparison.png')
    plt.show()

    # Plot Voltage vs Time
    plt.figure(figsize=(8, 4))
    plt.plot(log_ch_only['time_s'], log_ch_only['voltage_V'], label='CH-only', color='blue', linestyle='-')
    plt.plot(log_coupled['time_s'], log_coupled['voltage_V'], label='Coupled (CH+PFC)', color='red', linestyle='--')
    plt.xlabel('time [s]')
    plt.ylabel('voltage [V]')
    plt.title('Voltage vs Time Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('voltage_vs_time_comparison.png', dpi=150)
    print('Saved: voltage_vs_time_comparison.png')
    plt.show()

    # Optional: Plot Voltage vs Particle Concentration for comparison
    plt.figure(figsize=(8, 4))
    # Corrected: Use separate marker and linestyle arguments with valid values
    plt.plot(log_ch_only['c_particle'], log_ch_only['voltage_V'], label='CH-only', color='blue', linestyle='-', marker='.')
    plt.plot(log_coupled['c_particle'], log_coupled['voltage_V'], label='Coupled (CH+PFC)', color='red', linestyle='--', marker='.')
    plt.xlabel('particle-avg conc')
    plt.ylabel('voltage [V]')
    plt.title('Voltage vs Particle Concentration Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('voltage_vs_concentration_comparison.png', dpi=150)
    print('Saved: voltage_vs_concentration_comparison.png')
    plt.show()

Saved: current_vs_time_comparison.png
Saved: voltage_vs_time_comparison.png
Saved: voltage_vs_concentration_comparison.png
